# Notebook 4 — Chronological RTS-GMLC Baseline and Congestion Screening

## Purpose

This notebook extends the validated single-period DC optimal power flow developed
in Notebook 3 to the complete 8,784-hour RTS-GMLC day-ahead chronology.

The principal question is:

> Where, when, how often, and with what economic significance does transmission
> congestion occur in the native RTS-GMLC system?

The annual evidence developed here will be used to select a candidate corridor
for the later reconductoring experiments. The A27 corridor identified in
Notebook 3 remains only a candidate until the chronological analysis is complete.

## Initial modeling scope

Each hour is initially modeled as an independent linear DC-OPF.

Included:

- 73 buses;
- 120 physical branches, including parallel circuits;
- 158 generator objects;
- hourly regional load;
- fixed within-area bus load shares;
- hourly Hydro, PV, RTPV, and Wind availability;
- static thermal-generator maximum capacity;
- linear full-load-average generator costs;
- branch thermal limits;
- branch shadow prices and independent solution validation.

Not yet included:

- data-center load;
- workload flexibility;
- reconductoring;
- generator ramping;
- unit commitment;
- minimum generation constraints;
- storage state of charge;
- AC power-flow effects or transmission losses.

## Important interpretation

This notebook builds the native-system benchmark against which the thesis
interventions will later be measured. High line utilization alone does not
establish economically meaningful congestion. Corridor selection must consider
both the frequency of binding constraints and the economic value of additional
transmission capacity.

In [ ]:
from pathlib import Path
import platform
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyomo
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition

warnings.filterwarnings("default")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

S_BASE = 100.0
VALIDATION_TOL = 1e-6

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("Pyomo:", pyomo.__version__)

In [ ]:
PROJECT_PATH = Path.home() / "thesis-grid-ai"
RTS_ROOT = Path.home() / "RTS-GMLC"
RTS_PATH = RTS_ROOT / "RTS_Data" / "SourceData"

NOTEBOOKS_PATH = PROJECT_PATH / "notebooks"
SRC_PATH = PROJECT_PATH / "src"
RESULTS_PATH = PROJECT_PATH / "results"
FIGURES_PATH = PROJECT_PATH / "figures"

REGIONAL_LOAD_PATH = (
    RTS_PATH / "../timeseries_data_files/Load/DAY_AHEAD_regional_Load.csv"
).resolve()

required_paths = {
    "Project directory": PROJECT_PATH,
    "RTS source-data directory": RTS_PATH,
    "bus.csv": RTS_PATH / "bus.csv",
    "branch.csv": RTS_PATH / "branch.csv",
    "gen.csv": RTS_PATH / "gen.csv",
    "timeseries_pointers.csv": RTS_PATH / "timeseries_pointers.csv",
    "DAY_AHEAD regional load": REGIONAL_LOAD_PATH,
}

missing_paths = []

for label, path in required_paths.items():
    exists = path.exists()
    print(f"{label:<30} {'FOUND' if exists else 'MISSING'}  {path}")
    
    if not exists:
        missing_paths.append((label, path))

if missing_paths:
    missing_text = "\n".join(
        f"- {label}: {path}" for label, path in missing_paths
    )
    raise FileNotFoundError(
        "One or more required paths could not be found:\n" + missing_text
    )

In [ ]:
bus = pd.read_csv(RTS_PATH / "bus.csv")
branch = pd.read_csv(RTS_PATH / "branch.csv")
gen = pd.read_csv(RTS_PATH / "gen.csv")
pointers = pd.read_csv(RTS_PATH / "timeseries_pointers.csv")
regional_load_raw = pd.read_csv(REGIONAL_LOAD_PATH)

datasets = {
    "bus": bus,
    "branch": branch,
    "gen": gen,
    "timeseries_pointers": pointers,
    "DAY_AHEAD regional load": regional_load_raw,
}

input_audit = pd.DataFrame(
    [
        {
            "Dataset": name,
            "Rows": len(df),
            "Columns": len(df.columns),
            "Missing cells": int(df.isna().sum().sum()),
            "Duplicate complete rows": int(df.duplicated().sum()),
        }
        for name, df in datasets.items()
    ]
)

display(input_audit)

expected_shapes = {
    "bus": (73, 15),
    "branch": (120, 14),
    "gen": (158, 57),
    "timeseries_pointers": (282, 6),
    "DAY_AHEAD regional load": (8784, 7),
}

for name, expected_shape in expected_shapes.items():
    actual_shape = datasets[name].shape
    assert actual_shape == expected_shape, (
        f"{name}: expected shape {expected_shape}, "
        f"but found {actual_shape}"
    )

print("\nAll raw dataset shapes match the previously audited RTS-GMLC inputs.")

for name, df in datasets.items():
    print(f"\n{name} columns:")
    print(df.columns.tolist())

## 1. Constructing the hourly bus-demand chronology

The RTS-GMLC dataset provides hourly day-ahead load at the regional level, but
it does not provide an independent hourly demand series for every bus.

Hourly demand at bus `i`, located in area `a`, is reconstructed as:

$$
D_{i,t} = L_{a,t} s_i
$$

The fixed load share for bus `i` is:

$$
s_i =
\frac{D_i^{\mathrm{static}}}
{\displaystyle\sum_{j \in a} D_j^{\mathrm{static}}}
$$

where:

- `D_static_i` is the static MW load assigned to bus `i`;
- `s_i` is bus `i`'s fixed share of its area's static load;
- `L_a,t` is the regional load in area `a` during hour `t`;
- `D_i,t` is the reconstructed hourly demand at bus `i`.

Substituting the second equation into the first gives:

$$
D_{i,t}
=
L_{a,t}
\left(
\frac{D_i^{\mathrm{static}}}
{\displaystyle\sum_{j \in a} D_j^{\mathrm{static}}}
\right)
$$

This construction exactly preserves every hourly regional load total:

$$
\sum_{i \in a} D_{i,t} = L_{a,t}
$$

However, the fixed within-area spatial distribution is a modeling assumption,
not an observed bus-level hourly demand pattern.

In [ ]:
regional_load = regional_load_raw.copy()

# RTS periods are numbered 1 through 24.
# Period 1 represents 00:00 and Period 24 represents 23:00.
regional_load["Timestamp"] = (
    pd.to_datetime(
        {
            "year": regional_load["Year"],
            "month": regional_load["Month"],
            "day": regional_load["Day"],
        }
    )
    + pd.to_timedelta(regional_load["Period"] - 1, unit="h")
)

print("Number of rows:", len(regional_load))
print("First timestamp:", regional_load["Timestamp"].iloc[0])
print("Last timestamp:", regional_load["Timestamp"].iloc[-1])
print("Minimum period:", regional_load["Period"].min())
print("Maximum period:", regional_load["Period"].max())
print("Unique timestamps:", regional_load["Timestamp"].nunique())
print("Duplicate timestamps:", regional_load["Timestamp"].duplicated().sum())
print("Monotonically increasing:", regional_load["Timestamp"].is_monotonic_increasing)

hour_differences = regional_load["Timestamp"].diff().dropna()
non_hourly_gaps = hour_differences[hour_differences != pd.Timedelta(hours=1)]

print("Non-hourly gaps:", len(non_hourly_gaps))
print(
    "Leap day hours:",
    regional_load["Timestamp"]
    .dt.strftime("%Y-%m-%d")
    .eq("2020-02-29")
    .sum()
)

assert len(regional_load) == 8784
assert regional_load["Period"].between(1, 24).all()
assert regional_load["Timestamp"].is_unique
assert regional_load["Timestamp"].is_monotonic_increasing
assert len(non_hourly_gaps) == 0
assert (
    regional_load["Timestamp"]
    .dt.strftime("%Y-%m-%d")
    .eq("2020-02-29")
    .sum()
    == 24
)

print("\nTimestamp chronology passed all validation checks.")

In [ ]:
bus_load_data = bus[["Bus ID", "Area", "MW Load"]].copy()

assert bus_load_data["Bus ID"].is_unique
assert bus_load_data["MW Load"].ge(0).all()

static_area_load = (
    bus_load_data
    .groupby("Area", sort=True)["MW Load"]
    .sum()
)

bus_load_data["Static Area Load MW"] = (
    bus_load_data["Area"].map(static_area_load)
)

bus_load_data["Load Share Within Area"] = (
    bus_load_data["MW Load"]
    / bus_load_data["Static Area Load MW"]
)

area_share_audit = (
    bus_load_data
    .groupby("Area", sort=True)
    .agg(
        Number_of_Buses=("Bus ID", "count"),
        Positive_Load_Buses=("MW Load", lambda x: int((x > 0).sum())),
        Static_Load_MW=("MW Load", "sum"),
        Sum_of_Load_Shares=("Load Share Within Area", "sum"),
    )
)

display(area_share_audit)

assert set(static_area_load.index) == {1, 2, 3}
assert np.allclose(
    static_area_load.to_numpy(),
    np.array([2850.0, 2850.0, 2850.0]),
    atol=VALIDATION_TOL,
)
assert np.allclose(
    area_share_audit["Sum_of_Load_Shares"].to_numpy(),
    1.0,
    atol=VALIDATION_TOL,
)

print("Total static system load:", bus_load_data["MW Load"].sum(), "MW")
print(
    "Buses with positive static load:",
    int((bus_load_data["MW Load"] > 0).sum()),
)
print("\nWithin-area load shares passed all validation checks.")

In [ ]:
area_columns = ["1", "2", "3"]

regional_load_mw = (
    regional_load
    .set_index("Timestamp")[area_columns]
    .copy()
)

# Convert area labels from strings to integers so they match bus["Area"].
regional_load_mw.columns = regional_load_mw.columns.astype(int)
regional_load_mw.index.name = "Timestamp"

bus_ids = bus_load_data["Bus ID"].to_numpy(dtype=int)
bus_areas = bus_load_data["Area"].to_numpy(dtype=int)
bus_shares = bus_load_data["Load Share Within Area"].to_numpy(dtype=float)

# Repeat the appropriate regional series for every bus,
# then multiply by that bus's fixed within-area share.
regional_load_for_each_bus = (
    regional_load_mw
    .loc[:, bus_areas]
    .to_numpy(dtype=float)
)

hourly_bus_load = pd.DataFrame(
    regional_load_for_each_bus * bus_shares[np.newaxis, :],
    index=regional_load_mw.index,
    columns=pd.Index(bus_ids, name="Bus ID"),
)

print("Regional load matrix shape:", regional_load_mw.shape)
print("Hourly bus-load matrix shape:", hourly_bus_load.shape)

display(hourly_bus_load.head())

In [ ]:
area_reconstruction_records = []

for area in sorted(bus_load_data["Area"].unique()):
    area_bus_ids = bus_load_data.loc[
        bus_load_data["Area"] == area,
        "Bus ID",
    ].tolist()

    reconstructed_area_load = hourly_bus_load[area_bus_ids].sum(axis=1)
    original_area_load = regional_load_mw[area]

    reconstruction_error = reconstructed_area_load - original_area_load

    area_reconstruction_records.append(
        {
            "Area": area,
            "Number of buses": len(area_bus_ids),
            "Maximum absolute error MW": reconstruction_error.abs().max(),
            "Mean absolute error MW": reconstruction_error.abs().mean(),
        }
    )

area_reconstruction_audit = pd.DataFrame(area_reconstruction_records)
display(area_reconstruction_audit)

system_load_from_buses = hourly_bus_load.sum(axis=1)
system_load_from_regions = regional_load_mw.sum(axis=1)

system_reconstruction_error = (
    system_load_from_buses - system_load_from_regions
)

peak_timestamp = system_load_from_buses.idxmax()
peak_demand_mw = system_load_from_buses.max()

system_load_summary = pd.Series(
    {
        "Number of hours": len(system_load_from_buses),
        "Minimum demand MW": system_load_from_buses.min(),
        "Mean demand MW": system_load_from_buses.mean(),
        "Maximum demand MW": peak_demand_mw,
        "Peak timestamp": peak_timestamp,
        "Maximum system reconstruction error MW":
            system_reconstruction_error.abs().max(),
        "Negative bus-load values":
            int((hourly_bus_load < -VALIDATION_TOL).sum().sum()),
    },
    name="Value",
)

display(system_load_summary.to_frame())

assert hourly_bus_load.shape == (8784, 73)
assert hourly_bus_load.columns.is_unique
assert hourly_bus_load.index.is_unique
assert hourly_bus_load.index.equals(regional_load_mw.index)

assert (
    area_reconstruction_audit["Maximum absolute error MW"].max()
    <= VALIDATION_TOL
)

assert system_reconstruction_error.abs().max() <= VALIDATION_TOL
assert (hourly_bus_load >= -VALIDATION_TOL).all().all()

# Regression checks against Notebook 1
assert peak_timestamp == pd.Timestamp("2020-08-26 14:00:00")
assert np.isclose(
    peak_demand_mw,
    8191.835957,
    atol=1e-6,
)

print("Hourly bus-demand construction passed all validation checks.")

## 2. Hourly generator-availability inputs

Generator availability determines the upper production bound for generator `g`
in hour `t`:

$$
0 \leq P_{g,t} \leq \overline{P}_{g,t}
$$

For Hydro, PV, RTPV, and Wind generators, the RTS-GMLC day-ahead time-series
files already report available generation capacity in MW. Therefore:

$$
\overline{P}_{g,t}
=
A_{g,t}^{\mathrm{RTS}}
$$

where:

- `P_g,t` is the optimized dispatch of generator `g`;
- `A_RTS_g,t` is the raw RTS-GMLC day-ahead availability value in MW;
- `PMax_g,t` is the hourly upper production bound.

The `Scaling Factor` field from `timeseries_pointers.csv` is not multiplied by
the raw series in this model. Direct inspection shows that the time-series
values are already expressed in MW. Applying the scaling factor again would
double-scale the generator availability and could create physically impossible
capacities.

For generators without hourly availability time series, the initial
chronological baseline uses:

$$
\overline{P}_{g,t}
=
P_g^{\mathrm{static}}
$$

for every hour.

The complete initial treatment is:

- Hydro, PV, RTPV, and Wind use their raw hourly day-ahead MW series.
- Thermal generators use their static `PMax MW` limits.
- Storage availability is fixed at zero because independent hourly OPFs do not
  represent intertemporal state of charge.
- Synchronous condensers have zero active-power availability.

Generator availability is an upper bound, not actual generation. Actual
dispatch remains an optimization decision.

In [ ]:
day_ahead_pmax_pointers = pointers.loc[
    pointers["Simulation"].astype(str).str.strip().eq("DAY_AHEAD")
    & pointers["Parameter"].astype(str).str.strip().eq("PMax MW")
].copy()

day_ahead_pmax_pointers = (
    day_ahead_pmax_pointers
    .sort_values(["Data File", "Object"])
    .reset_index(drop=True)
)

# The pointer Category column says "Generator" for all 80 records.
# The renewable/hydro resource type is encoded in the file path.
file_text = (
    day_ahead_pmax_pointers["Data File"]
    .astype(str)
    .str.upper()
)

day_ahead_pmax_pointers["Resource"] = np.select(
    [
        file_text.str.contains("/HYDRO/", regex=False),
        file_text.str.contains("/PV/", regex=False),
        file_text.str.contains("/RTPV/", regex=False),
        file_text.str.contains("/WIND/", regex=False),
    ],
    [
        "Hydro",
        "PV",
        "RTPV",
        "Wind",
    ],
    default="Unknown",
)

pointer_resource_audit = (
    day_ahead_pmax_pointers
    .groupby("Resource", sort=True)
    .agg(
        Number_of_Objects=("Object", "count"),
        Unique_Objects=("Object", "nunique"),
        Number_of_Files=("Data File", "nunique"),
        Scaling_Factor_Min=("Scaling Factor", "min"),
        Scaling_Factor_Max=("Scaling Factor", "max"),
    )
)

display(pointer_resource_audit)

print(
    "Original pointer categories:",
    sorted(day_ahead_pmax_pointers["Category"].unique()),
)
print(
    "Total DAY_AHEAD PMax pointers:",
    len(day_ahead_pmax_pointers),
)
print(
    "Unique generator objects:",
    day_ahead_pmax_pointers["Object"].nunique(),
)
print(
    "Unique data files:",
    day_ahead_pmax_pointers["Data File"].nunique(),
)
print(
    "Duplicate generator objects:",
    day_ahead_pmax_pointers["Object"].duplicated().sum(),
)

pointer_objects = set(day_ahead_pmax_pointers["Object"])
generator_objects = set(gen["GEN UID"])

missing_from_gen = sorted(pointer_objects - generator_objects)

print("Pointer objects absent from gen.csv:", len(missing_from_gen))

if missing_from_gen:
    print(missing_from_gen)

expected_resource_counts = {
    "Hydro": 20,
    "PV": 25,
    "RTPV": 31,
    "Wind": 4,
}

actual_resource_counts = (
    day_ahead_pmax_pointers["Resource"]
    .value_counts()
    .to_dict()
)

assert len(day_ahead_pmax_pointers) == 80
assert day_ahead_pmax_pointers["Object"].nunique() == 80
assert day_ahead_pmax_pointers["Object"].duplicated().sum() == 0
assert len(missing_from_gen) == 0
assert "Unknown" not in actual_resource_counts
assert actual_resource_counts == expected_resource_counts

print("\nDAY_AHEAD PMax pointer audit passed.")

In [ ]:
generator_timeseries_files = {}

file_audit_records = []

for data_file, pointer_group in day_ahead_pmax_pointers.groupby(
    "Data File",
    sort=True,
):
    resolved_path = (RTS_PATH / data_file).resolve()

    print("\nFile:", data_file)
    print("Resolved path:", resolved_path)
    print("Exists:", resolved_path.exists())

    assert resolved_path.exists(), f"Missing time-series file: {resolved_path}"

    timeseries_df = pd.read_csv(resolved_path)
    generator_timeseries_files[data_file] = timeseries_df

    time_columns = ["Year", "Month", "Day", "Period"]
    series_columns = [
        column
        for column in timeseries_df.columns
        if column not in time_columns
    ]

    expected_objects = set(pointer_group["Object"])
    available_series = set(series_columns)

    missing_series = sorted(expected_objects - available_series)
    additional_series = sorted(available_series - expected_objects)

    file_audit_records.append(
        {
            "Data File": data_file,
            "Rows": len(timeseries_df),
            "Columns": len(timeseries_df.columns),
            "Pointers Using File": len(pointer_group),
            "Series Columns": len(series_columns),
            "Missing Expected Series": len(missing_series),
            "Additional Series": len(additional_series),
        }
    )

    print("Shape:", timeseries_df.shape)
    print("Pointers using file:", len(pointer_group))
    print("Series columns:", len(series_columns))
    print("Missing expected series:", missing_series)
    print("Additional series:", additional_series)

    if len(series_columns) <= 15:
        print("Series names:", series_columns)
    else:
        print("First series names:", series_columns[:10])
        print("Last series names:", series_columns[-5:])

generator_file_audit = pd.DataFrame(file_audit_records)

print("\nGenerator time-series file audit:")
display(generator_file_audit)

assert len(generator_timeseries_files) == 4
assert (generator_file_audit["Rows"] == 8784).all()
assert (generator_file_audit["Missing Expected Series"] == 0).all()

print("All generator time-series files passed the initial audit.")

In [ ]:
pointer_generator_audit = day_ahead_pmax_pointers.merge(
    gen[
        [
            "GEN UID",
            "Bus ID",
            "Unit Type",
            "Fuel",
            "PMax MW",
        ]
    ],
    left_on="Object",
    right_on="GEN UID",
    how="left",
    validate="one_to_one",
)

pointer_generator_audit["Scaling Minus Static PMax MW"] = (
    pointer_generator_audit["Scaling Factor"]
    - pointer_generator_audit["PMax MW"]
)

pointer_generator_audit["Scaling Matches Static PMax"] = np.isclose(
    pointer_generator_audit["Scaling Factor"],
    pointer_generator_audit["PMax MW"],
    rtol=0,
    atol=1e-9,
)

resource_scaling_audit = (
    pointer_generator_audit
    .groupby("Resource", sort=True)
    .agg(
        Number_of_Generators=("Object", "count"),
        Total_Static_PMax_MW=("PMax MW", "sum"),
        Total_Scaling_Factor=("Scaling Factor", "sum"),
        Minimum_Static_PMax_MW=("PMax MW", "min"),
        Maximum_Static_PMax_MW=("PMax MW", "max"),
        Maximum_Absolute_Difference_MW=(
            "Scaling Minus Static PMax MW",
            lambda x: x.abs().max(),
        ),
        All_Scaling_Factors_Match=(
            "Scaling Matches Static PMax",
            "all",
        ),
    )
)

display(resource_scaling_audit)

scaling_mismatches = pointer_generator_audit.loc[
    ~pointer_generator_audit["Scaling Matches Static PMax"],
    [
        "Object",
        "Resource",
        "Fuel",
        "Unit Type",
        "PMax MW",
        "Scaling Factor",
        "Scaling Minus Static PMax MW",
    ],
]

print(
    "Generators whose scaling factor differs from static PMax:",
    len(scaling_mismatches),
)

if not scaling_mismatches.empty:
    display(scaling_mismatches)

In [ ]:
generator_timeseries_aligned = {}
generator_profile_records = []

gen_pmax_lookup = (
    gen
    .set_index("GEN UID")["PMax MW"]
    .astype(float)
)

assert gen_pmax_lookup.index.is_unique

for data_file, pointer_group in day_ahead_pmax_pointers.groupby(
    "Data File",
    sort=True,
):
    timeseries_df = generator_timeseries_files[data_file].copy()

    file_timestamps = (
        pd.to_datetime(
            {
                "year": timeseries_df["Year"],
                "month": timeseries_df["Month"],
                "day": timeseries_df["Day"],
            }
        )
        + pd.to_timedelta(timeseries_df["Period"] - 1, unit="h")
    )

    file_index = pd.DatetimeIndex(
        file_timestamps,
        name="Timestamp",
    )

    assert file_index.is_unique
    assert file_index.is_monotonic_increasing
    assert file_index.equals(regional_load_mw.index)

    object_columns = pointer_group["Object"].tolist()

    raw_profiles = (
        timeseries_df[object_columns]
        .apply(pd.to_numeric, errors="coerce")
    )

    raw_profiles.index = file_index

    generator_timeseries_aligned[data_file] = raw_profiles

    resource = pointer_group["Resource"].iloc[0]

    for generator_uid in object_columns:
        profile = raw_profiles[generator_uid]
        static_pmax = gen_pmax_lookup.loc[generator_uid]

        above_static = profile > static_pmax + VALIDATION_TOL
        negative = profile < -VALIDATION_TOL

        maximum_excess = max(
            0.0,
            float((profile - static_pmax).max()),
        )

        generator_profile_records.append(
            {
                "GEN UID": generator_uid,
                "Resource": resource,
                "Static PMax MW": static_pmax,
                "Minimum Time-Series MW": profile.min(),
                "Maximum Time-Series MW": profile.max(),
                "Maximum Availability Factor":
                    profile.max() / static_pmax,
                "Missing Values": int(profile.isna().sum()),
                "Negative Values": int(negative.sum()),
                "Values Above Static PMax": int(above_static.sum()),
                "Maximum Excess Above PMax MW": maximum_excess,
            }
        )

generator_profile_audit = pd.DataFrame(generator_profile_records)

resource_profile_audit = (
    generator_profile_audit
    .groupby("Resource", sort=True)
    .agg(
        Number_of_Generators=("GEN UID", "count"),
        Total_Static_PMax_MW=("Static PMax MW", "sum"),
        Smallest_Observed_MW=("Minimum Time-Series MW", "min"),
        Largest_Observed_MW=("Maximum Time-Series MW", "max"),
        Maximum_Availability_Factor=(
            "Maximum Availability Factor",
            "max",
        ),
        Missing_Values=("Missing Values", "sum"),
        Negative_Values=("Negative Values", "sum"),
        Values_Above_Static_PMax=(
            "Values Above Static PMax",
            "sum",
        ),
        Maximum_Excess_Above_PMax_MW=(
            "Maximum Excess Above PMax MW",
            "max",
        ),
    )
)

display(resource_profile_audit)

problematic_generator_profiles = generator_profile_audit.loc[
    (generator_profile_audit["Missing Values"] > 0)
    | (generator_profile_audit["Negative Values"] > 0)
    | (generator_profile_audit["Values Above Static PMax"] > 0)
].copy()

print(
    "Generator profiles with a validation problem:",
    len(problematic_generator_profiles),
)

if not problematic_generator_profiles.empty:
    display(problematic_generator_profiles)

print(
    "All files aligned with demand chronology:",
    all(
        profile.index.equals(regional_load_mw.index)
        for profile in generator_timeseries_aligned.values()
    ),
)

print(
    "Total missing values:",
    generator_profile_audit["Missing Values"].sum(),
)

print(
    "Total negative values:",
    generator_profile_audit["Negative Values"].sum(),
)

print(
    "Total values above static PMax:",
    generator_profile_audit["Values Above Static PMax"].sum(),
)

assert len(generator_profile_audit) == 80
assert generator_profile_audit["GEN UID"].nunique() == 80
assert generator_profile_audit["Missing Values"].sum() == 0
assert generator_profile_audit["Negative Values"].sum() == 0
assert generator_profile_audit["Values Above Static PMax"].sum() == 0

print("\nAll hourly generator profiles passed MW-based validation.")

## 3. Complete hourly generator-availability matrix

The generator-availability matrix contains one row per hour and one column per
generator:

$$
\mathbf{\overline{P}}
\in
\mathbb{R}^{8784 \times 158}
$$

For generators with an RTS-GMLC day-ahead availability series:

$$
\overline{P}_{g,t}
=
A_{g,t}^{\mathrm{RTS}}
$$

For generators without an hourly series:

$$
\overline{P}_{g,t}
=
P_g^{\mathrm{static}}
$$

Storage and synchronous condensers are exceptions:

$$
\overline{P}_{g,t}=0
$$

Storage is fixed at zero because the independent-hour model has no state-of-charge
constraint. Otherwise, storage could generate energy without accounting for
charging. Synchronous condensers do not provide active-power generation.

No availability values are clipped or rescaled because the previous audit
established that every raw hourly value is already measured in MW and satisfies:

$$
0
\leq
A_{g,t}^{\mathrm{RTS}}
\leq
P_g^{\mathrm{static}}
$$

In [ ]:
generator_ids = gen["GEN UID"].tolist()

assert len(generator_ids) == 158
assert len(set(generator_ids)) == 158

generator_metadata = (
    gen
    .set_index("GEN UID")
    .loc[
        generator_ids,
        [
            "Bus ID",
            "Unit Type",
            "Fuel",
            "PMax MW",
        ],
    ]
    .copy()
)

static_generator_pmax = (
    generator_metadata["PMax MW"]
    .astype(float)
)

assert static_generator_pmax.notna().all()
assert static_generator_pmax.ge(0).all()

# Begin with static PMax for every generator in every hour.
hourly_gen_pmax = pd.DataFrame(
    np.broadcast_to(
        static_generator_pmax.to_numpy(),
        (len(regional_load_mw.index), len(generator_ids)),
    ).copy(),
    index=regional_load_mw.index,
    columns=pd.Index(generator_ids, name="GEN UID"),
)

# Storage and synchronous condensers receive zero active-power availability.
zero_active_power_fuels = {"Storage", "Sync_Cond"}

zero_active_power_generators = generator_metadata.index[
    generator_metadata["Fuel"].isin(zero_active_power_fuels)
].tolist()

hourly_gen_pmax.loc[:, zero_active_power_generators] = 0.0

# Overwrite the appropriate columns with raw hourly MW time series.
timeseries_generator_ids = set()

for data_file, pointer_group in day_ahead_pmax_pointers.groupby(
    "Data File",
    sort=True,
):
    aligned_profiles = generator_timeseries_aligned[data_file]

    for generator_uid in pointer_group["Object"]:
        hourly_gen_pmax.loc[:, generator_uid] = (
            aligned_profiles[generator_uid].to_numpy(dtype=float)
        )

        timeseries_generator_ids.add(generator_uid)

print("Hourly generator-availability shape:", hourly_gen_pmax.shape)
print(
    "Generators using hourly time series:",
    len(timeseries_generator_ids),
)
print(
    "Generators fixed at zero active power:",
    len(zero_active_power_generators),
)
print(
    "Generators using static PMax:",
    len(generator_ids)
    - len(timeseries_generator_ids)
    - len(zero_active_power_generators),
)

In [ ]:
availability_array = hourly_gen_pmax.to_numpy(dtype=float)
static_pmax_array = static_generator_pmax.to_numpy(dtype=float)

missing_availability_values = int(
    hourly_gen_pmax.isna().sum().sum()
)

negative_availability_values = int(
    (hourly_gen_pmax < -VALIDATION_TOL).sum().sum()
)

above_static_pmax_values = int(
    (
        availability_array
        > static_pmax_array[np.newaxis, :] + VALIDATION_TOL
    ).sum()
)

maximum_excess_above_static = max(
    0.0,
    float(
        (
            availability_array
            - static_pmax_array[np.newaxis, :]
        ).max()
    ),
)

print("Matrix shape:", hourly_gen_pmax.shape)
print("Unique timestamps:", hourly_gen_pmax.index.nunique())
print("Unique generator columns:", hourly_gen_pmax.columns.nunique())
print("Missing values:", missing_availability_values)
print("Negative values:", negative_availability_values)
print("Values above static PMax:", above_static_pmax_values)
print(
    "Maximum excess above static PMax:",
    maximum_excess_above_static,
    "MW",
)

assert hourly_gen_pmax.shape == (8784, 158)
assert hourly_gen_pmax.index.equals(regional_load_mw.index)
assert hourly_gen_pmax.columns.is_unique
assert set(hourly_gen_pmax.columns) == set(gen["GEN UID"])

assert missing_availability_values == 0
assert negative_availability_values == 0
assert above_static_pmax_values == 0

assert len(timeseries_generator_ids) == 80
assert len(zero_active_power_generators) == 4

assert (
    hourly_gen_pmax[zero_active_power_generators]
    .abs()
    .le(VALIDATION_TOL)
    .all()
    .all()
)

print("\nComplete hourly generator-availability matrix passed validation.")

In [ ]:
generator_treatment = generator_metadata.reset_index().copy()

generator_treatment["Availability Treatment"] = np.select(
    [
        generator_treatment["GEN UID"].isin(
            timeseries_generator_ids
        ),
        generator_treatment["GEN UID"].isin(
            zero_active_power_generators
        ),
    ],
    [
        "Hourly DAY_AHEAD MW series",
        "Fixed at zero active power",
    ],
    default="Static PMax MW",
)

generator_treatment_audit = (
    generator_treatment
    .groupby("Availability Treatment", sort=True)
    .agg(
        Number_of_Generators=("GEN UID", "count"),
        Total_Static_PMax_MW=("PMax MW", "sum"),
    )
)

display(generator_treatment_audit)

assert generator_treatment_audit["Number_of_Generators"].sum() == 158

In [ ]:
fuel_order = [
    "Nuclear",
    "Coal",
    "NG",
    "Oil",
    "Hydro",
    "Wind",
    "Solar",
    "Storage",
    "Sync_Cond",
]

hourly_availability_by_fuel = pd.DataFrame(
    {
        fuel: hourly_gen_pmax[
            generator_metadata.index[
                generator_metadata["Fuel"].eq(fuel)
            ]
        ].sum(axis=1)
        for fuel in fuel_order
    },
    index=hourly_gen_pmax.index,
)

installed_capacity_by_fuel = (
    generator_metadata
    .groupby("Fuel")["PMax MW"]
    .sum()
    .reindex(fuel_order)
)

annual_availability_summary = pd.DataFrame(
    {
        "Installed PMax MW": installed_capacity_by_fuel,
        "Minimum Available MW":
            hourly_availability_by_fuel.min(),
        "Mean Available MW":
            hourly_availability_by_fuel.mean(),
        "Maximum Available MW":
            hourly_availability_by_fuel.max(),
    }
)

annual_availability_summary["Mean Availability Factor"] = np.where(
    annual_availability_summary["Installed PMax MW"] > 0,
    annual_availability_summary["Mean Available MW"]
    / annual_availability_summary["Installed PMax MW"],
    np.nan,
)

display(annual_availability_summary)

In [ ]:
notebook_3_peak_timestamp = pd.Timestamp(
    "2020-08-26 14:00:00"
)

peak_availability_by_fuel = (
    hourly_availability_by_fuel
    .loc[notebook_3_peak_timestamp]
    .rename("Available PMax MW")
    .to_frame()
)

peak_availability_by_fuel["Installed PMax MW"] = (
    installed_capacity_by_fuel
)

peak_availability_by_fuel["Available / Installed"] = np.where(
    peak_availability_by_fuel["Installed PMax MW"] > 0,
    peak_availability_by_fuel["Available PMax MW"]
    / peak_availability_by_fuel["Installed PMax MW"],
    np.nan,
)

display(peak_availability_by_fuel)

peak_total_available = hourly_gen_pmax.loc[
    notebook_3_peak_timestamp
].sum()

peak_total_demand = system_load_from_buses.loc[
    notebook_3_peak_timestamp
]

peak_aggregate_surplus = (
    peak_total_available - peak_total_demand
)

print(
    "Peak-hour total available generation:",
    f"{peak_total_available:,.6f} MW",
)
print(
    "Peak-hour total demand:",
    f"{peak_total_demand:,.6f} MW",
)
print(
    "Peak-hour aggregate surplus:",
    f"{peak_aggregate_surplus:,.6f} MW",
)
print(
    "Peak-hour availability/demand ratio:",
    f"{peak_total_available / peak_total_demand:.6f}",
)

expected_peak_availability = {
    "Nuclear": 400.0,
    "Coal": 2317.0,
    "NG": 5035.0,
    "Oil": 324.0,
    "Hydro": 718.6,
    "Wind": 677.1,
    "Solar": 1458.8,
    "Storage": 0.0,
    "Sync_Cond": 0.0,
}

for fuel, expected_mw in expected_peak_availability.items():
    actual_mw = peak_availability_by_fuel.loc[
        fuel,
        "Available PMax MW",
    ]

    assert np.isclose(
        actual_mw,
        expected_mw,
        atol=1e-6,
    ), (
        f"{fuel}: expected {expected_mw} MW, "
        f"found {actual_mw} MW"
    )

assert np.isclose(
    peak_total_available,
    10930.5,
    atol=1e-6,
)

print("\nNotebook 3 peak-hour availability reproduced exactly.")

In [ ]:
print(
    "Reconstructed bus-level load:",
    bus_peak["Peak Load MW"].sum()
)

print(
    "Original regional system load:",
    peak_row["System Load MW"]
)

print(
    "Reconstruction error:",
    abs(
        bus_peak["Peak Load MW"].sum()
        -
        peak_row["System Load MW"]
    )
)

In [ ]:
hourly_total_available = hourly_gen_pmax.sum(axis=1)

hourly_aggregate_margin = (
    hourly_total_available - system_load_from_buses
)

aggregate_margin_summary = pd.Series(
    {
        "Minimum total availability MW":
            hourly_total_available.min(),
        "Mean total availability MW":
            hourly_total_available.mean(),
        "Maximum total availability MW":
            hourly_total_available.max(),
        "Minimum aggregate margin MW":
            hourly_aggregate_margin.min(),
        "Mean aggregate margin MW":
            hourly_aggregate_margin.mean(),
        "Maximum aggregate margin MW":
            hourly_aggregate_margin.max(),
        "Hours with negative aggregate margin":
            int((hourly_aggregate_margin < -VALIDATION_TOL).sum()),
    },
    name="Value",
)

display(aggregate_margin_summary.to_frame())

minimum_margin_timestamp = hourly_aggregate_margin.idxmin()

print("Minimum-margin timestamp:", minimum_margin_timestamp)
print(
    "Demand at minimum-margin timestamp:",
    f"{system_load_from_buses.loc[minimum_margin_timestamp]:,.6f} MW",
)
print(
    "Availability at minimum-margin timestamp:",
    f"{hourly_total_available.loc[minimum_margin_timestamp]:,.6f} MW",
)
print(
    "Aggregate margin:",
    f"{hourly_aggregate_margin.loc[minimum_margin_timestamp]:,.6f} MW",
)

## 4. Generator cost and network preparation

The independent hourly DC-OPF minimizes variable generation cost:

$$
\min
\sum_{g \in G} c_g P_{g,t}
$$

where:

- `P_g,t` is the dispatch of generator `g` in hour `t`;
- `c_g` is the linear cost coefficient in dollars per MWh.

As established in Notebook 3, the original RTS-GMLC heat-rate curves cannot
be represented as independent LP blocks because the first average-heat-rate
block can be more expensive than later incremental blocks. Independent blocks
would allow the optimizer to use later blocks without filling earlier ones.

Therefore, this notebook preserves the Notebook 3 approximation:

$$
c_g
=
\frac{C_g\left(P_g^{\max}\right)}
{P_g^{\max}}
$$

This coefficient is the generator's full-load average variable cost. It
preserves the total modeled cost of operating the generator at full output but
does not reproduce its exact non-convex production-cost curve.

The hourly objective becomes:

$$
C_t
=
\sum_{g \in G} c_g P_{g,t}
$$

In [ ]:
static_availability_audit = (
    generator_treatment
    .loc[
        generator_treatment["Availability Treatment"]
        .ne("Hourly DAY_AHEAD MW series")
    ]
    .groupby(
        [
            "Availability Treatment",
            "Fuel",
            "Unit Type",
        ],
        sort=True,
    )
    .agg(
        Number_of_Generators=("GEN UID", "count"),
        Total_Static_PMax_MW=("PMax MW", "sum"),
    )
)

display(static_availability_audit)

static_solar_generators = generator_treatment.loc[
    generator_treatment["Fuel"].eq("Solar")
    & generator_treatment["Availability Treatment"].eq(
        "Static PMax MW"
    ),
    [
        "GEN UID",
        "Bus ID",
        "Unit Type",
        "Fuel",
        "PMax MW",
        "Availability Treatment",
    ],
]

print("Solar generators using static availability:")
display(static_solar_generators)

In [ ]:
THERMAL_CURVE_FUELS = {
    "Coal",
    "NG",
    "Oil",
    "Nuclear",
}

OUTPUT_PERCENT_COLUMNS = [
    "Output_pct_0",
    "Output_pct_1",
    "Output_pct_2",
    "Output_pct_3",
    "Output_pct_4",
]

HEAT_RATE_COLUMNS = [
    "HR_avg_0",
    "HR_incr_1",
    "HR_incr_2",
    "HR_incr_3",
    "HR_incr_4",
]


def calculate_full_load_average_cost(row):
    """
    Calculate one linear full-load-average variable-cost coefficient.

    Coal, NG, Oil, and Nuclear generators use their RTS-GMLC
    heat-rate curves.

    Hydro, Wind, Solar, Storage, and Sync_Cond use VOM only.
    Numeric heat-rate placeholders for these resources are not
    interpreted as thermal production-cost curves.
    """
    generator_uid = row["GEN UID"]
    fuel = str(row["Fuel"]).strip()
    pmax = float(row["PMax MW"])

    vom = (
        0.0
        if pd.isna(row["VOM"])
        else float(row["VOM"])
    )

    if pmax <= VALIDATION_TOL:
        return 0.0

    # Non-thermal resources do not use the heat-rate integration.
    if fuel not in THERMAL_CURVE_FUELS:
        return vom

    fuel_price = row["Fuel Price $/MMBTU"]

    if pd.isna(fuel_price):
        raise ValueError(
            f"{generator_uid} is a {fuel} generator "
            "but has no fuel price."
        )

    fuel_price = float(fuel_price)

    first_output_fraction = row["Output_pct_0"]
    first_average_heat_rate = row["HR_avg_0"]

    if (
        pd.isna(first_output_fraction)
        or pd.isna(first_average_heat_rate)
    ):
        raise ValueError(
            f"{generator_uid} has an incomplete first cost block."
        )

    first_output_fraction = float(first_output_fraction)
    first_average_heat_rate = float(first_average_heat_rate)

    if not 0 <= first_output_fraction <= 1:
        raise ValueError(
            f"{generator_uid} has invalid Output_pct_0 = "
            f"{first_output_fraction}."
        )

    first_block_mw = (
        first_output_fraction * pmax
    )

    first_block_marginal_cost = (
        first_average_heat_rate
        / 1000.0
        * fuel_price
        + vom
    )

    total_full_load_cost = (
        first_block_mw
        * first_block_marginal_cost
    )

    previous_fraction = first_output_fraction

    for segment_number in range(1, 5):
        output_fraction = row[
            f"Output_pct_{segment_number}"
        ]

        incremental_heat_rate = row[
            f"HR_incr_{segment_number}"
        ]

        if (
            pd.isna(output_fraction)
            and pd.isna(incremental_heat_rate)
        ):
            continue

        if (
            pd.isna(output_fraction)
            or pd.isna(incremental_heat_rate)
        ):
            raise ValueError(
                f"{generator_uid} has an incomplete cost segment "
                f"{segment_number}."
            )

        output_fraction = float(output_fraction)
        incremental_heat_rate = float(
            incremental_heat_rate
        )

        if output_fraction < previous_fraction:
            raise ValueError(
                f"{generator_uid} has decreasing output "
                f"breakpoints in segment {segment_number}."
            )

        if output_fraction > 1 + VALIDATION_TOL:
            raise ValueError(
                f"{generator_uid} has an output breakpoint "
                f"above 100%."
            )

        segment_width_mw = (
            output_fraction - previous_fraction
        ) * pmax

        segment_marginal_cost = (
            incremental_heat_rate
            / 1000.0
            * fuel_price
            + vom
        )

        total_full_load_cost += (
            segment_width_mw
            * segment_marginal_cost
        )

        previous_fraction = output_fraction

    if not np.isclose(
        previous_fraction,
        1.0,
        atol=1e-9,
    ):
        raise ValueError(
            f"{generator_uid} cost curve ends at "
            f"{previous_fraction:.6f}, not 1.0."
        )

    return total_full_load_cost / pmax


gen_cost_data = gen.copy()

gen_cost_data["Cost Method"] = np.where(
    gen_cost_data["Fuel"].isin(THERMAL_CURVE_FUELS),
    "Full-load-average heat-rate cost",
    "VOM only",
)

gen_cost_data["Linear Cost $/MWh"] = gen_cost_data.apply(
    calculate_full_load_average_cost,
    axis=1,
)

generator_cost = (
    gen_cost_data
    .set_index("GEN UID")["Linear Cost $/MWh"]
    .to_dict()
)

cost_method_audit = (
    gen_cost_data
    .groupby(
        ["Cost Method", "Fuel"],
        sort=True,
    )
    .agg(
        Number_of_Generators=("GEN UID", "count"),
        Total_PMax_MW=("PMax MW", "sum"),
        Minimum_Cost_per_MWh=(
            "Linear Cost $/MWh",
            "min",
        ),
        Maximum_Cost_per_MWh=(
            "Linear Cost $/MWh",
            "max",
        ),
    )
)

display(cost_method_audit)

print("Cost coefficients created:", len(generator_cost))
print(
    "Missing cost coefficients:",
    gen_cost_data["Linear Cost $/MWh"].isna().sum(),
)
print(
    "Negative cost coefficients:",
    (
        gen_cost_data["Linear Cost $/MWh"]
        < -VALIDATION_TOL
    ).sum(),
)

assert len(generator_cost) == 158
assert gen_cost_data["Linear Cost $/MWh"].notna().all()
assert (
    gen_cost_data["Linear Cost $/MWh"]
    >= -VALIDATION_TOL
).all()

print("\nAll linear generator costs were reconstructed.")

In [ ]:
cost_summary_by_fuel = (
    gen_cost_data
    .groupby("Fuel", sort=True)
    .agg(
        Number_of_Generators=("GEN UID", "count"),
        Total_PMax_MW=("PMax MW", "sum"),
        Minimum_Cost_per_MWh=(
            "Linear Cost $/MWh",
            "min",
        ),
        Mean_Cost_per_MWh=(
            "Linear Cost $/MWh",
            "mean",
        ),
        Maximum_Cost_per_MWh=(
            "Linear Cost $/MWh",
            "max",
        ),
    )
)

display(cost_summary_by_fuel)

representative_costs = (
    gen_cost_data[
        [
            "GEN UID",
            "Bus ID",
            "Fuel",
            "Unit Type",
            "PMax MW",
            "Linear Cost $/MWh",
        ]
    ]
    .sort_values(
        ["Linear Cost $/MWh", "GEN UID"]
    )
    .reset_index(drop=True)
)

print("Fifteen least-expensive generators:")
display(representative_costs.head(15))

print("Fifteen most-expensive generators:")
display(representative_costs.tail(15))

nuclear_costs = gen_cost_data.loc[
    gen_cost_data["Fuel"].eq("Nuclear"),
    ["GEN UID", "Linear Cost $/MWh"],
]

display(nuclear_costs)

assert len(nuclear_costs) == 1
assert np.isclose(
    nuclear_costs["Linear Cost $/MWh"].iloc[0],
    8.022465,
    atol=1e-6,
)

print("Notebook 3 nuclear-cost regression check passed.")

In [ ]:
BUS_IDS = bus["Bus ID"].astype(int).tolist()
BRANCH_IDS = branch["UID"].astype(str).tolist()
GENERATOR_IDS = gen["GEN UID"].tolist()

assert len(BUS_IDS) == 73
assert len(set(BUS_IDS)) == 73

assert len(BRANCH_IDS) == 120
assert len(set(BRANCH_IDS)) == 120

assert len(GENERATOR_IDS) == 158
assert len(set(GENERATOR_IDS)) == 158

branch_from_bus = (
    branch
    .assign(UID=branch["UID"].astype(str))
    .set_index("UID")["From Bus"]
    .astype(int)
    .to_dict()
)

branch_to_bus = (
    branch
    .assign(UID=branch["UID"].astype(str))
    .set_index("UID")["To Bus"]
    .astype(int)
    .to_dict()
)

branch_reactance = (
    branch
    .assign(UID=branch["UID"].astype(str))
    .set_index("UID")["X"]
    .astype(float)
    .to_dict()
)

branch_rating = (
    branch
    .assign(UID=branch["UID"].astype(str))
    .set_index("UID")["Cont Rating"]
    .astype(float)
    .to_dict()
)

generator_bus = (
    gen
    .set_index("GEN UID")["Bus ID"]
    .astype(int)
    .to_dict()
)

generators_at_bus = {
    bus_id: [
        generator_uid
        for generator_uid in GENERATOR_IDS
        if generator_bus[generator_uid] == bus_id
    ]
    for bus_id in BUS_IDS
}

outgoing_branches = {
    bus_id: [
        branch_uid
        for branch_uid in BRANCH_IDS
        if branch_from_bus[branch_uid] == bus_id
    ]
    for bus_id in BUS_IDS
}

incoming_branches = {
    bus_id: [
        branch_uid
        for branch_uid in BRANCH_IDS
        if branch_to_bus[branch_uid] == bus_id
    ]
    for bus_id in BUS_IDS
}

assert all(
    branch_from_bus[line] in BUS_IDS
    and branch_to_bus[line] in BUS_IDS
    for line in BRANCH_IDS
)

assert all(
    branch_reactance[line] > 0
    for line in BRANCH_IDS
)

assert all(
    branch_rating[line] > 0
    for line in BRANCH_IDS
)

assert all(
    generator_bus[g] in BUS_IDS
    for g in GENERATOR_IDS
)

print("Buses:", len(BUS_IDS))
print("Physical branches:", len(BRANCH_IDS))
print("Generators:", len(GENERATOR_IDS))
print(
    "Buses containing generation:",
    sum(len(generators_at_bus[b]) > 0 for b in BUS_IDS),
)
print(
    "Maximum generators at one bus:",
    max(len(generators_at_bus[b]) for b in BUS_IDS),
)

In [ ]:
parallel_branch_audit = branch.copy()

parallel_branch_audit["Bus Pair"] = parallel_branch_audit.apply(
    lambda row: tuple(
        sorted(
            (
                int(row["From Bus"]),
                int(row["To Bus"]),
            )
        )
    ),
    axis=1,
)

parallel_pairs = (
    parallel_branch_audit
    .groupby("Bus Pair", sort=True)
    .agg(
        Number_of_Physical_Branches=("UID", "count"),
        Branch_UIDs=("UID", lambda x: list(x)),
        Total_Continuous_Rating_MW=("Cont Rating", "sum"),
    )
)

parallel_pairs = parallel_pairs.loc[
    parallel_pairs["Number_of_Physical_Branches"] > 1
]

display(parallel_pairs)

print("Parallel bus pairs:", len(parallel_pairs))
print(
    "Physical branches belonging to parallel pairs:",
    parallel_pairs["Number_of_Physical_Branches"].sum(),
)

assert len(parallel_pairs) == 12
assert (
    parallel_pairs["Number_of_Physical_Branches"] == 2
).all()

print("\nAll 120 physical branches remain individually represented.")

## 5. Reusable independent-hour DC optimal power flow

For every hour `t`, the model minimizes variable generation cost:

$$
\min
C_t
=
\sum_{g \in G} c_g P_{g,t}
$$

subject to hourly generator availability:

$$
0
\leq
P_{g,t}
\leq
\overline{P}_{g,t}
$$

DC power flow on branch `l` connecting buses `i` and `j`:

$$
f_{l,t}
=
\frac{\theta_{i,t}-\theta_{j,t}}{X_l}
S_{\mathrm{base}}
$$

Separate upper and lower thermal limits:

$$
f_{l,t}
\leq
F_l^{\max}
$$

$$
-f_{l,t}
\leq
F_l^{\max}
$$

Nodal power balance at every bus `i`:

$$
\sum_{g \in G_i} P_{g,t}
-
D_{i,t}
=
\sum_{l \in L_i^{\mathrm{out}}} f_{l,t}
-
\sum_{l \in L_i^{\mathrm{in}}} f_{l,t}
$$

Reference-angle constraint:

$$
\theta_{101,t}=0
$$

Demand and generator availability are mutable parameters. The network,
generator costs, and branch ratings remain fixed during the native-system
baseline.

Each hour is optimized independently. Therefore, dispatch in one hour does not
constrain dispatch in the preceding or following hour.

In [ ]:
available_solver_names = []

for candidate in ["appsi_highs", "highs"]:
    try:
        candidate_solver = pyo.SolverFactory(candidate)

        if candidate_solver.available(exception_flag=False):
            available_solver_names.append(candidate)
    except Exception:
        pass

print("Available HiGHS interfaces:", available_solver_names)

if not available_solver_names:
    raise RuntimeError(
        "No usable HiGHS solver interface was found."
    )

HOURLY_SOLVER_NAME = available_solver_names[0]
hourly_solver = pyo.SolverFactory(HOURLY_SOLVER_NAME)

print("Selected solver interface:", HOURLY_SOLVER_NAME)

In [ ]:
def build_hourly_rts_opf():
    model = pyo.ConcreteModel(
        name="RTS-GMLC Independent-Hour DC-OPF"
    )

    # Sets
    model.BUS = pyo.Set(
        initialize=BUS_IDS,
        ordered=True,
    )

    model.GENERATOR = pyo.Set(
        initialize=GENERATOR_IDS,
        ordered=True,
    )

    model.BRANCH = pyo.Set(
        initialize=BRANCH_IDS,
        ordered=True,
    )

    # Fixed parameters
    model.cost = pyo.Param(
        model.GENERATOR,
        initialize=generator_cost,
        within=pyo.NonNegativeReals,
    )

    model.reactance = pyo.Param(
        model.BRANCH,
        initialize=branch_reactance,
        within=pyo.PositiveReals,
    )

    # Mutable parameters
    model.demand = pyo.Param(
        model.BUS,
        initialize=0.0,
        within=pyo.NonNegativeReals,
        mutable=True,
    )

    model.available_pmax = pyo.Param(
        model.GENERATOR,
        initialize=0.0,
        within=pyo.NonNegativeReals,
        mutable=True,
    )

    model.line_limit = pyo.Param(
        model.BRANCH,
        initialize=branch_rating,
        within=pyo.PositiveReals,
        mutable=True,
    )

    # Variables
    model.generation = pyo.Var(
        model.GENERATOR,
        domain=pyo.NonNegativeReals,
    )

    model.theta = pyo.Var(
        model.BUS,
        domain=pyo.Reals,
    )

    model.flow = pyo.Var(
        model.BRANCH,
        domain=pyo.Reals,
    )

    # Objective
    model.total_cost = pyo.Objective(
        expr=sum(
            model.cost[g] * model.generation[g]
            for g in model.GENERATOR
        ),
        sense=pyo.minimize,
    )

    # Generator availability
    def generator_upper_rule(m, g):
        return (
            m.generation[g]
            <= m.available_pmax[g]
        )

    model.generator_upper = pyo.Constraint(
        model.GENERATOR,
        rule=generator_upper_rule,
    )

    # DC branch-flow equations
    def dc_flow_rule(m, line):
        from_bus = branch_from_bus[line]
        to_bus = branch_to_bus[line]

        return m.flow[line] == (
            (
                m.theta[from_bus]
                - m.theta[to_bus]
            )
            / m.reactance[line]
            * S_BASE
        )

    model.dc_flow = pyo.Constraint(
        model.BRANCH,
        rule=dc_flow_rule,
    )

    # Separate directional thermal limits
    def line_upper_rule(m, line):
        return m.flow[line] <= m.line_limit[line]

    def line_lower_rule(m, line):
        return -m.flow[line] <= m.line_limit[line]

    model.line_upper = pyo.Constraint(
        model.BRANCH,
        rule=line_upper_rule,
    )

    model.line_lower = pyo.Constraint(
        model.BRANCH,
        rule=line_lower_rule,
    )

    # Nodal power balance
    def nodal_balance_rule(m, bus_id):
        generation_at_bus = sum(
            m.generation[g]
            for g in generators_at_bus[bus_id]
        )

        net_outgoing_flow = (
            sum(
                m.flow[line]
                for line in outgoing_branches[bus_id]
            )
            - sum(
                m.flow[line]
                for line in incoming_branches[bus_id]
            )
        )

        return (
            generation_at_bus
            - m.demand[bus_id]
            == net_outgoing_flow
        )

    model.nodal_balance = pyo.Constraint(
        model.BUS,
        rule=nodal_balance_rule,
    )

    # Reference bus
    model.reference_angle = pyo.Constraint(
        expr=model.theta[101] == 0.0
    )

    # Import LP dual values after solving
    model.dual = pyo.Suffix(
        direction=pyo.Suffix.IMPORT
    )

    return model


hourly_model = build_hourly_rts_opf()

model_structure_audit = pd.Series(
    {
        "Generation variables":
            len(hourly_model.generation),
        "Angle variables":
            len(hourly_model.theta),
        "Flow variables":
            len(hourly_model.flow),
        "Generator upper constraints":
            len(hourly_model.generator_upper),
        "DC-flow constraints":
            len(hourly_model.dc_flow),
        "Upper line-limit constraints":
            len(hourly_model.line_upper),
        "Lower line-limit constraints":
            len(hourly_model.line_lower),
        "Nodal-balance constraints":
            len(hourly_model.nodal_balance),
        "Reference-angle constraints":
            len(hourly_model.reference_angle),
    },
    name="Count",
)

display(model_structure_audit.to_frame())

assert len(hourly_model.generation) == 158
assert len(hourly_model.theta) == 73
assert len(hourly_model.flow) == 120
assert len(hourly_model.generator_upper) == 158
assert len(hourly_model.dc_flow) == 120
assert len(hourly_model.line_upper) == 120
assert len(hourly_model.line_lower) == 120
assert len(hourly_model.nodal_balance) == 73
assert len(hourly_model.reference_angle) == 1

print("Reusable DC-OPF structure passed validation.")

In [ ]:
def load_hour_into_model(model, timestamp):
    timestamp = pd.Timestamp(timestamp)

    if timestamp not in hourly_bus_load.index:
        raise KeyError(
            f"Demand data are unavailable for {timestamp}."
        )

    if timestamp not in hourly_gen_pmax.index:
        raise KeyError(
            f"Generator availability is unavailable for {timestamp}."
        )

    demand_row = hourly_bus_load.loc[timestamp]
    availability_row = hourly_gen_pmax.loc[timestamp]

    for bus_id in BUS_IDS:
        demand_value = float(demand_row.loc[bus_id])

        if demand_value < -VALIDATION_TOL:
            raise ValueError(
                f"Negative demand at bus {bus_id}."
            )

        model.demand[bus_id] = max(
            0.0,
            demand_value,
        )

    for generator_uid in GENERATOR_IDS:
        availability_value = float(
            availability_row.loc[generator_uid]
        )

        if availability_value < -VALIDATION_TOL:
            raise ValueError(
                f"Negative availability for {generator_uid}."
            )

        model.available_pmax[generator_uid] = max(
            0.0,
            availability_value,
        )

    return timestamp


def solve_loaded_hour(model, solver, tee=False):
    results = solver.solve(
        model,
        tee=tee,
    )

    status = results.solver.status
    termination = results.solver.termination_condition

    if (
        status != SolverStatus.ok
        or termination != TerminationCondition.optimal
    ):
        raise RuntimeError(
            "Hourly OPF did not solve optimally. "
            f"Status: {status}; termination: {termination}"
        )

    return results

In [ ]:
def validate_hourly_solution(
    model,
    timestamp,
    tolerance=VALIDATION_TOL,
):
    timestamp = pd.Timestamp(timestamp)

    generation = {
        g: pyo.value(model.generation[g])
        for g in GENERATOR_IDS
    }

    theta = {
        b: pyo.value(model.theta[b])
        for b in BUS_IDS
    }

    flow = {
        line: pyo.value(model.flow[line])
        for line in BRANCH_IDS
    }

    demand = {
        b: pyo.value(model.demand[b])
        for b in BUS_IDS
    }

    available_pmax = {
        g: pyo.value(model.available_pmax[g])
        for g in GENERATOR_IDS
    }

    line_limit = {
        line: pyo.value(model.line_limit[line])
        for line in BRANCH_IDS
    }

    total_generation = sum(generation.values())
    total_demand = sum(demand.values())

    generator_violations = [
        max(
            0.0,
            -generation[g],
            generation[g] - available_pmax[g],
        )
        for g in GENERATOR_IDS
    ]

    thermal_violations = [
        max(
            0.0,
            abs(flow[line]) - line_limit[line],
        )
        for line in BRANCH_IDS
    ]

    dc_flow_errors = []

    for line in BRANCH_IDS:
        calculated_flow = (
            (
                theta[branch_from_bus[line]]
                - theta[branch_to_bus[line]]
            )
            / branch_reactance[line]
            * S_BASE
        )

        dc_flow_errors.append(
            abs(flow[line] - calculated_flow)
        )

    nodal_balance_errors = []

    for bus_id in BUS_IDS:
        generation_at_bus = sum(
            generation[g]
            for g in generators_at_bus[bus_id]
        )

        net_outgoing_flow = (
            sum(
                flow[line]
                for line in outgoing_branches[bus_id]
            )
            - sum(
                flow[line]
                for line in incoming_branches[bus_id]
            )
        )

        nodal_error = (
            generation_at_bus
            - demand[bus_id]
            - net_outgoing_flow
        )

        nodal_balance_errors.append(
            abs(nodal_error)
        )

    validation = pd.Series(
        {
            "Timestamp": timestamp,
            "Objective $/h":
                pyo.value(model.total_cost),
            "Total generation MW":
                total_generation,
            "Total demand MW":
                total_demand,
            "System balance error MW":
                abs(total_generation - total_demand),
            "Maximum generator violation MW":
                max(generator_violations),
            "Maximum thermal violation MW":
                max(thermal_violations),
            "Maximum DC-flow error MW":
                max(dc_flow_errors),
            "Maximum nodal-balance error MW":
                max(nodal_balance_errors),
            "Reference-angle error radians":
                abs(theta[101]),
        },
        name="Value",
    )

    numerical_metrics = [
        "System balance error MW",
        "Maximum generator violation MW",
        "Maximum thermal violation MW",
        "Maximum DC-flow error MW",
        "Maximum nodal-balance error MW",
        "Reference-angle error radians",
    ]

    for metric in numerical_metrics:
        if validation.loc[metric] > tolerance:
            raise AssertionError(
                f"{metric} exceeds tolerance: "
                f"{validation.loc[metric]}"
            )

    return validation

In [ ]:
PEAK_TIMESTAMP = pd.Timestamp(
    "2020-08-26 14:00:00"
)

loaded_timestamp = load_hour_into_model(
    hourly_model,
    PEAK_TIMESTAMP,
)

peak_results = solve_loaded_hour(
    hourly_model,
    hourly_solver,
)

print("Loaded timestamp:", loaded_timestamp)
print("Solver status:", peak_results.solver.status)
print(
    "Termination condition:",
    peak_results.solver.termination_condition,
)

peak_validation = validate_hourly_solution(
    hourly_model,
    PEAK_TIMESTAMP,
)

display(peak_validation.to_frame())

In [ ]:
peak_dispatch = pd.DataFrame(
    {
        "GEN UID": GENERATOR_IDS,
        "Dispatch MW": [
            pyo.value(
                hourly_model.generation[g]
            )
            for g in GENERATOR_IDS
        ],
        "Available PMax MW": [
            pyo.value(
                hourly_model.available_pmax[g]
            )
            for g in GENERATOR_IDS
        ],
    }
).merge(
    gen_cost_data[
        [
            "GEN UID",
            "Bus ID",
            "Fuel",
            "Unit Type",
            "Linear Cost $/MWh",
        ]
    ],
    on="GEN UID",
    how="left",
    validate="one_to_one",
)

peak_dispatch_by_fuel = (
    peak_dispatch
    .groupby("Fuel", sort=True)
    .agg(
        Available_PMax_MW=(
            "Available PMax MW",
            "sum",
        ),
        Dispatch_MW=(
            "Dispatch MW",
            "sum",
        ),
    )
)

peak_dispatch_by_fuel["Utilization"] = np.where(
    peak_dispatch_by_fuel["Available_PMax_MW"] > 0,
    peak_dispatch_by_fuel["Dispatch_MW"]
    / peak_dispatch_by_fuel["Available_PMax_MW"],
    np.nan,
)

display(peak_dispatch_by_fuel)

peak_line_results = pd.DataFrame(
    {
        "UID": BRANCH_IDS,
        "From Bus": [
            branch_from_bus[line]
            for line in BRANCH_IDS
        ],
        "To Bus": [
            branch_to_bus[line]
            for line in BRANCH_IDS
        ],
        "Flow MW": [
            pyo.value(hourly_model.flow[line])
            for line in BRANCH_IDS
        ],
        "Rating MW": [
            pyo.value(hourly_model.line_limit[line])
            for line in BRANCH_IDS
        ],
        "Upper-limit dual": [
            hourly_model.dual.get(
                hourly_model.line_upper[line],
                np.nan,
            )
            for line in BRANCH_IDS
        ],
        "Lower-limit dual": [
            hourly_model.dual.get(
                hourly_model.line_lower[line],
                np.nan,
            )
            for line in BRANCH_IDS
        ],
    }
)

peak_line_results["Utilization"] = (
    peak_line_results["Flow MW"].abs()
    / peak_line_results["Rating MW"]
)

peak_line_results["Headroom MW"] = (
    peak_line_results["Rating MW"]
    - peak_line_results["Flow MW"].abs()
)

peak_line_results["Binding"] = (
    peak_line_results["Headroom MW"].abs()
    <= 1e-5
)

display(
    peak_line_results
    .sort_values(
        "Utilization",
        ascending=False,
    )
    .head(10)
)

peak_objective = pyo.value(
    hourly_model.total_cost
)

a27_result = (
    peak_line_results
    .set_index("UID")
    .loc["A27"]
)

print("Peak objective:", f"${peak_objective:,.6f}/h")
print("Binding branches:", int(peak_line_results["Binding"].sum()))
print("A27 flow:", f"{a27_result['Flow MW']:,.6f} MW")
print("A27 utilization:", f"{a27_result['Utilization']:.6%}")
print("A27 headroom:", f"{a27_result['Headroom MW']:,.6f} MW")

In [ ]:
expected_peak_dispatch = {
    "NG": 2620.335957,
    "Coal": 2317.0,
    "Solar": 1458.8,
    "Hydro": 718.6,
    "Wind": 677.1,
    "Nuclear": 400.0,
    "Oil": 0.0,
    "Storage": 0.0,
    "Sync_Cond": 0.0,
}

assert np.isclose(
    peak_objective,
    130429.228095,
    atol=1e-3,
)

for fuel, expected_dispatch in expected_peak_dispatch.items():
    actual_dispatch = peak_dispatch_by_fuel.loc[
        fuel,
        "Dispatch_MW",
    ]

    assert np.isclose(
        actual_dispatch,
        expected_dispatch,
        atol=1e-5,
    ), (
        f"{fuel}: expected {expected_dispatch} MW, "
        f"found {actual_dispatch} MW"
    )

assert int(peak_line_results["Binding"].sum()) == 0

assert np.isclose(
    a27_result["Flow MW"],
    -496.238962,
    atol=1e-5,
)

assert np.isclose(
    a27_result["Utilization"],
    0.992477924,
    atol=1e-7,
)

print("Notebook 3 peak-hour OPF reproduced successfully.")

## 6. Representative-hour validation

Before solving all 8,784 hours, the reusable model is tested under several
distinct system conditions.

The representative hours are selected from the data rather than chosen
arbitrarily:

1. maximum system demand;
2. minimum system demand;
3. minimum aggregate capacity margin;
4. maximum variable-renewable availability;
5. maximum variable-renewable availability relative to demand;
6. maximum residual demand after variable-renewable availability.

Variable renewable energy, abbreviated VRE, includes:

- utility-scale PV;
- rooftop PV;
- wind.

CSP is excluded from the VRE definition because it is currently represented
with static availability. Hydro is reported separately because unused hydro
availability should not automatically be described as renewable curtailment.

The availability-based residual demand is:

$$
R_t
=
D_t
-
A_t^{\mathrm{VRE}}
$$

where:

- `D_t` is system demand;
- `A_VRE_t` is available PV, RTPV, and Wind generation.

This is an availability-screening metric, not optimized net load. Network
constraints may prevent all available VRE from being dispatched.

In [ ]:
variable_renewable_generator_ids = (
    generator_metadata.index[
        generator_metadata["Fuel"].eq("Wind")
        | generator_metadata["Unit Type"].isin(
            ["PV", "RTPV"]
        )
    ]
    .tolist()
)

hydro_generator_ids = (
    generator_metadata.index[
        generator_metadata["Fuel"].eq("Hydro")
    ]
    .tolist()
)

hourly_vre_available = hourly_gen_pmax[
    variable_renewable_generator_ids
].sum(axis=1)

hourly_vre_availability_share = (
    hourly_vre_available
    / system_load_from_buses
)

hourly_residual_demand = (
    system_load_from_buses
    - hourly_vre_available
)

selection_candidates = [
    (
        "Maximum system demand",
        system_load_from_buses.idxmax(),
    ),
    (
        "Minimum system demand",
        system_load_from_buses.idxmin(),
    ),
    (
        "Minimum aggregate capacity margin",
        hourly_aggregate_margin.idxmin(),
    ),
    (
        "Maximum VRE availability",
        hourly_vre_available.idxmax(),
    ),
    (
        "Maximum VRE availability share",
        hourly_vre_availability_share.idxmax(),
    ),
    (
        "Maximum residual demand",
        hourly_residual_demand.idxmax(),
    ),
]

# Combine labels if multiple criteria select the same timestamp.
representative_timestamp_labels = {}

for label, timestamp in selection_candidates:
    timestamp = pd.Timestamp(timestamp)

    representative_timestamp_labels.setdefault(
        timestamp,
        [],
    ).append(label)

representative_hour_records = []

for timestamp, labels in representative_timestamp_labels.items():
    representative_hour_records.append(
        {
            "Timestamp": timestamp,
            "Selection reason": "; ".join(labels),
            "Demand MW":
                system_load_from_buses.loc[timestamp],
            "Total availability MW":
                hourly_total_available.loc[timestamp],
            "Aggregate margin MW":
                hourly_aggregate_margin.loc[timestamp],
            "VRE availability MW":
                hourly_vre_available.loc[timestamp],
            "VRE availability share":
                hourly_vre_availability_share.loc[timestamp],
            "Residual demand MW":
                hourly_residual_demand.loc[timestamp],
        }
    )

representative_hours = (
    pd.DataFrame(representative_hour_records)
    .sort_values("Timestamp")
    .reset_index(drop=True)
)

display(representative_hours)

print(
    "Distinct representative timestamps:",
    len(representative_hours),
)

In [ ]:
def extract_solved_hour(model, timestamp):
    timestamp = pd.Timestamp(timestamp)

    generation = pd.Series(
        {
            g: pyo.value(model.generation[g])
            for g in GENERATOR_IDS
        },
        name="Dispatch MW",
    )

    availability = pd.Series(
        {
            g: pyo.value(model.available_pmax[g])
            for g in GENERATOR_IDS
        },
        name="Available PMax MW",
    )

    dispatch_by_fuel = generation.groupby(
        generator_metadata["Fuel"]
    ).sum()

    availability_by_fuel = availability.groupby(
        generator_metadata["Fuel"]
    ).sum()

    line_results = pd.DataFrame(
        {
            "UID": BRANCH_IDS,
            "From Bus": [
                branch_from_bus[line]
                for line in BRANCH_IDS
            ],
            "To Bus": [
                branch_to_bus[line]
                for line in BRANCH_IDS
            ],
            "Flow MW": [
                pyo.value(model.flow[line])
                for line in BRANCH_IDS
            ],
            "Rating MW": [
                pyo.value(model.line_limit[line])
                for line in BRANCH_IDS
            ],
            "Upper-limit dual": [
                model.dual.get(
                    model.line_upper[line],
                    np.nan,
                )
                for line in BRANCH_IDS
            ],
            "Lower-limit dual": [
                model.dual.get(
                    model.line_lower[line],
                    np.nan,
                )
                for line in BRANCH_IDS
            ],
        }
    )

    line_results["Utilization"] = (
        line_results["Flow MW"].abs()
        / line_results["Rating MW"]
    )

    line_results["Headroom MW"] = (
        line_results["Rating MW"]
        - line_results["Flow MW"].abs()
    )

    line_results["Binding"] = (
        line_results["Headroom MW"].abs()
        <= 1e-5
    )

    line_results = line_results.sort_values(
        "Utilization",
        ascending=False,
    ).reset_index(drop=True)

    vre_available = availability.loc[
        variable_renewable_generator_ids
    ].sum()

    vre_dispatch = generation.loc[
        variable_renewable_generator_ids
    ].sum()

    hydro_available = availability.loc[
        hydro_generator_ids
    ].sum()

    hydro_dispatch = generation.loc[
        hydro_generator_ids
    ].sum()

    summary = {
        "Timestamp": timestamp,
        "Objective $/h":
            pyo.value(model.total_cost),
        "Demand MW":
            sum(pyo.value(model.demand[b]) for b in BUS_IDS),
        "Total generation MW":
            generation.sum(),
        "Total availability MW":
            availability.sum(),
        "VRE available MW":
            vre_available,
        "VRE dispatch MW":
            vre_dispatch,
        "VRE curtailment MW":
            vre_available - vre_dispatch,
        "Hydro available MW":
            hydro_available,
        "Hydro dispatch MW":
            hydro_dispatch,
        "Unused hydro availability MW":
            hydro_available - hydro_dispatch,
        "Maximum line utilization":
            line_results["Utilization"].max(),
        "Most utilized branch":
            line_results.iloc[0]["UID"],
        "Branches >= 90%":
            int((line_results["Utilization"] >= 0.90).sum()),
        "Branches >= 95%":
            int((line_results["Utilization"] >= 0.95).sum()),
        "Branches >= 99%":
            int((line_results["Utilization"] >= 0.99).sum()),
        "Binding branches":
            int(line_results["Binding"].sum()),
    }

    return {
        "summary": summary,
        "generation": generation,
        "availability": availability,
        "dispatch_by_fuel": dispatch_by_fuel,
        "availability_by_fuel": availability_by_fuel,
        "line_results": line_results,
    }

In [ ]:
import time

# Restore all branch limits to their native values.
for line in BRANCH_IDS:
    hourly_model.line_limit[line] = branch_rating[line]

representative_solutions = {}
representative_summary_records = []
representative_validation_records = []
representative_dispatch_by_fuel = {}

total_representative_start = time.perf_counter()

for _, selected_hour in representative_hours.iterrows():
    timestamp = pd.Timestamp(
        selected_hour["Timestamp"]
    )

    selection_reason = selected_hour[
        "Selection reason"
    ]

    load_hour_into_model(
        hourly_model,
        timestamp,
    )

    solve_start = time.perf_counter()

    solve_results = solve_loaded_hour(
        hourly_model,
        hourly_solver,
    )

    solve_seconds = (
        time.perf_counter() - solve_start
    )

    validation = validate_hourly_solution(
        hourly_model,
        timestamp,
    )

    extracted = extract_solved_hour(
        hourly_model,
        timestamp,
    )

    extracted["summary"]["Selection reason"] = (
        selection_reason
    )

    extracted["summary"]["Solve seconds"] = (
        solve_seconds
    )

    representative_solutions[timestamp] = extracted

    representative_summary_records.append(
        extracted["summary"]
    )

    validation_record = validation.to_dict()
    validation_record["Selection reason"] = selection_reason
    validation_record["Solve seconds"] = solve_seconds

    representative_validation_records.append(
        validation_record
    )

    representative_dispatch_by_fuel[timestamp] = (
        extracted["dispatch_by_fuel"]
    )

total_representative_seconds = (
    time.perf_counter()
    - total_representative_start
)

representative_summary = (
    pd.DataFrame(representative_summary_records)
    .sort_values("Timestamp")
    .reset_index(drop=True)
)

representative_validation = (
    pd.DataFrame(representative_validation_records)
    .sort_values("Timestamp")
    .reset_index(drop=True)
)

representative_fuel_dispatch_table = (
    pd.DataFrame(representative_dispatch_by_fuel)
    .T
    .reindex(columns=fuel_order)
)

representative_fuel_dispatch_table.index.name = "Timestamp"

print(
    "Representative hours solved:",
    len(representative_summary),
)
print(
    "Total representative solve time:",
    f"{total_representative_seconds:.4f} seconds",
)
print(
    "Average time per hour:",
    f"{total_representative_seconds / len(representative_summary):.4f} seconds",
)

In [ ]:
display(
    representative_summary[
        [
            "Timestamp",
            "Selection reason",
            "Objective $/h",
            "Demand MW",
            "Total availability MW",
            "VRE available MW",
            "VRE dispatch MW",
            "VRE curtailment MW",
            "Unused hydro availability MW",
            "Maximum line utilization",
            "Most utilized branch",
            "Branches >= 90%",
            "Branches >= 95%",
            "Branches >= 99%",
            "Binding branches",
            "Solve seconds",
        ]
    ]
)

print("Dispatch by fuel:")
display(representative_fuel_dispatch_table)

validation_columns = [
    "Timestamp",
    "System balance error MW",
    "Maximum generator violation MW",
    "Maximum thermal violation MW",
    "Maximum DC-flow error MW",
    "Maximum nodal-balance error MW",
    "Reference-angle error radians",
]

print("Independent validation:")
display(
    representative_validation[
        validation_columns
    ]
)

residual_columns = [
    "System balance error MW",
    "Maximum generator violation MW",
    "Maximum thermal violation MW",
    "Maximum DC-flow error MW",
    "Maximum nodal-balance error MW",
    "Reference-angle error radians",
]

maximum_representative_residual = (
    representative_validation[residual_columns]
    .to_numpy(dtype=float)
    .max()
)

assert len(representative_summary) == len(
    representative_hours
)

assert maximum_representative_residual <= VALIDATION_TOL

assert (
    representative_summary["VRE curtailment MW"]
    >= -VALIDATION_TOL
).all()

assert (
    representative_summary[
        "Unused hydro availability MW"
    ]
    >= -VALIDATION_TOL
).all()

print(
    "Maximum validation residual:",
    maximum_representative_residual,
)

print(
    "All representative-hour solutions passed validation."
)

In [ ]:
load_hour_into_model(
    hourly_model,
    PEAK_TIMESTAMP,
)

peak_recheck_results = solve_loaded_hour(
    hourly_model,
    hourly_solver,
)

peak_recheck_validation = validate_hourly_solution(
    hourly_model,
    PEAK_TIMESTAMP,
)

peak_recheck_objective = pyo.value(
    hourly_model.total_cost
)

peak_recheck_a27_flow = pyo.value(
    hourly_model.flow["A27"]
)

print(
    "Peak recheck objective:",
    f"${peak_recheck_objective:,.6f}/h",
)

print(
    "Peak recheck A27 flow:",
    f"{peak_recheck_a27_flow:,.6f} MW",
)

assert np.isclose(
    peak_recheck_objective,
    130429.228095,
    atol=1e-3,
)

assert np.isclose(
    peak_recheck_a27_flow,
    -496.238962,
    atol=1e-5,
)

print(
    "Reusable model returned exactly to the validated "
    "Notebook 3 peak solution."
)

## 7. Economic-congestion and dispatch-degeneracy diagnostics

The representative-hour tests produced valid optimal solutions, but several
hours contain more zero-cost generation availability than demand.

When multiple generators have the same marginal cost, the primary economic
objective may have multiple optimal dispatch solutions. The solver can be
indifferent between:

- dispatching Hydro;
- dispatching Wind;
- dispatching PV;
- curtailing one zero-cost resource instead of another.

Consequently:

- optimized total cost remains valid;
- feasibility remains valid;
- the selected dispatch may not be unique;
- zero-valued line shadow prices can occur on binding lines;
- reported renewable curtailment may depend on the selected optimum.

A copper-plate benchmark removes transmission constraints while preserving
the same demand, availability, and generator costs.

The hourly congestion cost is:

$$
C_t^{\mathrm{congestion}}
=
C_t^{\mathrm{network}}
-
C_t^{\mathrm{copper}}
$$

A binding line with zero capacity shadow value is physically at its limit in
the selected solution, but an additional MW of capacity has no marginal
operating-cost value in that solution.

In [ ]:
def solve_copper_plate_hour(timestamp):
    """
    Solve the linear copper-plate dispatch by merit order.

    Because there are no network constraints, the least-cost solution
    is obtained by filling available generators in ascending cost order.
    """
    timestamp = pd.Timestamp(timestamp)

    dispatch_table = pd.DataFrame(
        {
            "GEN UID": GENERATOR_IDS,
            "Available PMax MW": (
                hourly_gen_pmax
                .loc[timestamp, GENERATOR_IDS]
                .to_numpy(dtype=float)
            ),
            "Linear Cost $/MWh": [
                generator_cost[g]
                for g in GENERATOR_IDS
            ],
        }
    )

    # GEN UID is a deterministic tie-breaker among equal-cost units.
    # It does not affect the minimum total cost.
    dispatch_table = dispatch_table.sort_values(
        [
            "Linear Cost $/MWh",
            "GEN UID",
        ]
    ).reset_index(drop=True)

    remaining_demand = float(
        system_load_from_buses.loc[timestamp]
    )

    dispatch_values = []

    for _, generator_row in dispatch_table.iterrows():
        dispatch_mw = min(
            generator_row["Available PMax MW"],
            remaining_demand,
        )

        dispatch_values.append(dispatch_mw)
        remaining_demand -= dispatch_mw

        if remaining_demand < VALIDATION_TOL:
            remaining_demand = 0.0

    if remaining_demand > VALIDATION_TOL:
        raise RuntimeError(
            f"Copper-plate generation shortage of "
            f"{remaining_demand:.6f} MW at {timestamp}."
        )

    dispatch_table["Dispatch MW"] = dispatch_values

    dispatch_table["Generation Cost $/h"] = (
        dispatch_table["Dispatch MW"]
        * dispatch_table["Linear Cost $/MWh"]
    )

    total_cost = dispatch_table[
        "Generation Cost $/h"
    ].sum()

    return total_cost, dispatch_table


representative_copper_plate_records = []
representative_copper_plate_dispatch = {}

for _, selected_hour in representative_hours.iterrows():
    timestamp = pd.Timestamp(
        selected_hour["Timestamp"]
    )

    copper_cost, copper_dispatch = (
        solve_copper_plate_hour(timestamp)
    )

    network_cost = representative_solutions[
        timestamp
    ]["summary"]["Objective $/h"]

    congestion_cost = network_cost - copper_cost

    representative_copper_plate_records.append(
        {
            "Timestamp": timestamp,
            "Selection reason":
                selected_hour["Selection reason"],
            "Network cost $/h": network_cost,
            "Copper-plate cost $/h": copper_cost,
            "Congestion cost $/h": congestion_cost,
        }
    )

    representative_copper_plate_dispatch[
        timestamp
    ] = copper_dispatch

representative_cost_comparison = pd.DataFrame(
    representative_copper_plate_records
).sort_values("Timestamp")

display(representative_cost_comparison)

assert (
    representative_cost_comparison[
        "Congestion cost $/h"
    ]
    >= -1e-5
).all()

print(
    "All network-constrained costs are greater than or "
    "equal to their copper-plate benchmarks."
)

In [ ]:
representative_line_diagnostic_records = []
representative_dual_summary_records = []

for timestamp, solution in representative_solutions.items():
    line_diagnostic = solution[
        "line_results"
    ].copy()

    line_diagnostic["Upper-limit dual"] = (
        pd.to_numeric(
            line_diagnostic["Upper-limit dual"],
            errors="coerce",
        )
        .fillna(0.0)
    )

    line_diagnostic["Lower-limit dual"] = (
        pd.to_numeric(
            line_diagnostic["Lower-limit dual"],
            errors="coerce",
        )
        .fillna(0.0)
    )

    # For this minimization formulation, an active capacity
    # constraint generally has a nonpositive dual. Its negative
    # gives the local value of relaxing that constraint.
    line_diagnostic["Capacity shadow value $/MW-h"] = (
        np.maximum.reduce(
            [
                np.zeros(len(line_diagnostic)),
                -line_diagnostic[
                    "Upper-limit dual"
                ].to_numpy(),
                -line_diagnostic[
                    "Lower-limit dual"
                ].to_numpy(),
            ]
        )
    )

    economically_valuable = (
        line_diagnostic[
            "Capacity shadow value $/MW-h"
        ]
        > 1e-7
    )

    diagnostic_selection = (
        line_diagnostic["Binding"]
        | economically_valuable
        | (
            line_diagnostic["Utilization"]
            >= 0.95
        )
    )

    selected_lines = line_diagnostic.loc[
        diagnostic_selection,
        [
            "UID",
            "From Bus",
            "To Bus",
            "Flow MW",
            "Rating MW",
            "Utilization",
            "Headroom MW",
            "Binding",
            "Upper-limit dual",
            "Lower-limit dual",
            "Capacity shadow value $/MW-h",
        ],
    ].copy()

    selected_lines.insert(
        0,
        "Timestamp",
        timestamp,
    )

    representative_line_diagnostic_records.append(
        selected_lines
    )

    representative_dual_summary_records.append(
        {
            "Timestamp": timestamp,
            "Binding branches":
                int(line_diagnostic["Binding"].sum()),
            "Binding branches with positive shadow value":
                int(
                    (
                        line_diagnostic["Binding"]
                        & economically_valuable
                    ).sum()
                ),
            "Maximum capacity shadow value $/MW-h":
                line_diagnostic[
                    "Capacity shadow value $/MW-h"
                ].max(),
            "Number of lines with positive shadow value":
                int(economically_valuable.sum()),
        }
    )

representative_line_diagnostics = pd.concat(
    representative_line_diagnostic_records,
    ignore_index=True,
)

representative_dual_summary = pd.DataFrame(
    representative_dual_summary_records
).sort_values("Timestamp")

print("Representative congestion-dual summary:")
display(representative_dual_summary)

print("Binding, economically valuable, or >=95% utilized lines:")
display(
    representative_line_diagnostics.sort_values(
        [
            "Timestamp",
            "Utilization",
        ],
        ascending=[True, False],
    )
)

In [ ]:
zero_cost_generator_ids = [
    generator_uid
    for generator_uid in GENERATOR_IDS
    if generator_cost[generator_uid] <= 1e-12
]

curtailment_diagnostic_records = []

for timestamp, solution in representative_solutions.items():
    summary = solution["summary"]
    generation = solution["generation"]
    availability = solution["availability"]

    demand = summary["Demand MW"]
    vre_available = summary["VRE available MW"]
    vre_dispatch = summary["VRE dispatch MW"]
    actual_vre_curtailment = (
        vre_available - vre_dispatch
    )

    minimum_vre_curtailment_from_surplus = max(
        0.0,
        vre_available - demand,
    )

    additional_vre_curtailment = (
        actual_vre_curtailment
        - minimum_vre_curtailment_from_surplus
    )

    zero_cost_available = availability.loc[
        zero_cost_generator_ids
    ].sum()

    zero_cost_dispatch = generation.loc[
        zero_cost_generator_ids
    ].sum()

    curtailment_diagnostic_records.append(
        {
            "Timestamp": timestamp,
            "Demand MW": demand,
            "VRE available MW": vre_available,
            "VRE dispatch MW": vre_dispatch,
            "Actual VRE curtailment MW":
                actual_vre_curtailment,
            "Minimum VRE curtailment from energy surplus MW":
                minimum_vre_curtailment_from_surplus,
            "Additional VRE curtailment MW":
                additional_vre_curtailment,
            "Hydro dispatch MW":
                summary["Hydro dispatch MW"],
            "Unused hydro availability MW":
                summary["Unused hydro availability MW"],
            "Zero-cost availability MW":
                zero_cost_available,
            "Zero-cost dispatch MW":
                zero_cost_dispatch,
            "Positive-cost dispatch MW":
                generation.sum() - zero_cost_dispatch,
        }
    )

representative_curtailment_diagnostics = (
    pd.DataFrame(curtailment_diagnostic_records)
    .sort_values("Timestamp")
)

display(representative_curtailment_diagnostics)

assert (
    representative_curtailment_diagnostics[
        "Actual VRE curtailment MW"
    ]
    >= -VALIDATION_TOL
).all()

assert (
    representative_curtailment_diagnostics[
        "Additional VRE curtailment MW"
    ]
    >= -VALIDATION_TOL
).all()

print("Representative curtailment diagnostics completed.")

## 8. Lexicographic VRE-priority dispatch

The primary optimization objective remains minimum operating cost:

$$
C_t^*
=
\min
\sum_{g \in G} c_g P_{g,t}
$$

When multiple dispatch solutions have the same minimum cost, a secondary
optimization selects the solution with maximum VRE dispatch:

$$
\max
\sum_{g \in G^{\mathrm{VRE}}} P_{g,t}
$$

subject to all original physical constraints and the additional cost-optimality
condition:

$$
\sum_{g \in G} c_g P_{g,t}
\leq
C_t^*+\varepsilon_t
$$

where the numerical tolerance is:

$$
\varepsilon_t
=
\max
\left(
10^{-6},
10^{-9}|C_t^*|
\right)
$$

This is a lexicographic optimization:

1. minimize operating cost;
2. among cost-optimal solutions, maximize VRE dispatch.

The primary solution continues to provide:

- operating cost;
- economic line shadow values;
- network-versus-copper-plate congestion cost.

The secondary solution provides:

- minimum VRE curtailment among cost-optimal solutions;
- corresponding Hydro dispatch;
- a less arbitrary dispatch during zero-cost surplus hours.

Dual values from the secondary solve are not interpreted as economic
transmission values because its objective is VRE dispatch, not operating cost.

In [ ]:
if not hasattr(hourly_model, "primary_cost_cap_value"):
    hourly_model.primary_cost_cap_value = pyo.Param(
        initialize=0.0,
        within=pyo.NonNegativeReals,
        mutable=True,
    )

    hourly_model.primary_cost_cap = pyo.Constraint(
        expr=(
            hourly_model.total_cost.expr
            <= hourly_model.primary_cost_cap_value
        )
    )

    hourly_model.maximum_vre_dispatch = pyo.Objective(
        expr=sum(
            hourly_model.generation[g]
            for g in variable_renewable_generator_ids
        ),
        sense=pyo.maximize,
    )

    # The primary economic objective is active by default.
    hourly_model.primary_cost_cap.deactivate()
    hourly_model.maximum_vre_dispatch.deactivate()

print(
    "Primary objective active:",
    hourly_model.total_cost.active,
)
print(
    "Cost-cap constraint active:",
    hourly_model.primary_cost_cap.active,
)
print(
    "Secondary VRE objective active:",
    hourly_model.maximum_vre_dispatch.active,
)

assert hourly_model.total_cost.active
assert not hourly_model.primary_cost_cap.active
assert not hourly_model.maximum_vre_dispatch.active

In [ ]:
def solve_hour_lexicographically(
    model,
    solver,
    timestamp,
):
    timestamp = pd.Timestamp(timestamp)

    # Ensure the model begins in primary-economic mode.
    model.maximum_vre_dispatch.deactivate()
    model.primary_cost_cap.deactivate()
    model.total_cost.activate()

    load_hour_into_model(
        model,
        timestamp,
    )

    primary_results = solve_loaded_hour(
        model,
        solver,
    )

    primary_validation = validate_hourly_solution(
        model,
        timestamp,
    )

    # Extract before the secondary solve overwrites variables and duals.
    primary_solution = extract_solved_hour(
        model,
        timestamp,
    )

    primary_cost = float(
        pyo.value(model.total_cost)
    )

    primary_vre_dispatch = primary_solution[
        "summary"
    ]["VRE dispatch MW"]

    cost_tolerance = max(
        1e-6,
        abs(primary_cost) * 1e-9,
    )

    cost_cap = (
        primary_cost + cost_tolerance
    )

    model.primary_cost_cap_value.set_value(
        cost_cap
    )

    # Switch to lexicographic stage two.
    model.total_cost.deactivate()
    model.primary_cost_cap.activate()
    model.maximum_vre_dispatch.activate()

    try:
        secondary_results = solve_loaded_hour(
            model,
            solver,
        )

        secondary_validation = validate_hourly_solution(
            model,
            timestamp,
        )

        secondary_solution = extract_solved_hour(
            model,
            timestamp,
        )

        secondary_cost = float(
            pyo.value(model.total_cost.expr)
        )

        secondary_vre_dispatch = secondary_solution[
            "summary"
        ]["VRE dispatch MW"]

        if secondary_cost > cost_cap + 1e-5:
            raise AssertionError(
                "Secondary solution violates the primary cost cap."
            )

        if (
            secondary_vre_dispatch
            < primary_vre_dispatch - VALIDATION_TOL
        ):
            raise AssertionError(
                "Secondary VRE dispatch is lower than "
                "primary VRE dispatch."
            )

    finally:
        # Always restore primary-economic mode.
        model.maximum_vre_dispatch.deactivate()
        model.primary_cost_cap.deactivate()
        model.total_cost.activate()

    return {
        "timestamp": timestamp,
        "primary_results": primary_results,
        "primary_validation": primary_validation,
        "primary_solution": primary_solution,
        "primary_cost": primary_cost,
        "primary_vre_dispatch": primary_vre_dispatch,
        "cost_tolerance": cost_tolerance,
        "cost_cap": cost_cap,
        "secondary_results": secondary_results,
        "secondary_validation": secondary_validation,
        "secondary_solution": secondary_solution,
        "secondary_cost": secondary_cost,
        "secondary_vre_dispatch": secondary_vre_dispatch,
    }

In [ ]:
lexicographic_representative_results = {}
lexicographic_comparison_records = []

lexicographic_start = time.perf_counter()

for _, selected_hour in representative_hours.iterrows():
    timestamp = pd.Timestamp(
        selected_hour["Timestamp"]
    )

    result = solve_hour_lexicographically(
        hourly_model,
        hourly_solver,
        timestamp,
    )

    lexicographic_representative_results[
        timestamp
    ] = result

    primary_summary = result[
        "primary_solution"
    ]["summary"]

    secondary_summary = result[
        "secondary_solution"
    ]["summary"]

    minimum_curtailment_from_surplus = max(
        0.0,
        secondary_summary["VRE available MW"]
        - secondary_summary["Demand MW"],
    )

    primary_lines = result[
        "primary_solution"
    ]["line_results"]

    secondary_lines = result[
        "secondary_solution"
    ]["line_results"]

    lexicographic_comparison_records.append(
        {
            "Timestamp": timestamp,
            "Selection reason":
                selected_hour["Selection reason"],
            "Primary cost $/h":
                result["primary_cost"],
            "Secondary cost $/h":
                result["secondary_cost"],
            "Cost difference $/h":
                result["secondary_cost"]
                - result["primary_cost"],
            "Cost tolerance $/h":
                result["cost_tolerance"],
            "Primary VRE dispatch MW":
                primary_summary["VRE dispatch MW"],
            "Secondary VRE dispatch MW":
                secondary_summary["VRE dispatch MW"],
            "Increase in VRE dispatch MW":
                secondary_summary["VRE dispatch MW"]
                - primary_summary["VRE dispatch MW"],
            "Primary VRE curtailment MW":
                primary_summary["VRE curtailment MW"],
            "Minimum-cost VRE curtailment MW":
                secondary_summary["VRE curtailment MW"],
            "Energy-surplus minimum curtailment MW":
                minimum_curtailment_from_surplus,
            "Primary Hydro dispatch MW":
                primary_summary["Hydro dispatch MW"],
            "Secondary Hydro dispatch MW":
                secondary_summary["Hydro dispatch MW"],
            "Primary binding branches":
                int(primary_lines["Binding"].sum()),
            "Secondary binding branches":
                int(secondary_lines["Binding"].sum()),
            "Primary most utilized branch":
                primary_lines.iloc[0]["UID"],
            "Secondary most utilized branch":
                secondary_lines.iloc[0]["UID"],
            "Primary maximum utilization":
                primary_lines["Utilization"].max(),
            "Secondary maximum utilization":
                secondary_lines["Utilization"].max(),
        }
    )

lexicographic_elapsed = (
    time.perf_counter() - lexicographic_start
)

lexicographic_comparison = (
    pd.DataFrame(lexicographic_comparison_records)
    .sort_values("Timestamp")
    .reset_index(drop=True)
)

print(
    "Lexicographic representative solve time:",
    f"{lexicographic_elapsed:.4f} seconds",
)

print(
    "Average time per primary-plus-secondary hour:",
    f"{lexicographic_elapsed / len(representative_hours):.4f} seconds",
)

display(lexicographic_comparison)

In [ ]:
assert (
    lexicographic_comparison["Cost difference $/h"]
    <= (
        lexicographic_comparison["Cost tolerance $/h"]
        + 1e-5
    )
).all()

assert (
    lexicographic_comparison[
        "Increase in VRE dispatch MW"
    ]
    >= -VALIDATION_TOL
).all()

assert (
    lexicographic_comparison[
        "Minimum-cost VRE curtailment MW"
    ]
    >= (
        lexicographic_comparison[
            "Energy-surplus minimum curtailment MW"
        ]
        - VALIDATION_TOL
    )
).all()

for timestamp, result in (
    lexicographic_representative_results.items()
):
    for validation_name in [
        "primary_validation",
        "secondary_validation",
    ]:
        validation = result[validation_name]

        for metric in [
            "System balance error MW",
            "Maximum generator violation MW",
            "Maximum thermal violation MW",
            "Maximum DC-flow error MW",
            "Maximum nodal-balance error MW",
            "Reference-angle error radians",
        ]:
            assert (
                validation.loc[metric]
                <= VALIDATION_TOL
            )

print(
    "All primary and secondary representative "
    "solutions passed validation."
)

print(
    "The model has been restored to primary-economic mode:"
)

print(
    {
        "primary_objective_active":
            hourly_model.total_cost.active,
        "cost_cap_active":
            hourly_model.primary_cost_cap.active,
        "secondary_objective_active":
            hourly_model.maximum_vre_dispatch.active,
    }
)

assert hourly_model.total_cost.active
assert not hourly_model.primary_cost_cap.active
assert not hourly_model.maximum_vre_dispatch.active

## 9. Chronological results architecture

Every hour will produce two related optimal solutions.

### Primary economic solution

The primary solution minimizes operating cost and provides:

- operating cost;
- network-versus-copper-plate congestion cost;
- economically meaningful line shadow values;
- primary dispatch and flows.

### Secondary VRE-priority solution

The secondary solution maximizes VRE dispatch while preserving the primary
minimum cost within a numerical tolerance. It provides:

- minimum VRE curtailment among cost-optimal solutions;
- corresponding Hydro dispatch;
- VRE-priority branch flows.

### Interpretation of branch metrics

A branch can be physically binding while having zero economic value.

Therefore, annual corridor screening will distinguish:

$$
\text{Binding hours}
$$

from:

$$
\text{Positive-shadow-value hours}
$$

A positive capacity shadow value means that marginally relaxing the active
line constraint reduces the primary operating-cost objective.

Raw utilization and binding frequency remain useful physical diagnostics, but
they are not sufficient measures of economic transmission value.

In [ ]:
generator_position = {
    generator_uid: position
    for position, generator_uid in enumerate(GENERATOR_IDS)
}

branch_position = {
    branch_uid: position
    for position, branch_uid in enumerate(BRANCH_IDS)
}

generator_cost_array = np.array(
    [
        generator_cost[g]
        for g in GENERATOR_IDS
    ],
    dtype=float,
)

merit_order_positions = sorted(
    range(len(GENERATOR_IDS)),
    key=lambda position: (
        generator_cost_array[position],
        GENERATOR_IDS[position],
    ),
)


def calculate_copper_plate_cost_fast(
    demand_mw,
    availability_array,
):
    remaining_demand = float(demand_mw)
    copper_dispatch = np.zeros(
        len(GENERATOR_IDS),
        dtype=float,
    )

    for position in merit_order_positions:
        dispatched_mw = min(
            availability_array[position],
            remaining_demand,
        )

        copper_dispatch[position] = dispatched_mw
        remaining_demand -= dispatched_mw

        if remaining_demand <= VALIDATION_TOL:
            remaining_demand = 0.0
            break

    if remaining_demand > VALIDATION_TOL:
        raise RuntimeError(
            "Copper-plate generation shortage of "
            f"{remaining_demand:.6f} MW."
        )

    copper_cost = float(
        np.dot(
            copper_dispatch,
            generator_cost_array,
        )
    )

    return copper_cost, copper_dispatch


def run_chronological_opf(
    timestamps,
    model,
    solver,
    progress_every=24,
):
    timestamps = pd.DatetimeIndex(timestamps)
    number_of_hours = len(timestamps)

    number_of_generators = len(GENERATOR_IDS)
    number_of_branches = len(BRANCH_IDS)

    primary_dispatch_array = np.full(
        (number_of_hours, number_of_generators),
        np.nan,
    )

    secondary_dispatch_array = np.full(
        (number_of_hours, number_of_generators),
        np.nan,
    )

    primary_flow_array = np.full(
        (number_of_hours, number_of_branches),
        np.nan,
    )

    secondary_flow_array = np.full(
        (number_of_hours, number_of_branches),
        np.nan,
    )

    primary_utilization_array = np.full(
        (number_of_hours, number_of_branches),
        np.nan,
    )

    secondary_utilization_array = np.full(
        (number_of_hours, number_of_branches),
        np.nan,
    )

    upper_dual_array = np.full(
        (number_of_hours, number_of_branches),
        np.nan,
    )

    lower_dual_array = np.full(
        (number_of_hours, number_of_branches),
        np.nan,
    )

    capacity_shadow_array = np.full(
        (number_of_hours, number_of_branches),
        np.nan,
    )

    hourly_summary_records = []
    validation_records = []

    overall_start = time.perf_counter()

    for hour_position, timestamp in enumerate(timestamps):
        hour_start = time.perf_counter()

        result = solve_hour_lexicographically(
            model,
            solver,
            timestamp,
        )

        primary_solution = result["primary_solution"]
        secondary_solution = result["secondary_solution"]

        primary_summary = primary_solution["summary"]
        secondary_summary = secondary_solution["summary"]

        primary_generation = (
            primary_solution["generation"]
            .reindex(GENERATOR_IDS)
            .to_numpy(dtype=float)
        )

        secondary_generation = (
            secondary_solution["generation"]
            .reindex(GENERATOR_IDS)
            .to_numpy(dtype=float)
        )

        primary_lines = (
            primary_solution["line_results"]
            .set_index("UID")
            .reindex(BRANCH_IDS)
        )

        secondary_lines = (
            secondary_solution["line_results"]
            .set_index("UID")
            .reindex(BRANCH_IDS)
        )

        primary_flows = primary_lines[
            "Flow MW"
        ].to_numpy(dtype=float)

        secondary_flows = secondary_lines[
            "Flow MW"
        ].to_numpy(dtype=float)

        primary_utilization = primary_lines[
            "Utilization"
        ].to_numpy(dtype=float)

        secondary_utilization = secondary_lines[
            "Utilization"
        ].to_numpy(dtype=float)

        upper_duals = (
            pd.to_numeric(
                primary_lines["Upper-limit dual"],
                errors="coerce",
            )
            .fillna(0.0)
            .to_numpy(dtype=float)
        )

        lower_duals = (
            pd.to_numeric(
                primary_lines["Lower-limit dual"],
                errors="coerce",
            )
            .fillna(0.0)
            .to_numpy(dtype=float)
        )

        capacity_shadow_values = np.maximum.reduce(
            [
                np.zeros(number_of_branches),
                -upper_duals,
                -lower_duals,
            ]
        )

        primary_dispatch_array[
            hour_position, :
        ] = primary_generation

        secondary_dispatch_array[
            hour_position, :
        ] = secondary_generation

        primary_flow_array[
            hour_position, :
        ] = primary_flows

        secondary_flow_array[
            hour_position, :
        ] = secondary_flows

        primary_utilization_array[
            hour_position, :
        ] = primary_utilization

        secondary_utilization_array[
            hour_position, :
        ] = secondary_utilization

        upper_dual_array[
            hour_position, :
        ] = upper_duals

        lower_dual_array[
            hour_position, :
        ] = lower_duals

        capacity_shadow_array[
            hour_position, :
        ] = capacity_shadow_values

        demand_mw = primary_summary["Demand MW"]

        availability_array = (
            hourly_gen_pmax
            .loc[timestamp, GENERATOR_IDS]
            .to_numpy(dtype=float)
        )

        copper_cost, _ = (
            calculate_copper_plate_cost_fast(
                demand_mw,
                availability_array,
            )
        )

        network_cost = result["primary_cost"]
        congestion_cost = network_cost - copper_cost

        primary_binding = (
            np.abs(
                np.abs(primary_flows)
                - np.array(
                    [
                        branch_rating[line]
                        for line in BRANCH_IDS
                    ],
                    dtype=float,
                )
            )
            <= 1e-5
        )

        secondary_binding = (
            np.abs(
                np.abs(secondary_flows)
                - np.array(
                    [
                        branch_rating[line]
                        for line in BRANCH_IDS
                    ],
                    dtype=float,
                )
            )
            <= 1e-5
        )

        positive_shadow = (
            capacity_shadow_values > 1e-7
        )

        hour_elapsed = (
            time.perf_counter() - hour_start
        )

        hourly_summary_records.append(
            {
                "Timestamp": timestamp,
                "Demand MW": demand_mw,
                "Total availability MW":
                    primary_summary[
                        "Total availability MW"
                    ],
                "Primary network cost $/h":
                    network_cost,
                "Copper-plate cost $/h":
                    copper_cost,
                "Congestion cost $/h":
                    congestion_cost,
                "VRE available MW":
                    primary_summary["VRE available MW"],
                "Primary VRE dispatch MW":
                    primary_summary["VRE dispatch MW"],
                "Primary VRE curtailment MW":
                    primary_summary["VRE curtailment MW"],
                "Minimum-cost VRE dispatch MW":
                    secondary_summary["VRE dispatch MW"],
                "Minimum-cost VRE curtailment MW":
                    secondary_summary[
                        "VRE curtailment MW"
                    ],
                "Primary Hydro dispatch MW":
                    primary_summary["Hydro dispatch MW"],
                "Secondary Hydro dispatch MW":
                    secondary_summary["Hydro dispatch MW"],
                "Primary maximum utilization":
                    primary_utilization.max(),
                "Secondary maximum utilization":
                    secondary_utilization.max(),
                "Primary branches >=90%":
                    int((primary_utilization >= 0.90).sum()),
                "Primary branches >=95%":
                    int((primary_utilization >= 0.95).sum()),
                "Primary branches >=99%":
                    int((primary_utilization >= 0.99).sum()),
                "Primary binding branches":
                    int(primary_binding.sum()),
                "Secondary binding branches":
                    int(secondary_binding.sum()),
                "Positive-shadow lines":
                    int(positive_shadow.sum()),
                "Maximum shadow value $/MW-h":
                    capacity_shadow_values.max(),
                "Primary-secondary cost difference $/h":
                    result["secondary_cost"]
                    - result["primary_cost"],
                "Cost tolerance $/h":
                    result["cost_tolerance"],
                "Solve time seconds":
                    hour_elapsed,
            }
        )

        primary_validation = result[
            "primary_validation"
        ]

        secondary_validation = result[
            "secondary_validation"
        ]

        validation_record = {
            "Timestamp": timestamp,
        }

        residual_metrics = [
            "System balance error MW",
            "Maximum generator violation MW",
            "Maximum thermal violation MW",
            "Maximum DC-flow error MW",
            "Maximum nodal-balance error MW",
            "Reference-angle error radians",
        ]

        for metric in residual_metrics:
            validation_record[
                f"Primary {metric}"
            ] = primary_validation.loc[metric]

            validation_record[
                f"Secondary {metric}"
            ] = secondary_validation.loc[metric]

        validation_records.append(validation_record)

        if (
            progress_every is not None
            and (
                (hour_position + 1) % progress_every == 0
                or hour_position + 1 == number_of_hours
            )
        ):
            elapsed = time.perf_counter() - overall_start
            average_seconds = elapsed / (hour_position + 1)

            print(
                f"Solved {hour_position + 1:,}/"
                f"{number_of_hours:,} hours | "
                f"elapsed {elapsed:.2f}s | "
                f"average {average_seconds:.4f}s/hour"
            )

    total_elapsed = time.perf_counter() - overall_start

    results = {
        "hourly_summary": (
            pd.DataFrame(hourly_summary_records)
            .set_index("Timestamp")
        ),
        "validation": (
            pd.DataFrame(validation_records)
            .set_index("Timestamp")
        ),
        "primary_dispatch": pd.DataFrame(
            primary_dispatch_array,
            index=timestamps,
            columns=GENERATOR_IDS,
        ),
        "secondary_dispatch": pd.DataFrame(
            secondary_dispatch_array,
            index=timestamps,
            columns=GENERATOR_IDS,
        ),
        "primary_flow": pd.DataFrame(
            primary_flow_array,
            index=timestamps,
            columns=BRANCH_IDS,
        ),
        "secondary_flow": pd.DataFrame(
            secondary_flow_array,
            index=timestamps,
            columns=BRANCH_IDS,
        ),
        "primary_utilization": pd.DataFrame(
            primary_utilization_array,
            index=timestamps,
            columns=BRANCH_IDS,
        ),
        "secondary_utilization": pd.DataFrame(
            secondary_utilization_array,
            index=timestamps,
            columns=BRANCH_IDS,
        ),
        "upper_dual": pd.DataFrame(
            upper_dual_array,
            index=timestamps,
            columns=BRANCH_IDS,
        ),
        "lower_dual": pd.DataFrame(
            lower_dual_array,
            index=timestamps,
            columns=BRANCH_IDS,
        ),
        "capacity_shadow_value": pd.DataFrame(
            capacity_shadow_array,
            index=timestamps,
            columns=BRANCH_IDS,
        ),
        "elapsed_seconds": total_elapsed,
    }

    return results

In [ ]:
# Reset all line limits to their native ratings.
for line in BRANCH_IDS:
    hourly_model.line_limit[line] = branch_rating[line]

pilot_start = pd.Timestamp(
    "2020-03-09 00:00:00"
)

pilot_end = pd.Timestamp(
    "2020-03-15 23:00:00"
)

pilot_timestamps = hourly_bus_load.loc[
    pilot_start:pilot_end
].index

print("Pilot start:", pilot_timestamps[0])
print("Pilot end:", pilot_timestamps[-1])
print("Pilot hours:", len(pilot_timestamps))

pilot_results = run_chronological_opf(
    timestamps=pilot_timestamps,
    model=hourly_model,
    solver=hourly_solver,
    progress_every=24,
)

pilot_runtime = pilot_results["elapsed_seconds"]

estimated_full_year_seconds = (
    pilot_runtime
    / len(pilot_timestamps)
    * 8784
)

print(
    "\nPilot runtime:",
    f"{pilot_runtime:.2f} seconds",
)

print(
    "Estimated full-year runtime:",
    f"{estimated_full_year_seconds / 60:.2f} minutes",
)

In [ ]:
pilot_summary = pilot_results["hourly_summary"]
pilot_validation = pilot_results["validation"]

maximum_pilot_residual = (
    pilot_validation
    .to_numpy(dtype=float)
    .max()
)

maximum_cost_cap_difference = (
    pilot_summary[
        "Primary-secondary cost difference $/h"
    ]
    - pilot_summary["Cost tolerance $/h"]
).max()

minimum_congestion_cost = pilot_summary[
    "Congestion cost $/h"
].min()

print("Pilot hours solved:", len(pilot_summary))
print(
    "Maximum validation residual:",
    maximum_pilot_residual,
)
print(
    "Minimum congestion cost:",
    minimum_congestion_cost,
)
print(
    "Maximum cost-cap excess:",
    maximum_cost_cap_difference,
)

assert len(pilot_summary) == 168
assert maximum_pilot_residual <= VALIDATION_TOL
assert minimum_congestion_cost >= -1e-5
assert maximum_cost_cap_difference <= 1e-5

print("\nPilot hourly summary:")
display(
    pilot_summary[
        [
            "Demand MW",
            "Primary network cost $/h",
            "Copper-plate cost $/h",
            "Congestion cost $/h",
            "VRE available MW",
            "Primary VRE curtailment MW",
            "Minimum-cost VRE curtailment MW",
            "Primary maximum utilization",
            "Primary binding branches",
            "Positive-shadow lines",
            "Maximum shadow value $/MW-h",
            "Solve time seconds",
        ]
    ].describe()
)

pilot_primary_utilization = (
    pilot_results["primary_utilization"]
)

pilot_secondary_utilization = (
    pilot_results["secondary_utilization"]
)

pilot_shadow_value = (
    pilot_results["capacity_shadow_value"]
)

pilot_branch_metrics = pd.DataFrame(
    {
        "Primary maximum utilization":
            pilot_primary_utilization.max(),
        "Secondary maximum utilization":
            pilot_secondary_utilization.max(),
        "Primary hours >=90%":
            (pilot_primary_utilization >= 0.90).sum(),
        "Primary hours >=95%":
            (pilot_primary_utilization >= 0.95).sum(),
        "Primary hours >=99%":
            (pilot_primary_utilization >= 0.99).sum(),
        "Primary binding hours":
            (
                np.abs(
                    pilot_primary_utilization - 1.0
                )
                <= 1e-5
            ).sum(),
        "Positive-shadow hours":
            (pilot_shadow_value > 1e-7).sum(),
        "Sum of hourly shadow values":
            pilot_shadow_value.sum(),
        "Maximum shadow value $/MW-h":
            pilot_shadow_value.max(),
    }
)

pilot_branch_metrics.index.name = "UID"

pilot_branch_metrics = (
    pilot_branch_metrics
    .join(
        branch.set_index("UID")[
            [
                "From Bus",
                "To Bus",
                "Cont Rating",
            ]
        ]
    )
)

print("\nTop pilot branches by positive-shadow hours:")
display(
    pilot_branch_metrics
    .sort_values(
        [
            "Positive-shadow hours",
            "Sum of hourly shadow values",
        ],
        ascending=False,
    )
    .head(15)
)

march_test_timestamp = pd.Timestamp(
    "2020-03-12 11:00:00"
)

print("\nMarch representative hour from pilot:")
display(
    pilot_summary.loc[
        [march_test_timestamp]
    ]
)

assert np.isclose(
    pilot_summary.loc[
        march_test_timestamp,
        "Primary network cost $/h",
    ],
    90.505706,
    atol=1e-5,
)

print(
    "The 168-hour chronological pilot passed all checks."
)

## 10. Full-year chronological baseline

The complete baseline contains 8,784 independent hourly DC-OPFs.

For each hour, the workflow is:

1. solve the primary network-constrained economic dispatch;
2. record economic line duals before changing the objective;
3. solve the secondary VRE-priority problem;
4. independently validate both solutions;
5. calculate the copper-plate benchmark;
6. retain dispatch, flow, utilization, congestion, curtailment, and validation results.

The year is processed in monthly chunks. Each completed month is saved as a
checkpoint so an interrupted run can resume without repeating completed months.

Monthly chunking also creates a fixed reproducibility structure. A fresh model
and solver are constructed at the beginning of each month.

In [ ]:
def add_lexicographic_components(model):
    if hasattr(model, "primary_cost_cap_value"):
        return model

    model.primary_cost_cap_value = pyo.Param(
        initialize=0.0,
        within=pyo.NonNegativeReals,
        mutable=True,
    )

    model.primary_cost_cap = pyo.Constraint(
        expr=(
            model.total_cost.expr
            <= model.primary_cost_cap_value
        )
    )

    model.maximum_vre_dispatch = pyo.Objective(
        expr=sum(
            model.generation[g]
            for g in variable_renewable_generator_ids
        ),
        sense=pyo.maximize,
    )

    model.primary_cost_cap.deactivate()
    model.maximum_vre_dispatch.deactivate()

    return model

In [ ]:
FRAME_RESULT_KEYS = [
    "hourly_summary",
    "validation",
    "primary_dispatch",
    "secondary_dispatch",
    "primary_flow",
    "secondary_flow",
    "primary_utilization",
    "secondary_utilization",
    "upper_dual",
    "lower_dual",
    "capacity_shadow_value",
]


def validate_chronological_chunk(
    chunk_results,
    expected_index,
):
    expected_index = pd.DatetimeIndex(
        expected_index
    )

    for key in FRAME_RESULT_KEYS:
        result_frame = chunk_results[key]

        assert result_frame.index.equals(
            expected_index
        ), f"{key} has an incorrect timestamp index."

        assert not result_frame.isna().any().any(), (
            f"{key} contains missing values."
        )

    summary = chunk_results["hourly_summary"]
    validation = chunk_results["validation"]

    maximum_residual = (
        validation
        .to_numpy(dtype=float)
        .max()
    )

    minimum_congestion_cost = summary[
        "Congestion cost $/h"
    ].min()

    maximum_cost_cap_excess = (
        summary[
            "Primary-secondary cost difference $/h"
        ]
        - summary["Cost tolerance $/h"]
    ).max()

    assert maximum_residual <= VALIDATION_TOL
    assert minimum_congestion_cost >= -1e-5
    assert maximum_cost_cap_excess <= 1e-5

    return {
        "hours": len(summary),
        "maximum_residual": maximum_residual,
        "minimum_congestion_cost":
            minimum_congestion_cost,
        "maximum_cost_cap_excess":
            maximum_cost_cap_excess,
    }

In [ ]:
CHECKPOINT_VERSION = "notebook4_v1"

NOTEBOOK4_CHECKPOINT_PATH = (
    RESULTS_PATH
    / "notebook4_checkpoints"
)

NOTEBOOK4_CHECKPOINT_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

# Change to True only if you intentionally want to replace
# every existing monthly checkpoint.
FORCE_RERUN_FULL_YEAR = False

monthly_chronological_results = {}
monthly_run_records = []

full_year_run_start = time.perf_counter()

month_periods = pd.period_range(
    start="2020-01",
    end="2020-12",
    freq="M",
)

for month_period in month_periods:
    month_label = str(month_period)

    month_mask = (
        hourly_bus_load.index.to_period("M")
        == month_period
    )

    month_timestamps = hourly_bus_load.index[
        month_mask
    ]

    checkpoint_path = (
        NOTEBOOK4_CHECKPOINT_PATH
        / (
            f"{CHECKPOINT_VERSION}_"
            f"{month_label}.pkl.gz"
        )
    )

    if (
        checkpoint_path.exists()
        and not FORCE_RERUN_FULL_YEAR
    ):
        print(
            f"\nLoading checkpoint for {month_label}: "
            f"{checkpoint_path.name}"
        )

        month_results = pd.read_pickle(
            checkpoint_path,
            compression="gzip",
        )

        checkpoint_status = "loaded"

    else:
        print(
            f"\nSolving {month_label}: "
            f"{len(month_timestamps)} hours"
        )

        month_model = build_hourly_rts_opf()
        month_model = add_lexicographic_components(
            month_model
        )

        month_solver = pyo.SolverFactory(
            HOURLY_SOLVER_NAME
        )

        month_results = run_chronological_opf(
            timestamps=month_timestamps,
            model=month_model,
            solver=month_solver,
            progress_every=168,
        )

        month_results["metadata"] = {
            "checkpoint_version":
                CHECKPOINT_VERSION,
            "month": month_label,
            "number_of_hours":
                len(month_timestamps),
            "solver_interface":
                HOURLY_SOLVER_NAME,
        }

        chunk_audit = validate_chronological_chunk(
            month_results,
            month_timestamps,
        )

        temporary_checkpoint_path = (
            checkpoint_path.with_suffix(
                checkpoint_path.suffix + ".tmp"
            )
        )

        pd.to_pickle(
            month_results,
            temporary_checkpoint_path,
            compression="gzip",
        )

        temporary_checkpoint_path.replace(
            checkpoint_path
        )

        print(
            f"Saved checkpoint: "
            f"{checkpoint_path.name}"
        )

        checkpoint_status = "solved"

        del month_model
        del month_solver

    chunk_audit = validate_chronological_chunk(
        month_results,
        month_timestamps,
    )

    metadata = month_results.get(
        "metadata",
        {},
    )

    if metadata:
        assert (
            metadata["checkpoint_version"]
            == CHECKPOINT_VERSION
        )
        assert metadata["month"] == month_label
        assert (
            metadata["number_of_hours"]
            == len(month_timestamps)
        )

    monthly_chronological_results[
        month_label
    ] = month_results

    monthly_run_records.append(
        {
            "Month": month_label,
            "Status": checkpoint_status,
            "Hours": chunk_audit["hours"],
            "Maximum residual":
                chunk_audit["maximum_residual"],
            "Minimum congestion cost $/h":
                chunk_audit[
                    "minimum_congestion_cost"
                ],
            "Chunk solve time seconds":
                month_results["elapsed_seconds"],
            "Checkpoint":
                checkpoint_path.name,
        }
    )

full_year_wall_seconds = (
    time.perf_counter()
    - full_year_run_start
)

monthly_run_audit = pd.DataFrame(
    monthly_run_records
)

display(monthly_run_audit)

print(
    "Total hours across monthly chunks:",
    monthly_run_audit["Hours"].sum(),
)

print(
    "Total current-session wall time:",
    f"{full_year_wall_seconds:.2f} seconds",
)

In [ ]:
full_year_results = {}

for key in FRAME_RESULT_KEYS:
    combined_frame = pd.concat(
        [
            monthly_chronological_results[
                str(month_period)
            ][key]
            for month_period in month_periods
        ],
        axis=0,
    ).sort_index()

    full_year_results[key] = combined_frame

full_year_results["elapsed_seconds"] = sum(
    monthly_chronological_results[
        str(month_period)
    ]["elapsed_seconds"]
    for month_period in month_periods
)

expected_full_year_index = (
    hourly_bus_load.index
)

for key in FRAME_RESULT_KEYS:
    result_frame = full_year_results[key]

    assert len(result_frame) == 8784
    assert result_frame.index.is_unique
    assert result_frame.index.equals(
        expected_full_year_index
    )

print(
    "Combined full-year hours:",
    len(full_year_results["hourly_summary"]),
)

print(
    "Combined monthly solve time:",
    f"{full_year_results['elapsed_seconds']:.2f} seconds",
)

print(
    "All monthly results were combined successfully."
)

In [ ]:
annual_summary = full_year_results[
    "hourly_summary"
]

annual_validation = full_year_results[
    "validation"
]

maximum_annual_residual = (
    annual_validation
    .to_numpy(dtype=float)
    .max()
)

minimum_annual_congestion_cost = annual_summary[
    "Congestion cost $/h"
].min()

maximum_annual_cost_cap_excess = (
    annual_summary[
        "Primary-secondary cost difference $/h"
    ]
    - annual_summary["Cost tolerance $/h"]
).max()

# Remove only tiny negative numerical noise for reporting.
reported_hourly_congestion_cost = (
    annual_summary["Congestion cost $/h"]
    .clip(lower=0.0)
)

annual_network_cost = annual_summary[
    "Primary network cost $/h"
].sum()

annual_copper_plate_cost = annual_summary[
    "Copper-plate cost $/h"
].sum()

annual_congestion_cost = (
    reported_hourly_congestion_cost.sum()
)

hours_with_positive_congestion_cost = int(
    (
        reported_hourly_congestion_cost
        > 1e-6
    ).sum()
)

hours_with_positive_shadow = int(
    (
        annual_summary["Positive-shadow lines"]
        > 0
    ).sum()
)

hours_with_primary_binding = int(
    (
        annual_summary[
            "Primary binding branches"
        ]
        > 0
    ).sum()
)

annual_initial_totals = pd.Series(
    {
        "Hours solved": len(annual_summary),
        "Maximum validation residual":
            maximum_annual_residual,
        "Minimum raw congestion cost $/h":
            minimum_annual_congestion_cost,
        "Maximum cost-cap excess $/h":
            maximum_annual_cost_cap_excess,
        "Annual network operating cost $":
            annual_network_cost,
        "Annual copper-plate operating cost $":
            annual_copper_plate_cost,
        "Annual congestion cost $":
            annual_congestion_cost,
        "Hours with positive congestion cost":
            hours_with_positive_congestion_cost,
        "Hours with at least one positive-shadow line":
            hours_with_positive_shadow,
        "Hours with at least one primary binding line":
            hours_with_primary_binding,
        "Primary VRE curtailment MWh":
            annual_summary[
                "Primary VRE curtailment MW"
            ].sum(),
        "Minimum-cost VRE curtailment MWh":
            annual_summary[
                "Minimum-cost VRE curtailment MW"
            ].sum(),
        "Combined solve time seconds":
            full_year_results["elapsed_seconds"],
    },
    name="Value",
)

display(annual_initial_totals.to_frame())

assert len(annual_summary) == 8784
assert maximum_annual_residual <= VALIDATION_TOL
assert minimum_annual_congestion_cost >= -1e-5
assert maximum_annual_cost_cap_excess <= 1e-5

# Notebook 3 peak-hour regression checks inside the annual results.
assert np.isclose(
    annual_summary.loc[
        PEAK_TIMESTAMP,
        "Primary network cost $/h",
    ],
    130429.228095,
    atol=1e-3,
)

assert np.isclose(
    full_year_results[
        "primary_flow"
    ].loc[
        PEAK_TIMESTAMP,
        "A27",
    ],
    -496.238962,
    atol=1e-5,
)

print(
    "The complete 8,784-hour chronological baseline "
    "passed all validation and regression checks."
)

## 11. Annual baseline and branch-level congestion metrics

Annual congestion cost is the hourly difference between the network-constrained
and copper-plate objectives:

$$
C^{\mathrm{congestion}}_{\mathrm{annual}}
=
\sum_t
\left(
C_t^{\mathrm{network}}
-
C_t^{\mathrm{copper}}
\right)
$$

For each branch, the annual marginal-capacity-value screening metric is:

$$
V_l^{\mathrm{marginal}}
=
\sum_t \mu_{l,t}
$$

where `mu_l,t` is the nonnegative shadow value of relaxing the active thermal
limit by 1 MW in hour `t`.

Because every observation represents one hour, this sum has units of:

$$
\frac{\$}{\mathrm{MW\text{-}year}}
$$

This is a local marginal value. It must not be multiplied by a large
reconductoring capacity increase because the shadow value can fall to zero
after only a small capacity expansion.

Finalists selected from these metrics will therefore undergo explicit
finite-difference capacity tests.

In [ ]:
annual_analysis = annual_summary.copy()

annual_analysis["Reported congestion cost $/h"] = (
    annual_analysis["Congestion cost $/h"]
    .clip(lower=0.0)
)

annual_analysis["Economically congested"] = (
    annual_analysis[
        "Reported congestion cost $/h"
    ]
    > 1e-6
)

annual_analysis["Month"] = (
    annual_analysis.index.to_period("M")
)

monthly_baseline_summary = (
    annual_analysis
    .groupby("Month")
    .agg(
        Hours=("Demand MW", "count"),
        Demand_Energy_MWh=("Demand MW", "sum"),
        Network_Cost_Dollars=(
            "Primary network cost $/h",
            "sum",
        ),
        Copper_Plate_Cost_Dollars=(
            "Copper-plate cost $/h",
            "sum",
        ),
        Congestion_Cost_Dollars=(
            "Reported congestion cost $/h",
            "sum",
        ),
        Economically_Congested_Hours=(
            "Economically congested",
            "sum",
        ),
        Primary_VRE_Curtailment_MWh=(
            "Primary VRE curtailment MW",
            "sum",
        ),
        Minimum_Cost_VRE_Curtailment_MWh=(
            "Minimum-cost VRE curtailment MW",
            "sum",
        ),
        Maximum_Primary_Utilization=(
            "Primary maximum utilization",
            "max",
        ),
    )
)

monthly_baseline_summary[
    "Congestion Share of Copper-Plate Cost"
] = (
    monthly_baseline_summary[
        "Congestion_Cost_Dollars"
    ]
    / monthly_baseline_summary[
        "Copper_Plate_Cost_Dollars"
    ]
)

monthly_baseline_summary[
    "Economically Congested Hour Share"
] = (
    monthly_baseline_summary[
        "Economically_Congested_Hours"
    ]
    / monthly_baseline_summary["Hours"]
)

display(monthly_baseline_summary)

annual_congestion_premium = (
    annual_congestion_cost
    / annual_copper_plate_cost
)

primary_vs_lexicographic_curtailment_difference = (
    annual_summary[
        "Primary VRE curtailment MW"
    ].sum()
    - annual_summary[
        "Minimum-cost VRE curtailment MW"
    ].sum()
)

print(
    "Annual congestion premium:",
    f"{annual_congestion_premium:.4%}",
)

print(
    "Economically congested-hour share:",
    f"{hours_with_positive_congestion_cost / 8784:.4%}",
)

print(
    "Primary-minus-minimum-cost VRE curtailment:",
    f"{primary_vs_lexicographic_curtailment_difference:,.6f} MWh",
)

In [ ]:
annual_primary_utilization = full_year_results[
    "primary_utilization"
]

annual_secondary_utilization = full_year_results[
    "secondary_utilization"
]

annual_primary_flow = full_year_results[
    "primary_flow"
]

annual_shadow_value = full_year_results[
    "capacity_shadow_value"
]

annual_upper_dual = full_year_results[
    "upper_dual"
]

annual_lower_dual = full_year_results[
    "lower_dual"
]

primary_binding_matrix = (
    (annual_primary_utilization - 1.0).abs()
    <= 1e-5
)

secondary_binding_matrix = (
    (annual_secondary_utilization - 1.0).abs()
    <= 1e-5
)

positive_shadow_matrix = (
    annual_shadow_value > 1e-7
)

upper_shadow_value = (
    -annual_upper_dual
).clip(lower=0.0)

lower_shadow_value = (
    -annual_lower_dual
).clip(lower=0.0)

branch_annual_metrics = pd.DataFrame(
    {
        "Primary maximum utilization":
            annual_primary_utilization.max(),
        "Primary 99th-percentile utilization":
            annual_primary_utilization.quantile(0.99),
        "Primary mean utilization":
            annual_primary_utilization.mean(),
        "Secondary maximum utilization":
            annual_secondary_utilization.max(),
        "Secondary 99th-percentile utilization":
            annual_secondary_utilization.quantile(0.99),
        "Maximum absolute flow MW":
            annual_primary_flow.abs().max(),
        "Hours >=90%":
            (annual_primary_utilization >= 0.90).sum(),
        "Hours >=95%":
            (annual_primary_utilization >= 0.95).sum(),
        "Hours >=99%":
            (annual_primary_utilization >= 0.99).sum(),
        "Primary binding hours":
            primary_binding_matrix.sum(),
        "Secondary binding hours":
            secondary_binding_matrix.sum(),
        "Positive-shadow hours":
            positive_shadow_matrix.sum(),
        "Zero-value binding hours":
            (
                primary_binding_matrix
                & ~positive_shadow_matrix
            ).sum(),
        "Upper-direction valuable hours":
            (upper_shadow_value > 1e-7).sum(),
        "Lower-direction valuable hours":
            (lower_shadow_value > 1e-7).sum(),
        "Annual marginal value $/MW-year":
            annual_shadow_value.sum(),
        "Mean shadow when positive $/MW-h":
            (
                annual_shadow_value.sum()
                / positive_shadow_matrix.sum().replace(
                    0,
                    np.nan,
                )
            ),
        "Maximum shadow value $/MW-h":
            annual_shadow_value.max(),
    }
)

branch_annual_metrics.index.name = "UID"

branch_metadata = (
    branch
    .assign(UID=branch["UID"].astype(str))
    .set_index("UID")
)

bus_metadata = (
    bus
    .set_index("Bus ID")
)

branch_annual_metrics = branch_annual_metrics.join(
    branch_metadata[
        [
            "From Bus",
            "To Bus",
            "Cont Rating",
            "X",
            "Length",
        ]
    ]
)

branch_annual_metrics[
    "From Bus Name"
] = branch_annual_metrics["From Bus"].map(
    bus_metadata["Bus Name"]
)

branch_annual_metrics[
    "To Bus Name"
] = branch_annual_metrics["To Bus"].map(
    bus_metadata["Bus Name"]
)

branch_annual_metrics[
    "From Area"
] = branch_annual_metrics["From Bus"].map(
    bus_metadata["Area"]
)

branch_annual_metrics[
    "To Area"
] = branch_annual_metrics["To Bus"].map(
    bus_metadata["Area"]
)

assert len(branch_annual_metrics) == 120

print(
    "Physical branches with at least one positive-shadow hour:",
    int(
        (
            branch_annual_metrics[
                "Positive-shadow hours"
            ]
            > 0
        ).sum()
    ),
)

print(
    "Physical branches binding at least once:",
    int(
        (
            branch_annual_metrics[
                "Primary binding hours"
            ]
            > 0
        ).sum()
    ),
)

In [ ]:
economic_ranking_columns = [
    "From Bus",
    "To Bus",
    "From Area",
    "To Area",
    "Cont Rating",
    "Primary maximum utilization",
    "Primary 99th-percentile utilization",
    "Hours >=90%",
    "Primary binding hours",
    "Positive-shadow hours",
    "Zero-value binding hours",
    "Annual marginal value $/MW-year",
    "Mean shadow when positive $/MW-h",
    "Maximum shadow value $/MW-h",
]

print("Top 20 branches by annual marginal capacity value:")

top_branches_by_value = (
    branch_annual_metrics
    .sort_values(
        [
            "Annual marginal value $/MW-year",
            "Positive-shadow hours",
        ],
        ascending=False,
    )
    .head(20)
)

display(
    top_branches_by_value[
        economic_ranking_columns
    ]
)

print("Top 20 branches by positive-shadow hours:")

top_branches_by_hours = (
    branch_annual_metrics
    .sort_values(
        [
            "Positive-shadow hours",
            "Annual marginal value $/MW-year",
        ],
        ascending=False,
    )
    .head(20)
)

display(
    top_branches_by_hours[
        economic_ranking_columns
    ]
)

print("Top 20 branches by physical binding hours:")

top_branches_by_binding = (
    branch_annual_metrics
    .sort_values(
        [
            "Primary binding hours",
            "Positive-shadow hours",
        ],
        ascending=False,
    )
    .head(20)
)

display(
    top_branches_by_binding[
        economic_ranking_columns
    ]
)

In [ ]:
branch_corridor_data = branch_metadata.copy()

branch_corridor_data["Low Bus"] = (
    branch_corridor_data[
        ["From Bus", "To Bus"]
    ].min(axis=1).astype(int)
)

branch_corridor_data["High Bus"] = (
    branch_corridor_data[
        ["From Bus", "To Bus"]
    ].max(axis=1).astype(int)
)

branch_corridor_data["Corridor"] = (
    branch_corridor_data["Low Bus"].astype(str)
    + "-"
    + branch_corridor_data["High Bus"].astype(str)
)

corridor_records = []

for corridor, corridor_branches in (
    branch_corridor_data.groupby("Corridor")
):
    line_ids = corridor_branches.index.tolist()

    low_bus = int(
        corridor_branches["Low Bus"].iloc[0]
    )

    high_bus = int(
        corridor_branches["High Bus"].iloc[0]
    )

    primary_oriented_flow = pd.Series(
        0.0,
        index=annual_primary_flow.index,
    )

    secondary_oriented_flow = pd.Series(
        0.0,
        index=annual_primary_flow.index,
    )

    for line in line_ids:
        orientation = (
            1.0
            if branch_from_bus[line] == low_bus
            else -1.0
        )

        primary_oriented_flow += (
            orientation
            * full_year_results[
                "primary_flow"
            ][line]
        )

        secondary_oriented_flow += (
            orientation
            * full_year_results[
                "secondary_flow"
            ][line]
        )

    total_rating = sum(
        branch_rating[line]
        for line in line_ids
    )

    primary_corridor_utilization = (
        primary_oriented_flow.abs()
        / total_rating
    )

    secondary_corridor_utilization = (
        secondary_oriented_flow.abs()
        / total_rating
    )

    any_constituent_binding = (
        primary_binding_matrix[line_ids]
        .any(axis=1)
    )

    any_constituent_positive_shadow = (
        positive_shadow_matrix[line_ids]
        .any(axis=1)
    )

    # This sum represents the marginal value of adding 1 MW
    # to every physical circuit in the corridor.
    corridor_hourly_shadow_index = (
        annual_shadow_value[line_ids]
        .sum(axis=1)
    )

    corridor_records.append(
        {
            "Corridor": corridor,
            "Low Bus": low_bus,
            "High Bus": high_bus,
            "Physical branches": len(line_ids),
            "Branch UIDs": line_ids,
            "Total rating MW": total_rating,
            "Primary maximum corridor utilization":
                primary_corridor_utilization.max(),
            "Primary 99th-percentile corridor utilization":
                primary_corridor_utilization.quantile(0.99),
            "Secondary maximum corridor utilization":
                secondary_corridor_utilization.max(),
            "Hours corridor >=90%":
                int(
                    (
                        primary_corridor_utilization
                        >= 0.90
                    ).sum()
                ),
            "Hours corridor >=95%":
                int(
                    (
                        primary_corridor_utilization
                        >= 0.95
                    ).sum()
                ),
            "Hours corridor >=99%":
                int(
                    (
                        primary_corridor_utilization
                        >= 0.99
                    ).sum()
                ),
            "Hours any constituent binds":
                int(any_constituent_binding.sum()),
            "Positive-shadow hours":
                int(
                    any_constituent_positive_shadow.sum()
                ),
            "Annual shadow index $/year":
                corridor_hourly_shadow_index.sum(),
            "Maximum combined shadow $/h":
                corridor_hourly_shadow_index.max(),
        }
    )

corridor_annual_metrics = (
    pd.DataFrame(corridor_records)
    .set_index("Corridor")
)

corridor_annual_metrics[
    "Low Bus Area"
] = corridor_annual_metrics["Low Bus"].map(
    bus_metadata["Area"]
)

corridor_annual_metrics[
    "High Bus Area"
] = corridor_annual_metrics["High Bus"].map(
    bus_metadata["Area"]
)

print("Unique bus-pair corridors:", len(corridor_annual_metrics))

display(
    corridor_annual_metrics
    .sort_values(
        [
            "Annual shadow index $/year",
            "Positive-shadow hours",
        ],
        ascending=False,
    )
    .head(20)
)

## 12. Annual finite-difference validation of marginal transmission value

The shadow-price ranking measures the local value of transmission capacity.
For branch `l`, its predicted annual value for the first additional MW is:

$$
V_l^{\mathrm{dual}}
=
\sum_t \mu_{l,t}
$$

We independently test this using a 1 MW increase in the branch's symmetric
thermal rating:

$$
F_l^{\max,\mathrm{test}}
=
F_l^{\max}+1
$$

The annual finite-difference value is:

$$
V_l^{\mathrm{FD}}
=
\sum_t
\left[
C_t(F_l^{\max})
-
C_t(F_l^{\max}+1)
\right]
$$

If the 1 MW increment does not cross major dispatch breakpoints, the dual and
finite-difference values should be similar.

The finite-difference value may be lower than the dual prediction because the
shadow value is local and can decline within the 1 MW interval.

This experiment estimates the annual value of the first additional MW. It does
not yet estimate the value of a full reconductoring project.

In [ ]:
def run_primary_capacity_scenario(
    line_limit_overrides,
    timestamps,
    progress_every=2196,
):
    timestamps = pd.DatetimeIndex(timestamps)

    scenario_model = build_hourly_rts_opf()
    scenario_solver = pyo.SolverFactory(
        HOURLY_SOLVER_NAME
    )

    for line, new_limit in line_limit_overrides.items():
        if line not in BRANCH_IDS:
            raise KeyError(
                f"Unknown branch UID: {line}"
            )

        if new_limit <= 0:
            raise ValueError(
                f"Invalid limit for {line}: {new_limit}"
            )

        scenario_model.line_limit[line] = float(
            new_limit
        )

    number_of_hours = len(timestamps)

    hourly_cost = np.full(
        number_of_hours,
        np.nan,
    )

    overridden_line_flow = {
        line: np.full(number_of_hours, np.nan)
        for line in line_limit_overrides
    }

    maximum_validation_residual = 0.0

    residual_metrics = [
        "System balance error MW",
        "Maximum generator violation MW",
        "Maximum thermal violation MW",
        "Maximum DC-flow error MW",
        "Maximum nodal-balance error MW",
        "Reference-angle error radians",
    ]

    scenario_start = time.perf_counter()

    for position, timestamp in enumerate(timestamps):
        load_hour_into_model(
            scenario_model,
            timestamp,
        )

        solve_loaded_hour(
            scenario_model,
            scenario_solver,
        )

        validation = validate_hourly_solution(
            scenario_model,
            timestamp,
        )

        hourly_cost[position] = pyo.value(
            scenario_model.total_cost
        )

        for line in line_limit_overrides:
            overridden_line_flow[line][position] = (
                pyo.value(
                    scenario_model.flow[line]
                )
            )

        hour_maximum_residual = max(
            float(validation.loc[metric])
            for metric in residual_metrics
        )

        maximum_validation_residual = max(
            maximum_validation_residual,
            hour_maximum_residual,
        )

        if (
            progress_every is not None
            and (
                (position + 1) % progress_every == 0
                or position + 1 == number_of_hours
            )
        ):
            elapsed = (
                time.perf_counter()
                - scenario_start
            )

            print(
                f"Solved {position + 1:,}/"
                f"{number_of_hours:,} hours | "
                f"{elapsed:.2f}s elapsed"
            )

    elapsed_seconds = (
        time.perf_counter() - scenario_start
    )

    result = {
        "hourly_cost": pd.Series(
            hourly_cost,
            index=timestamps,
            name="Network cost $/h",
        ),
        "overridden_line_flow": pd.DataFrame(
            overridden_line_flow,
            index=timestamps,
        ),
        "line_limit_overrides":
            dict(line_limit_overrides),
        "maximum_validation_residual":
            maximum_validation_residual,
        "elapsed_seconds":
            elapsed_seconds,
    }

    return result

In [ ]:
economically_valuable_lines = (
    branch_annual_metrics
    .loc[
        branch_annual_metrics[
            "Positive-shadow hours"
        ]
        > 0
    ]
    .sort_values(
        "Annual marginal value $/MW-year",
        ascending=False,
    )
    .index
    .tolist()
)

print(
    "Economically valuable branches:",
    economically_valuable_lines,
)

print(
    "Number of +1 MW scenarios:",
    len(economically_valuable_lines),
)

assert len(economically_valuable_lines) == 10

CAPACITY_TEST_VERSION = "notebook4_plus1_v1"

CAPACITY_TEST_CHECKPOINT_PATH = (
    RESULTS_PATH
    / "notebook4_capacity_tests"
)

CAPACITY_TEST_CHECKPOINT_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

FORCE_RERUN_PLUS_ONE = False

plus_one_results = {}
plus_one_summary_records = []

baseline_hourly_network_cost = (
    annual_summary[
        "Primary network cost $/h"
    ]
)

for line in economically_valuable_lines:
    baseline_limit = float(
        branch_rating[line]
    )

    tested_limit = baseline_limit + 1.0

    safe_line_name = (
        line
        .replace("/", "_")
        .replace(" ", "_")
    )

    checkpoint_path = (
        CAPACITY_TEST_CHECKPOINT_PATH
        / (
            f"{CAPACITY_TEST_VERSION}_"
            f"{safe_line_name}.pkl.gz"
        )
    )

    if (
        checkpoint_path.exists()
        and not FORCE_RERUN_PLUS_ONE
    ):
        print(
            f"\nLoading +1 MW checkpoint for {line}"
        )

        scenario_result = pd.read_pickle(
            checkpoint_path,
            compression="gzip",
        )

        scenario_status = "loaded"

    else:
        print(
            f"\nTesting {line}: "
            f"{baseline_limit:.1f} MW "
            f"-> {tested_limit:.1f} MW"
        )

        scenario_result = (
            run_primary_capacity_scenario(
                line_limit_overrides={
                    line: tested_limit,
                },
                timestamps=expected_full_year_index,
                progress_every=2196,
            )
        )

        scenario_result["metadata"] = {
            "version": CAPACITY_TEST_VERSION,
            "branch": line,
            "baseline_limit_mw":
                baseline_limit,
            "tested_limit_mw":
                tested_limit,
            "capacity_increment_mw": 1.0,
        }

        temporary_path = (
            checkpoint_path.with_suffix(
                checkpoint_path.suffix + ".tmp"
            )
        )

        pd.to_pickle(
            scenario_result,
            temporary_path,
            compression="gzip",
        )

        temporary_path.replace(
            checkpoint_path
        )

        print(
            f"Saved: {checkpoint_path.name}"
        )

        scenario_status = "solved"

    metadata = scenario_result["metadata"]

    assert metadata["branch"] == line
    assert np.isclose(
        metadata["tested_limit_mw"],
        tested_limit,
    )

    assert scenario_result[
        "hourly_cost"
    ].index.equals(expected_full_year_index)

    assert (
        scenario_result[
            "maximum_validation_residual"
        ]
        <= VALIDATION_TOL
    )

    raw_hourly_savings = (
        baseline_hourly_network_cost
        - scenario_result["hourly_cost"]
    )

    minimum_raw_hourly_savings = (
        raw_hourly_savings.min()
    )

    assert minimum_raw_hourly_savings >= -1e-5

    reported_hourly_savings = (
        raw_hourly_savings.clip(lower=0.0)
    )

    annual_finite_difference_savings = (
        reported_hourly_savings.sum()
    )

    hours_with_positive_savings = int(
        (
            reported_hourly_savings > 1e-6
        ).sum()
    )

    tested_flow = scenario_result[
        "overridden_line_flow"
    ][line]

    tested_binding_hours = int(
        (
            (
                tested_flow.abs()
                - tested_limit
            ).abs()
            <= 1e-5
        ).sum()
    )

    dual_prediction = float(
        branch_annual_metrics.loc[
            line,
            "Annual marginal value $/MW-year",
        ]
    )

    plus_one_results[line] = scenario_result

    plus_one_summary_records.append(
        {
            "UID": line,
            "Status": scenario_status,
            "From Bus":
                branch_from_bus[line],
            "To Bus":
                branch_to_bus[line],
            "Baseline rating MW":
                baseline_limit,
            "Tested rating MW":
                tested_limit,
            "Baseline positive-shadow hours":
                int(
                    branch_annual_metrics.loc[
                        line,
                        "Positive-shadow hours",
                    ]
                ),
            "Hours with positive finite-difference savings":
                hours_with_positive_savings,
            "Binding hours at tested rating":
                tested_binding_hours,
            "Dual prediction $/MW-year":
                dual_prediction,
            "Finite-difference savings $/year":
                annual_finite_difference_savings,
            "Dual minus finite difference $":
                dual_prediction
                - annual_finite_difference_savings,
            "Finite difference / dual prediction":
                (
                    annual_finite_difference_savings
                    / dual_prediction
                    if dual_prediction > 0
                    else np.nan
                ),
            "Maximum hourly savings $/h":
                reported_hourly_savings.max(),
            "Minimum raw hourly savings $/h":
                minimum_raw_hourly_savings,
            "Maximum validation residual":
                scenario_result[
                    "maximum_validation_residual"
                ],
            "Runtime seconds":
                scenario_result["elapsed_seconds"],
        }
    )

plus_one_summary = (
    pd.DataFrame(plus_one_summary_records)
    .set_index("UID")
    .sort_values(
        "Finite-difference savings $/year",
        ascending=False,
    )
)

In [ ]:
display(plus_one_summary)

print(
    "Total +1 MW scenario runtime:",
    f"{plus_one_summary['Runtime seconds'].sum():.2f} seconds",
)

print(
    "Largest validation residual:",
    plus_one_summary[
        "Maximum validation residual"
    ].max(),
)

assert (
    plus_one_summary[
        "Maximum validation residual"
    ].max()
    <= VALIDATION_TOL
)

assert (
    plus_one_summary[
        "Minimum raw hourly savings $/h"
    ].min()
    >= -1e-5
)

print(
    "All +1 MW annual finite-difference "
    "scenarios passed validation."
)

## 13. Annual capacity sweeps for corridor finalists

The annual screening identifies three leading reconductoring candidates:

1. C6, buses 303–309;
2. CB-1, buses 318–223;
3. C29, buses 317–318.

A27 is retained as a benchmark because it was the leading candidate from the
single-hour peak analysis in Notebook 3.

For each branch, annual operating-cost savings at tested capacity `F` are:

$$
S_l(F)
=
\sum_t
\left[
C_t(F_l^{\mathrm{base}})
-
C_t(F)
\right]
$$

The tested capacities include:

- the original capacity;
- an additional 1 MW;
- 1%, 5%, 10%, 25%, 50%, and 100% rating increases;
- a 5,000 MW diagnostic limit representing an effectively unconstrained line.

The 5,000 MW case is not an engineering reconductoring proposal. It estimates
the maximum annual operating value obtainable by relaxing that one branch
while leaving the rest of the network unchanged.

In [ ]:
capacity_sweep_finalists = [
    "C6",
    "CB-1",
    "C29",
    "A27",
]

for line in capacity_sweep_finalists:
    assert line in BRANCH_IDS
    assert (
        branch_annual_metrics.loc[
            line,
            "Positive-shadow hours",
        ]
        > 0
    )

capacity_sweep_multipliers = [
    1.01,
    1.05,
    1.10,
    1.25,
    1.50,
    2.00,
]

capacity_sweep_grids = {}

for line in capacity_sweep_finalists:
    baseline_limit = float(
        branch_rating[line]
    )

    tested_limits = {
        baseline_limit,
        baseline_limit + 1.0,
        *[
            baseline_limit * multiplier
            for multiplier
            in capacity_sweep_multipliers
        ],
        5000.0,
    }

    capacity_sweep_grids[line] = sorted(
        tested_limits
    )

    print(
        line,
        "baseline:",
        baseline_limit,
        "tested limits:",
        capacity_sweep_grids[line],
    )

In [ ]:
CAPACITY_SWEEP_VERSION = "notebook4_sweep_v1"

CAPACITY_SWEEP_CHECKPOINT_PATH = (
    RESULTS_PATH
    / "notebook4_capacity_sweeps"
)

CAPACITY_SWEEP_CHECKPOINT_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

FORCE_RERUN_CAPACITY_SWEEPS = False

capacity_sweep_results = {}
capacity_sweep_records = []

for line in capacity_sweep_finalists:
    baseline_limit = float(
        branch_rating[line]
    )

    baseline_binding_hours = (
        primary_binding_matrix[line]
    )

    baseline_valuable_hours = (
        positive_shadow_matrix[line]
    )

    # A nonbinding constraint cannot affect the solution when relaxed.
    screening_mask = (
        baseline_binding_hours
        | baseline_valuable_hours
    )

    screened_timestamps = (
        expected_full_year_index[
            screening_mask.to_numpy()
        ]
    )

    print(
        f"\n{line}: screening "
        f"{len(screened_timestamps):,} relevant hours "
        f"instead of 8,784."
    )

    baseline_screened_cost = (
        annual_summary.loc[
            screened_timestamps,
            "Primary network cost $/h",
        ]
    )

    capacity_sweep_results[line] = {}

    for tested_limit in capacity_sweep_grids[line]:
        capacity_uplift = (
            tested_limit - baseline_limit
        )

        capacity_multiplier = (
            tested_limit / baseline_limit
        )

        if np.isclose(
            tested_limit,
            baseline_limit,
            atol=1e-12,
        ):
            baseline_flow = (
                full_year_results[
                    "primary_flow"
                ]
                .loc[screened_timestamps, line]
            )

            annual_savings = 0.0
            positive_savings_hours = 0

            binding_hours_at_tested_limit = int(
                (
                    (
                        baseline_flow.abs()
                        - baseline_limit
                    ).abs()
                    <= 1e-5
                ).sum()
            )

            maximum_absolute_flow = (
                baseline_flow.abs().max()
            )

            maximum_utilization = (
                maximum_absolute_flow
                / tested_limit
            )

            maximum_validation_residual = 0.0
            runtime_seconds = 0.0
            scenario_status = "baseline"

            scenario_result = {
                "hourly_cost":
                    baseline_screened_cost.copy(),
                "overridden_line_flow":
                    baseline_flow.to_frame(line),
                "line_limit_overrides": {
                    line: baseline_limit,
                },
                "maximum_validation_residual": 0.0,
                "elapsed_seconds": 0.0,
                "metadata": {
                    "version":
                        CAPACITY_SWEEP_VERSION,
                    "branch": line,
                    "baseline_limit_mw":
                        baseline_limit,
                    "tested_limit_mw":
                        tested_limit,
                    "screened_hours":
                        len(screened_timestamps),
                },
            }

        else:
            safe_line_name = (
                line
                .replace("/", "_")
                .replace(" ", "_")
            )

            safe_limit_name = (
                f"{tested_limit:.6f}"
                .replace(".", "p")
            )

            checkpoint_path = (
                CAPACITY_SWEEP_CHECKPOINT_PATH
                / (
                    f"{CAPACITY_SWEEP_VERSION}_"
                    f"{safe_line_name}_"
                    f"{safe_limit_name}MW.pkl.gz"
                )
            )

            if (
                checkpoint_path.exists()
                and not FORCE_RERUN_CAPACITY_SWEEPS
            ):
                print(
                    f"Loading {line} at "
                    f"{tested_limit:.3f} MW"
                )

                scenario_result = pd.read_pickle(
                    checkpoint_path,
                    compression="gzip",
                )

                scenario_status = "loaded"

            else:
                print(
                    f"Solving {line}: "
                    f"{baseline_limit:.3f} -> "
                    f"{tested_limit:.3f} MW"
                )

                scenario_result = (
                    run_primary_capacity_scenario(
                        line_limit_overrides={
                            line: tested_limit,
                        },
                        timestamps=screened_timestamps,
                        progress_every=None,
                    )
                )

                scenario_result["metadata"] = {
                    "version":
                        CAPACITY_SWEEP_VERSION,
                    "branch": line,
                    "baseline_limit_mw":
                        baseline_limit,
                    "tested_limit_mw":
                        tested_limit,
                    "screened_hours":
                        len(screened_timestamps),
                }

                temporary_path = (
                    checkpoint_path.with_suffix(
                        checkpoint_path.suffix
                        + ".tmp"
                    )
                )

                pd.to_pickle(
                    scenario_result,
                    temporary_path,
                    compression="gzip",
                )

                temporary_path.replace(
                    checkpoint_path
                )

                scenario_status = "solved"

            metadata = scenario_result[
                "metadata"
            ]

            assert metadata["branch"] == line
            assert np.isclose(
                metadata["tested_limit_mw"],
                tested_limit,
            )
            assert (
                metadata["screened_hours"]
                == len(screened_timestamps)
            )

            assert scenario_result[
                "hourly_cost"
            ].index.equals(
                screened_timestamps
            )

            raw_hourly_savings = (
                baseline_screened_cost
                - scenario_result["hourly_cost"]
            )

            assert raw_hourly_savings.min() >= -1e-5

            reported_hourly_savings = (
                raw_hourly_savings.clip(lower=0.0)
            )

            annual_savings = (
                reported_hourly_savings.sum()
            )

            positive_savings_hours = int(
                (
                    reported_hourly_savings
                    > 1e-6
                ).sum()
            )

            tested_flow = scenario_result[
                "overridden_line_flow"
            ][line]

            binding_hours_at_tested_limit = int(
                (
                    (
                        tested_flow.abs()
                        - tested_limit
                    ).abs()
                    <= 1e-5
                ).sum()
            )

            maximum_absolute_flow = (
                tested_flow.abs().max()
            )

            maximum_utilization = (
                maximum_absolute_flow
                / tested_limit
            )

            maximum_validation_residual = (
                scenario_result[
                    "maximum_validation_residual"
                ]
            )

            runtime_seconds = (
                scenario_result["elapsed_seconds"]
            )

        capacity_sweep_results[
            line
        ][tested_limit] = scenario_result

        capacity_sweep_records.append(
            {
                "UID": line,
                "Baseline rating MW":
                    baseline_limit,
                "Tested rating MW":
                    tested_limit,
                "Capacity uplift MW":
                    capacity_uplift,
                "Capacity multiplier":
                    capacity_multiplier,
                "Screened hours":
                    len(screened_timestamps),
                "Annual operating-cost savings $":
                    annual_savings,
                "Share of annual congestion cost removed":
                    annual_savings
                    / annual_congestion_cost,
                "Hours with positive savings":
                    positive_savings_hours,
                "Binding hours at tested rating":
                    binding_hours_at_tested_limit,
                "Maximum absolute flow MW":
                    maximum_absolute_flow,
                "Maximum utilization":
                    maximum_utilization,
                "Maximum validation residual":
                    maximum_validation_residual,
                "Runtime seconds":
                    runtime_seconds,
                "Status":
                    scenario_status,
            }
        )

capacity_sweep_summary = (
    pd.DataFrame(capacity_sweep_records)
    .sort_values(
        [
            "UID",
            "Tested rating MW",
        ]
    )
    .reset_index(drop=True)
)

In [ ]:
for line in capacity_sweep_finalists:
    line_sweep = (
        capacity_sweep_summary
        .loc[
            capacity_sweep_summary["UID"].eq(line)
        ]
        .sort_values("Tested rating MW")
    )

    savings_changes = (
        line_sweep[
            "Annual operating-cost savings $"
        ]
        .diff()
        .dropna()
    )

    assert (
        savings_changes >= -1e-4
    ).all(), (
        f"Annual savings are not monotonic for {line}."
    )

    assert (
        line_sweep[
            "Maximum validation residual"
        ].max()
        <= VALIDATION_TOL
    )

    plus_one_row = line_sweep.loc[
        np.isclose(
            line_sweep["Capacity uplift MW"],
            1.0,
            atol=1e-9,
        )
    ]

    assert len(plus_one_row) == 1

    previous_plus_one_value = (
        plus_one_summary.loc[
            line,
            "Finite-difference savings $/year",
        ]
    )

    sweep_plus_one_value = (
        plus_one_row[
            "Annual operating-cost savings $"
        ].iloc[0]
    )

    assert np.isclose(
        sweep_plus_one_value,
        previous_plus_one_value,
        atol=1e-3,
    )

print(
    "All finalist capacity sweeps passed validation."
)

for line in capacity_sweep_finalists:
    print(f"\nCapacity sweep for {line}:")

    display(
        capacity_sweep_summary.loc[
            capacity_sweep_summary["UID"].eq(line),
            [
                "UID",
                "Baseline rating MW",
                "Tested rating MW",
                "Capacity uplift MW",
                "Capacity multiplier",
                "Annual operating-cost savings $",
                "Share of annual congestion cost removed",
                "Hours with positive savings",
                "Binding hours at tested rating",
                "Maximum absolute flow MW",
                "Maximum utilization",
                "Runtime seconds",
            ],
        ]
    )

In [ ]:
unconstrained_comparison_records = []

for line in capacity_sweep_finalists:
    line_sweep = capacity_sweep_summary.loc[
        capacity_sweep_summary["UID"].eq(line)
    ]

    unconstrained_row = line_sweep.loc[
        line_sweep["Tested rating MW"].eq(5000.0)
    ].iloc[0]

    one_and_half_row = line_sweep.loc[
        np.isclose(
            line_sweep["Capacity multiplier"],
            1.50,
            atol=1e-9,
        )
    ].iloc[0]

    unconstrained_comparison_records.append(
        {
            "UID": line,
            "Baseline rating MW":
                unconstrained_row[
                    "Baseline rating MW"
                ],
            "Maximum annual operating value $":
                unconstrained_row[
                    "Annual operating-cost savings $"
                ],
            "Maximum share of congestion cost removable":
                unconstrained_row[
                    "Share of annual congestion cost removed"
                ],
            "Economically desired maximum flow MW":
                unconstrained_row[
                    "Maximum absolute flow MW"
                ],
            "Savings at 1.5x rating $":
                one_and_half_row[
                    "Annual operating-cost savings $"
                ],
            "1.5x share of maximum line value":
                (
                    one_and_half_row[
                        "Annual operating-cost savings $"
                    ]
                    / unconstrained_row[
                        "Annual operating-cost savings $"
                    ]
                    if unconstrained_row[
                        "Annual operating-cost savings $"
                    ] > 0
                    else np.nan
                ),
        }
    )

unconstrained_comparison = (
    pd.DataFrame(
        unconstrained_comparison_records
    )
    .set_index("UID")
    .sort_values(
        "Maximum annual operating value $",
        ascending=False,
    )
)

display(unconstrained_comparison)

print(
    "Total capacity-sweep runtime:",
    f"{capacity_sweep_summary['Runtime seconds'].sum():.2f} seconds",
)

## 14. Engineering audit and temporal profile of the leading corridor

C6 is the leading economic candidate, but economic ranking alone is not
sufficient for a reconductoring interpretation.

Before selecting it, the branch is checked for:

- endpoint voltage levels;
- transformer ratio;
- resistance and reactance;
- line length;
- geographic area;
- original thermal rating.

The annual sweep also distinguishes technical capacity uplift from economic
value.

For a tested rating `F`, the share of the line's maximum modeled value captured is:

$$
\eta(F)
=
\frac{S(F)}
{S(F^{\mathrm{unconstrained}})}
$$

The incremental value over a capacity interval is:

$$
MV_k
=
\frac{
S(F_k)-S(F_{k-1})
}{
F_k-F_{k-1}
}
$$

This quantity shows the average annual value per additional MW within each
capacity interval.

In [ ]:
LEADING_CORRIDOR = "C6"

c6_branch_data = branch_metadata.loc[
    LEADING_CORRIDOR
].copy()

c6_from_bus = int(
    c6_branch_data["From Bus"]
)

c6_to_bus = int(
    c6_branch_data["To Bus"]
)

c6_endpoint_data = bus_metadata.loc[
    [c6_from_bus, c6_to_bus],
    [
        "Bus Name",
        "BaseKV",
        "Area",
        "Sub Area",
        "Zone",
        "lat",
        "lng",
    ],
].copy()

print("C6 branch metadata:")

display(
    c6_branch_data[
        [
            "From Bus",
            "To Bus",
            "R",
            "X",
            "B",
            "Cont Rating",
            "LTE Rating",
            "STE Rating",
            "Tr Ratio",
            "Length",
        ]
    ].to_frame(name="Value")
)

print("C6 endpoint buses:")

display(c6_endpoint_data)

same_voltage_level = np.isclose(
    c6_endpoint_data.loc[
        c6_from_bus,
        "BaseKV",
    ],
    c6_endpoint_data.loc[
        c6_to_bus,
        "BaseKV",
    ],
)

print(
    "Endpoints have the same base voltage:",
    same_voltage_level,
)

print(
    "Endpoint voltage:",
    c6_endpoint_data["BaseKV"].tolist(),
    "kV",
)

print(
    "Transformer ratio field:",
    c6_branch_data["Tr Ratio"],
)

print(
    "Both endpoints are in Area 3:",
    (
        c6_endpoint_data["Area"]
        .eq(3)
        .all()
    ),
)

In [ ]:
c6_sweep = (
    capacity_sweep_summary
    .loc[
        capacity_sweep_summary[
            "UID"
        ].eq(LEADING_CORRIDOR)
    ]
    .sort_values("Tested rating MW")
    .copy()
)

c6_maximum_value = (
    c6_sweep.loc[
        c6_sweep["Tested rating MW"].eq(5000.0),
        "Annual operating-cost savings $",
    ]
    .iloc[0]
)

c6_sweep[
    "Share of maximum C6 value"
] = (
    c6_sweep[
        "Annual operating-cost savings $"
    ]
    / c6_maximum_value
)

c6_sweep[
    "Incremental capacity MW"
] = (
    c6_sweep["Tested rating MW"].diff()
)

c6_sweep[
    "Incremental annual savings $"
] = (
    c6_sweep[
        "Annual operating-cost savings $"
    ].diff()
)

c6_sweep[
    "Average interval value $/MW-year"
] = (
    c6_sweep[
        "Incremental annual savings $"
    ]
    / c6_sweep[
        "Incremental capacity MW"
    ]
)

display(
    c6_sweep[
        [
            "Tested rating MW",
            "Capacity uplift MW",
            "Capacity multiplier",
            "Annual operating-cost savings $",
            "Share of maximum C6 value",
            "Incremental annual savings $",
            "Average interval value $/MW-year",
            "Binding hours at tested rating",
            "Maximum absolute flow MW",
        ]
    ]
)

c6_unconstrained_row = c6_sweep.loc[
    c6_sweep["Tested rating MW"].eq(5000.0)
].iloc[0]

c6_one_and_half_row = c6_sweep.loc[
    np.isclose(
        c6_sweep["Capacity multiplier"],
        1.5,
        atol=1e-9,
    )
].iloc[0]

c6_sufficient_rating = (
    c6_unconstrained_row[
        "Maximum absolute flow MW"
    ]
)

c6_sufficient_uplift = (
    c6_sufficient_rating
    - branch_rating[LEADING_CORRIDOR]
)

c6_sufficient_multiplier = (
    c6_sufficient_rating
    / branch_rating[LEADING_CORRIDOR]
)

print(
    "C6 maximum annual operating value:",
    f"${c6_maximum_value:,.2f}",
)

print(
    "C6 1.5x annual savings:",
    f"${c6_one_and_half_row['Annual operating-cost savings $']:,.2f}",
)

print(
    "C6 1.5x share of maximum value:",
    f"{c6_one_and_half_row['Share of maximum C6 value']:.4%}",
)

print(
    "Rating sufficient to reproduce the "
    "unconstrained-line flow:",
    f"{c6_sufficient_rating:.6f} MW",
)

print(
    "Sufficient capacity uplift:",
    f"{c6_sufficient_uplift:.6f} MW",
)

print(
    "Sufficient capacity multiplier:",
    f"{c6_sufficient_multiplier:.6f}x",
)

In [ ]:
def reconstruct_full_year_savings(
    line,
    tested_limit,
):
    scenario_result = (
        capacity_sweep_results[
            line
        ][tested_limit]
    )

    scenario_index = scenario_result[
        "hourly_cost"
    ].index

    full_year_savings = pd.Series(
        0.0,
        index=expected_full_year_index,
        name="Operating-cost savings $/h",
    )

    raw_savings = (
        annual_summary.loc[
            scenario_index,
            "Primary network cost $/h",
        ]
        - scenario_result["hourly_cost"]
    )

    full_year_savings.loc[
        scenario_index
    ] = raw_savings.clip(lower=0.0)

    return full_year_savings


c6_limit_1_5x = (
    branch_rating[LEADING_CORRIDOR]
    * 1.5
)

c6_savings_1_5x = (
    reconstruct_full_year_savings(
        LEADING_CORRIDOR,
        c6_limit_1_5x,
    )
)

c6_savings_unconstrained = (
    reconstruct_full_year_savings(
        LEADING_CORRIDOR,
        5000.0,
    )
)

c6_hourly_profile = pd.DataFrame(
    {
        "Demand MW":
            annual_summary["Demand MW"],
        "VRE available MW":
            annual_summary["VRE available MW"],
        "Flow MW":
            full_year_results[
                "primary_flow"
            ][LEADING_CORRIDOR],
        "Utilization":
            full_year_results[
                "primary_utilization"
            ][LEADING_CORRIDOR],
        "Binding":
            primary_binding_matrix[
                LEADING_CORRIDOR
            ],
        "Positive shadow":
            positive_shadow_matrix[
                LEADING_CORRIDOR
            ],
        "Shadow value $/MW-h":
            annual_shadow_value[
                LEADING_CORRIDOR
            ],
        "Savings at 1.5x $/h":
            c6_savings_1_5x,
        "Maximum possible savings $/h":
            c6_savings_unconstrained,
    }
)

c6_hourly_profile["Month"] = (
    c6_hourly_profile.index.to_period("M")
)

c6_hourly_profile["Hour of day"] = (
    c6_hourly_profile.index.hour
)

c6_monthly_profile = (
    c6_hourly_profile
    .groupby("Month")
    .agg(
        Hours=("Demand MW", "count"),
        Mean_Demand_MW=("Demand MW", "mean"),
        Mean_VRE_Available_MW=(
            "VRE available MW",
            "mean",
        ),
        Hours_Above_90_Percent=(
            "Utilization",
            lambda x: int((x >= 0.90).sum()),
        ),
        Binding_Hours=("Binding", "sum"),
        Positive_Shadow_Hours=(
            "Positive shadow",
            "sum",
        ),
        Sum_Shadow_Value=(
            "Shadow value $/MW-h",
            "sum",
        ),
        Savings_At_1_5x_Dollars=(
            "Savings at 1.5x $/h",
            "sum",
        ),
        Maximum_Possible_Savings_Dollars=(
            "Maximum possible savings $/h",
            "sum",
        ),
    )
)

c6_monthly_profile[
    "1.5x Share of Maximum Monthly Value"
] = (
    c6_monthly_profile[
        "Savings_At_1_5x_Dollars"
    ]
    / c6_monthly_profile[
        "Maximum_Possible_Savings_Dollars"
    ].replace(0.0, np.nan)
)

print("C6 monthly profile:")

display(c6_monthly_profile)

c6_hour_of_day_profile = (
    c6_hourly_profile
    .groupby("Hour of day")
    .agg(
        Observations=("Demand MW", "count"),
        Mean_Demand_MW=("Demand MW", "mean"),
        Mean_VRE_Available_MW=(
            "VRE available MW",
            "mean",
        ),
        Hours_Above_90_Percent=(
            "Utilization",
            lambda x: int((x >= 0.90).sum()),
        ),
        Binding_Hours=("Binding", "sum"),
        Positive_Shadow_Hours=(
            "Positive shadow",
            "sum",
        ),
        Sum_Shadow_Value=(
            "Shadow value $/MW-h",
            "sum",
        ),
        Savings_At_1_5x_Dollars=(
            "Savings at 1.5x $/h",
            "sum",
        ),
    )
)

print("C6 time-of-day profile:")

display(c6_hour_of_day_profile)

c6_direction_summary = pd.Series(
    {
        "Positive upper-direction shadow hours":
            int(
                (
                    upper_shadow_value[
                        LEADING_CORRIDOR
                    ]
                    > 1e-7
                ).sum()
            ),
        "Positive lower-direction shadow hours":
            int(
                (
                    lower_shadow_value[
                        LEADING_CORRIDOR
                    ]
                    > 1e-7
                ).sum()
            ),
        "Upper-direction annual marginal value":
            upper_shadow_value[
                LEADING_CORRIDOR
            ].sum(),
        "Lower-direction annual marginal value":
            lower_shadow_value[
                LEADING_CORRIDOR
            ].sum(),
    },
    name="Value",
)

print("C6 congestion direction:")

display(c6_direction_summary.to_frame())

## 12. Selection of the Baseline Reconductoring Scenario

The annual screening identifies branch **C6**, connecting buses 303 and
309, as the principal reconductoring candidate.

C6 is a 138 kV line located entirely within Area 3. Its transformer-ratio
field is zero, and both endpoints have the same base voltage. It is therefore
treated as a transmission line rather than a transformer.

The baseline continuous rating is

$$
\overline{F}^{\,0}_{\mathrm{C6}} = 175\ \mathrm{MW}.
$$

The primary reconductoring scenario applies a 50% thermal-rating increase:

$$
m^{R}=1.50,
$$

so that the post-project rating is

$$
\overline{F}^{\,R}_{\mathrm{C6}}
=
m^{R}\overline{F}^{\,0}_{\mathrm{C6}}
=
1.50(175)
=
262.5\ \mathrm{MW}.
$$

The resulting capacity uplift is

$$
\Delta \overline{F}_{\mathrm{C6}}
=
262.5-175
=
87.5\ \mathrm{MW}.
$$

In the baseline 2020 simulation, this increase reduces annual operating cost
by approximately \$5.301 million. It captures 92.18% of the maximum operating
value obtainable by completely relaxing C6 and removes approximately 68.95%
of total baseline annual congestion cost.

The economic value of C6 is strongly directional. Every hour with a positive
C6 capacity shadow value is associated with the upper-flow constraint, meaning
that the valuable direction is from bus 303 toward bus 309.

The value is also temporally concentrated. Congestion occurs most frequently
during winter months and during overnight or late-evening hours. Consequently,
neither annual peak demand nor maximum solar availability alone explains the
economic importance of this line.

The 1.50 multiplier is the canonical reconductoring case. Multipliers of
1.25 and 2.00 will be retained as sensitivity cases:

$$
m^{R}\in\{1.25,\ 1.50,\ 2.00\}.
$$

The approximately 325 MW flow observed when C6 is effectively unconstrained
shows that a rating near 1.86 times the original rating is sufficient to
reproduce the unconstrained-line dispatch. It is not interpreted as an exact
engineering minimum because alternative cost-equivalent dispatches may exist.

This experiment changes only the modeled continuous thermal limit. Resistance,
reactance, topology, and all other branch parameters remain fixed. Therefore,
the scenario is a reduced-form representation of reconductoring or line
uprating rather than a detailed electrical and structural conductor design.

Finally, the calculated operating-cost savings exclude capital cost and
construction feasibility. C6 is selected as the most economically relevant
network intervention for the data-center flexibility experiments, not as a
complete transmission investment recommendation.

In [ ]:
# Diagnose the stored capacity-sweep structure without modifying it.

print("capacity_sweep_results type:")
print(type(capacity_sweep_results))

print("\nTop-level keys:")
print(list(capacity_sweep_results.keys()))

c6_payload = capacity_sweep_results["C6"]

print("\nC6 payload type:")
print(type(c6_payload))

if isinstance(c6_payload, pd.DataFrame):
    print("\nC6 DataFrame shape:")
    print(c6_payload.shape)

    print("\nC6 DataFrame columns:")
    print(c6_payload.columns.tolist())

    print("\nC6 DataFrame index:")
    print(c6_payload.index.tolist()[:20])

    display(c6_payload.head())

elif isinstance(c6_payload, dict):
    print("\nC6 nested dictionary keys:")
    print(list(c6_payload.keys()))

    print("\nNested value types:")
    for key, value in c6_payload.items():
        print(
            repr(key),
            "->",
            type(value),
            getattr(value, "shape", ""),
        )

elif isinstance(c6_payload, (list, tuple)):
    print("\nC6 payload length:")
    print(len(c6_payload))

    if len(c6_payload) > 0:
        print("\nFirst element type:")
        print(type(c6_payload[0]))

        print("\nFirst element:")
        print(c6_payload[0])

else:
    print("\nC6 payload representation:")
    print(repr(c6_payload)[:2000])

# Locate any existing DataFrame containing the completed sweep table.
required_columns = {
    "UID",
    "Tested rating MW",
    "Capacity multiplier",
    "Annual operating-cost savings $",
}

print("\nExisting compatible DataFrames:")

compatible_frames = []

for variable_name, value in list(globals().items()):
    if (
        isinstance(value, pd.DataFrame)
        and required_columns.issubset(value.columns)
    ):
        compatible_frames.append(variable_name)
        print(
            variable_name,
            "| shape:",
            value.shape,
            "| C6 rows:",
            int(value["UID"].eq("C6").sum()),
        )

if not compatible_frames:
    print("No compatible DataFrame was found.")

In [ ]:
# Formalize the reconductoring case used in subsequent notebooks.

SELECTED_RECONDUCTORING_UID = "C6"
SELECTED_RECONDUCTORING_MULTIPLIER = 1.50
RECONDUCTORING_SENSITIVITY_MULTIPLIERS = (1.25, 1.50, 2.00)

selected_branch = (
    branch.loc[
        branch["UID"].eq(SELECTED_RECONDUCTORING_UID)
    ]
    .iloc[0]
)

# Use the combined capacity-sweep summary table.
c6_sweep_for_selection = (
    capacity_sweep_summary.loc[
        capacity_sweep_summary["UID"].eq(
            SELECTED_RECONDUCTORING_UID
        )
    ]
    .sort_values("Tested rating MW")
    .copy()
)

assert len(c6_sweep_for_selection) == 9

selected_rows = c6_sweep_for_selection.loc[
    np.isclose(
        c6_sweep_for_selection["Capacity multiplier"],
        SELECTED_RECONDUCTORING_MULTIPLIER,
    )
]

assert len(selected_rows) == 1, (
    "The C6 sweep must contain exactly one 1.5x case."
)

selected_sweep_row = selected_rows.iloc[0]

maximum_value_row = c6_sweep_for_selection.loc[
    c6_sweep_for_selection[
        "Annual operating-cost savings $"
    ].idxmax()
]

baseline_rating_mw = float(selected_branch["Cont Rating"])
selected_rating_mw = float(
    selected_sweep_row["Tested rating MW"]
)
capacity_uplift_mw = (
    selected_rating_mw - baseline_rating_mw
)

maximum_c6_value = float(
    maximum_value_row["Annual operating-cost savings $"]
)

selected_c6_value = float(
    selected_sweep_row["Annual operating-cost savings $"]
)

from_bus = int(selected_branch["From Bus"])
to_bus = int(selected_branch["To Bus"])

endpoint_data = (
    bus.set_index("Bus ID")
    .loc[[from_bus, to_bus]]
)

# Engineering and numerical validation.
assert np.isclose(baseline_rating_mw, 175.0)
assert np.isclose(selected_rating_mw, 262.5)
assert np.isclose(capacity_uplift_mw, 87.5)

assert np.isclose(
    selected_rating_mw,
    baseline_rating_mw
    * SELECTED_RECONDUCTORING_MULTIPLIER,
)

assert endpoint_data["BaseKV"].nunique() == 1
assert np.isclose(float(selected_branch["Tr Ratio"]), 0.0)
assert endpoint_data["Area"].nunique() == 1

assert selected_c6_value > 0.0
assert selected_c6_value / maximum_c6_value >= 0.90

selected_reconductoring_case = pd.Series(
    {
        "Selected branch UID": (
            SELECTED_RECONDUCTORING_UID
        ),
        "From bus": from_bus,
        "To bus": to_bus,
        "Endpoint voltage kV": float(
            endpoint_data["BaseKV"].iloc[0]
        ),
        "Branch length field": float(
            selected_branch["Length"]
        ),
        "Baseline continuous rating MW": (
            baseline_rating_mw
        ),
        "Selected capacity multiplier": (
            SELECTED_RECONDUCTORING_MULTIPLIER
        ),
        "Selected continuous rating MW": (
            selected_rating_mw
        ),
        "Capacity uplift MW": capacity_uplift_mw,
        "Annual operating-cost savings $": (
            selected_c6_value
        ),
        "Share of maximum C6 value": (
            selected_c6_value / maximum_c6_value
        ),
        "Share of annual congestion cost removed": float(
            selected_sweep_row[
                "Share of annual congestion cost removed"
            ]
        ),
        "Binding hours after uprating": int(
            selected_sweep_row[
                "Binding hours at tested rating"
            ]
        ),
        "Sufficient unconstrained-flow rating MW": float(
            maximum_value_row[
                "Maximum absolute flow MW"
            ]
        ),
        "Economically valuable direction": (
            f"{from_bus} -> {to_bus}"
        ),
        "Sensitivity multipliers": (
            RECONDUCTORING_SENSITIVITY_MULTIPLIERS
        ),
        "Impedance treatment": "R and X held fixed",
        "LTE/STE treatment": (
            "Not used in the energy-only DC-OPF"
        ),
    },
    name="Value",
)

display(selected_reconductoring_case.to_frame())

print(
    "\nSelected reconductoring case:"
    f" {SELECTED_RECONDUCTORING_UID},"
    f" {baseline_rating_mw:.1f} MW"
    f" -> {selected_rating_mw:.1f} MW."
)

print(
    "The canonical 1.5x case captures"
    f" {selected_c6_value / maximum_c6_value:.4%}"
    " of C6's maximum annual operating value."
)

print(
    "Sensitivity multipliers:",
    RECONDUCTORING_SENSITIVITY_MULTIPLIERS,
)

## 13. Formal Definition of the Reconductoring Case

Two network configurations are retained for the subsequent experiments.

The original network is denoted by **C0**. For branch C6, its continuous
thermal limit is

$$
\overline{F}_{\mathrm{C6}}^{C0}
=
175\ \mathrm{MW}.
$$

The selected reconductoring configuration is denoted by **CR**. Its C6
thermal limit is

$$
\overline{F}_{\mathrm{C6}}^{CR}
=
262.5\ \mathrm{MW}.
$$

All other branch ratings and network parameters remain unchanged.

For the baseline demand and generation profiles, the annual operating-cost
saving produced by the reconductoring case is

$$
S^{R}
=
C^{C0}-C^{CR},
$$

where:

- $C^{C0}$ is annual operating cost with the original network; and
- $C^{CR}$ is annual operating cost after increasing the C6 rating.

The calculated baseline-system saving is

$$
S^{R}
=
\$5{,}301{,}048.13.
$$

This saving is equal to 68.95% of the total congestion cost observed in the
original network. The residual annual congestion cost is therefore

$$
C_{\mathrm{cong}}^{CR}
=
C_{\mathrm{cong}}^{C0}-S^{R}.
$$

Although the selected upgrade captures 92.18% of the maximum value available
from relaxing C6, it does not eliminate all C6 congestion. The line remains
binding during 661 hours after the rating increase.

The calculated saving is a baseline calibration result. It must not be assumed
to remain constant after data-center demand or data-center flexibility is
introduced. In the later experiments, each case will be solved independently
so that the interaction between flexibility and reconductoring is measured
endogenously.

In [ ]:
# Define and validate the baseline-network and reconductoring cases.

CASE_C0 = "C0"
CASE_CR = "CR"

AUDITED_BASELINE_CONGESTION_COST_DOLLARS = (
    7_688_308.591549
)

baseline_c6_row = c6_sweep_for_selection.loc[
    np.isclose(
        c6_sweep_for_selection["Capacity multiplier"],
        1.0,
    )
].iloc[0]

selected_congestion_share = float(
    selected_sweep_row[
        "Share of annual congestion cost removed"
    ]
)

# Recover the same baseline congestion cost from the sweep result.
implied_baseline_congestion_cost = (
    selected_c6_value / selected_congestion_share
)

residual_congestion_cost = (
    implied_baseline_congestion_cost
    - selected_c6_value
)

remaining_c6_value = (
    maximum_c6_value - selected_c6_value
)

baseline_binding_hours = int(
    baseline_c6_row["Binding hours at tested rating"]
)

selected_binding_hours = int(
    selected_sweep_row["Binding hours at tested rating"]
)

binding_hours_removed = (
    baseline_binding_hours - selected_binding_hours
)

binding_hour_reduction_share = (
    binding_hours_removed / baseline_binding_hours
)

hours_with_operating_savings = int(
    selected_sweep_row["Hours with positive savings"]
)

# Numerical consistency checks.
assert np.isclose(
    implied_baseline_congestion_cost,
    AUDITED_BASELINE_CONGESTION_COST_DOLLARS,
    atol=0.01,
)

assert np.isclose(
    selected_c6_value + residual_congestion_cost,
    implied_baseline_congestion_cost,
)

assert np.isclose(
    selected_c6_value / implied_baseline_congestion_cost,
    selected_congestion_share,
)

assert selected_binding_hours < baseline_binding_hours
assert remaining_c6_value >= -1e-6
assert residual_congestion_cost >= -1e-6

reconductoring_case_comparison = pd.DataFrame(
    [
        {
            "Case": CASE_C0,
            "Network configuration": "Original network",
            "C6 rating MW": baseline_rating_mw,
            "C6 capacity uplift MW": 0.0,
            "C6 binding hours": baseline_binding_hours,
            "Hours with operating-cost savings": 0,
            "Annual operating-cost savings $": 0.0,
            "Residual annual congestion cost $": (
                implied_baseline_congestion_cost
            ),
            "Share of baseline congestion removed": 0.0,
            "Share of maximum C6 value captured": 0.0,
        },
        {
            "Case": CASE_CR,
            "Network configuration": "C6 thermal uprating",
            "C6 rating MW": selected_rating_mw,
            "C6 capacity uplift MW": capacity_uplift_mw,
            "C6 binding hours": selected_binding_hours,
            "Hours with operating-cost savings": (
                hours_with_operating_savings
            ),
            "Annual operating-cost savings $": (
                selected_c6_value
            ),
            "Residual annual congestion cost $": (
                residual_congestion_cost
            ),
            "Share of baseline congestion removed": (
                selected_congestion_share
            ),
            "Share of maximum C6 value captured": (
                selected_c6_value / maximum_c6_value
            ),
        },
    ]
).set_index("Case")

display(reconductoring_case_comparison)

reconductoring_accounting = pd.Series(
    {
        "Baseline annual congestion cost $": (
            implied_baseline_congestion_cost
        ),
        "CR annual operating-cost savings $": (
            selected_c6_value
        ),
        "CR residual annual congestion cost $": (
            residual_congestion_cost
        ),
        "Remaining potential C6 value $": (
            remaining_c6_value
        ),
        "Baseline C6 binding hours": (
            baseline_binding_hours
        ),
        "CR C6 binding hours": selected_binding_hours,
        "C6 binding hours removed": binding_hours_removed,
        "C6 binding-hour reduction share": (
            binding_hour_reduction_share
        ),
    },
    name="Value",
)

print("\nReconductoring accounting:")
display(reconductoring_accounting.to_frame())

print(
    f"\n{CASE_CR} is now defined as C6 uprated from "
    f"{baseline_rating_mw:.1f} MW to "
    f"{selected_rating_mw:.1f} MW."
)

print(
    "No data-center demand or flexibility has been "
    "introduced at this stage."
)

## 14. Final Results Figures

The first figure summarizes the economic screening of the four finalist
branches and the diminishing returns associated with additional C6 capacity.

For branch $l$ and tested rating multiplier $m$, annual operating value is
defined as

$$
V_l(m)
=
C^{C0}-C_l(m),
$$

where $C^{C0}$ is baseline annual operating cost and $C_l(m)$ is annual
operating cost after changing the rating of branch $l$.

The branch comparison uses each candidate's maximum observed annual value.
The C6 curve then shows how much of that value is obtained at progressively
larger capacity multipliers. Ratings above twice the original limit are
excluded from the plotted horizontal scale because the very large 5,000 MW
test is only an unconstrained-line benchmark.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Create a dedicated directory for final Notebook 4 figures.
NOTEBOOK4_FIGURE_DIR = (
    Path.home()
    / "thesis-grid-ai"
    / "results"
    / "notebook4_figures"
)

NOTEBOOK4_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

candidate_order = ["C6", "CB-1", "C29", "A27"]

figure_sweep_data = (
    capacity_sweep_summary.loc[
        capacity_sweep_summary["UID"].isin(candidate_order)
    ]
    .copy()
)

# Maximum annual value observed for each candidate branch.
candidate_maximum_values = (
    figure_sweep_data
    .groupby("UID")["Annual operating-cost savings $"]
    .max()
    .reindex(candidate_order)
)

# C6 curve through the 2.0x case.
c6_curve = (
    figure_sweep_data.loc[
        figure_sweep_data["UID"].eq("C6")
        & figure_sweep_data[
            "Capacity multiplier"
        ].le(2.0 + 1e-9)
    ]
    .sort_values("Capacity multiplier")
    .copy()
)

selected_curve_point = c6_curve.loc[
    np.isclose(
        c6_curve["Capacity multiplier"],
        SELECTED_RECONDUCTORING_MULTIPLIER,
    )
].iloc[0]

maximum_c6_value_millions = (
    maximum_c6_value / 1_000_000
)

sufficient_c6_multiplier = (
    float(
        maximum_value_row["Maximum absolute flow MW"]
    )
    / baseline_rating_mw
)

million_dollar_formatter = FuncFormatter(
    lambda value, position: f"${value:,.1f}M"
)

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13.5, 5.4),
)

# Panel A: candidate ranking.
candidate_values_millions = (
    candidate_maximum_values / 1_000_000
)

bars = axes[0].bar(
    candidate_values_millions.index,
    candidate_values_millions.values,
    color=["#1f4e79", "#6f8fa6", "#8ca6b8", "#a9bdca"],
    edgecolor="black",
    linewidth=0.6,
)

axes[0].set_title(
    "A. Maximum annual operating value",
    loc="left",
    fontweight="bold",
)

axes[0].set_xlabel("Candidate branch")
axes[0].set_ylabel("Annual operating-cost savings")
axes[0].yaxis.set_major_formatter(
    million_dollar_formatter
)
axes[0].grid(
    axis="y",
    alpha=0.25,
)
axes[0].set_axisbelow(True)

axes[0].bar_label(
    bars,
    labels=[
        f"${value:,.2f}M"
        for value in candidate_values_millions.values
    ],
    padding=4,
    fontsize=9,
)

axes[0].set_ylim(
    0,
    candidate_values_millions.max() * 1.15,
)

# Panel B: C6 diminishing-return curve.
axes[1].plot(
    c6_curve["Capacity multiplier"],
    (
        c6_curve["Annual operating-cost savings $"]
        / 1_000_000
    ),
    marker="o",
    linewidth=2.2,
    markersize=5.5,
    color="#1f4e79",
    label="C6 annual savings",
)

axes[1].scatter(
    [SELECTED_RECONDUCTORING_MULTIPLIER],
    [
        float(
            selected_curve_point[
                "Annual operating-cost savings $"
            ]
        )
        / 1_000_000
    ],
    s=90,
    color="#c43c35",
    edgecolor="black",
    linewidth=0.7,
    zorder=5,
    label="Selected 1.5x case",
)

axes[1].axhline(
    maximum_c6_value_millions,
    color="#555555",
    linestyle="--",
    linewidth=1.2,
    label="Maximum C6 value",
)

axes[1].axvline(
    sufficient_c6_multiplier,
    color="#7a7a7a",
    linestyle=":",
    linewidth=1.3,
    label="Sufficient unconstrained-flow rating",
)

axes[1].annotate(
    "Selected: 262.5 MW\n$5.301M annual savings",
    xy=(
        SELECTED_RECONDUCTORING_MULTIPLIER,
        selected_c6_value / 1_000_000,
    ),
    xytext=(1.28, 4.25),
    arrowprops={
        "arrowstyle": "->",
        "color": "#333333",
        "linewidth": 0.9,
    },
    fontsize=9,
)

axes[1].set_title(
    "B. Diminishing returns from C6 uprating",
    loc="left",
    fontweight="bold",
)

axes[1].set_xlabel(
    "C6 capacity multiplier"
)
axes[1].set_ylabel(
    "Annual operating-cost savings"
)
axes[1].yaxis.set_major_formatter(
    million_dollar_formatter
)
axes[1].set_xlim(0.98, 2.02)
axes[1].set_ylim(0, maximum_c6_value_millions * 1.12)
axes[1].grid(alpha=0.25)
axes[1].set_axisbelow(True)

axes[1].legend(
    loc="lower right",
    frameon=True,
    fontsize=8.5,
)

fig.suptitle(
    "Transmission-uprating value in the RTS-GMLC baseline",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

fig.text(
    0.5,
    -0.015,
    (
        "Operating-cost savings exclude capital cost. "
        "The 5,000 MW cases are used only to approximate "
        "an unconstrained branch."
    ),
    ha="center",
    fontsize=9,
)

fig.tight_layout()

figure_png_path = (
    NOTEBOOK4_FIGURE_DIR
    / "notebook4_reconductoring_value.png"
)

figure_pdf_path = (
    NOTEBOOK4_FIGURE_DIR
    / "notebook4_reconductoring_value.pdf"
)

fig.savefig(
    figure_png_path,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    figure_pdf_path,
    bbox_inches="tight",
)

plt.show()

print("Saved PNG:", figure_png_path)
print("Saved PDF:", figure_pdf_path)

assert figure_png_path.exists()
assert figure_pdf_path.exists()

print(
    "\nFinal reconductoring-value figure created successfully."
)

### Temporal Concentration of C6 Reconductoring Value

The annual value of reconductoring is not distributed uniformly throughout
the year.

The monthly results separate the operating-cost savings captured by the
selected 1.5-times rating from the additional value that would require a
larger C6 expansion.

The hourly profile groups annual savings by hour of day. It therefore shows
when the transmission expansion is most economically useful, rather than
representing the savings of a single day.

These temporal patterns will later be important when comparing reconductoring
with data-center load flexibility. Flexibility can substitute for or complement
the network expansion only if the data center can move consumption across the
hours in which congestion value is concentrated.

In [ ]:
# Create the final temporal profile figure for C6.

monthly_required_columns = {
    "Savings_At_1_5x_Dollars",
    "Maximum_Possible_Savings_Dollars",
}

hourly_required_columns = {
    "Savings_At_1_5x_Dollars",
    "Positive_Shadow_Hours",
}


def locate_profile_dataframe(
    preferred_name,
    required_columns,
    expected_rows,
):
    """Locate an existing validated profile DataFrame."""

    preferred_value = globals().get(preferred_name)

    if (
        isinstance(preferred_value, pd.DataFrame)
        and required_columns.issubset(
            preferred_value.columns
        )
        and len(preferred_value) == expected_rows
    ):
        return preferred_name, preferred_value.copy()

    candidates = []

    for variable_name, value in list(globals().items()):
        if (
            isinstance(value, pd.DataFrame)
            and required_columns.issubset(value.columns)
            and len(value) == expected_rows
        ):
            candidates.append((variable_name, value.copy()))

    assert candidates, (
        f"No {expected_rows}-row DataFrame containing "
        f"{sorted(required_columns)} was found."
    )

    return candidates[0]


monthly_source_name, monthly_temporal_data = (
    locate_profile_dataframe(
        preferred_name="c6_monthly_profile",
        required_columns=monthly_required_columns,
        expected_rows=12,
    )
)

hourly_source_name, hourly_temporal_data = (
    locate_profile_dataframe(
        preferred_name="c6_hourly_profile",
        required_columns=hourly_required_columns,
        expected_rows=24,
    )
)

print("Monthly profile source:", monthly_source_name)
print("Hourly profile source:", hourly_source_name)

# Prepare monthly plotting data.
monthly_temporal_data = monthly_temporal_data.copy()

monthly_temporal_data["Captured_Million_Dollars"] = (
    monthly_temporal_data[
        "Savings_At_1_5x_Dollars"
    ]
    / 1_000_000
)

monthly_temporal_data["Remaining_Million_Dollars"] = (
    (
        monthly_temporal_data[
            "Maximum_Possible_Savings_Dollars"
        ]
        - monthly_temporal_data[
            "Savings_At_1_5x_Dollars"
        ]
    )
    .clip(lower=0.0)
    / 1_000_000
)

month_labels = [
    pd.Period(str(month), freq="M").strftime("%b")
    for month in monthly_temporal_data.index
]

# Prepare hourly plotting data.
hourly_temporal_data = (
    hourly_temporal_data
    .sort_index()
    .copy()
)

hour_values = [
    int(hour)
    for hour in hourly_temporal_data.index
]

hourly_savings_thousands = (
    hourly_temporal_data[
        "Savings_At_1_5x_Dollars"
    ]
    / 1_000
)

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13.5, 5.4),
)

# Panel A: monthly captured and remaining value.
axes[0].bar(
    month_labels,
    monthly_temporal_data[
        "Captured_Million_Dollars"
    ],
    color="#1f4e79",
    edgecolor="black",
    linewidth=0.4,
    label="Captured by 1.5x rating",
)

axes[0].bar(
    month_labels,
    monthly_temporal_data[
        "Remaining_Million_Dollars"
    ],
    bottom=monthly_temporal_data[
        "Captured_Million_Dollars"
    ],
    color="#c9d6df",
    edgecolor="black",
    linewidth=0.4,
    label="Remaining C6 value",
)

axes[0].set_title(
    "A. Monthly operating value",
    loc="left",
    fontweight="bold",
)

axes[0].set_xlabel("Month")
axes[0].set_ylabel("Annual operating-cost savings")
axes[0].yaxis.set_major_formatter(
    FuncFormatter(
        lambda value, position: f"${value:,.1f}M"
    )
)
axes[0].grid(axis="y", alpha=0.25)
axes[0].set_axisbelow(True)
axes[0].legend(
    loc="upper right",
    fontsize=8.5,
    frameon=True,
)

# Panel B: savings aggregated by hour of day.
axes[1].plot(
    hour_values,
    hourly_savings_thousands,
    color="#1f4e79",
    linewidth=2.2,
    marker="o",
    markersize=4.5,
)

maximum_hour_position = int(
    hourly_savings_thousands.idxmax()
)

maximum_hour_savings = float(
    hourly_savings_thousands.max()
)

axes[1].scatter(
    [maximum_hour_position],
    [maximum_hour_savings],
    color="#c43c35",
    edgecolor="black",
    linewidth=0.7,
    s=75,
    zorder=5,
)

axes[1].annotate(
    (
        f"Maximum at hour {maximum_hour_position}:00\n"
        f"${maximum_hour_savings:,.0f} thousand"
    ),
    xy=(
        maximum_hour_position,
        maximum_hour_savings,
    ),
    xytext=(13.5, maximum_hour_savings * 0.78),
    arrowprops={
        "arrowstyle": "->",
        "color": "#333333",
        "linewidth": 0.9,
    },
    fontsize=9,
)

axes[1].set_title(
    "B. Annual savings by hour of day",
    loc="left",
    fontweight="bold",
)

axes[1].set_xlabel("Hour of day")
axes[1].set_ylabel(
    "Annual operating-cost savings"
)
axes[1].yaxis.set_major_formatter(
    FuncFormatter(
        lambda value, position: f"${value:,.0f}k"
    )
)
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_xlim(-0.5, 23.5)
axes[1].set_ylim(
    0,
    maximum_hour_savings * 1.15,
)
axes[1].grid(alpha=0.25)
axes[1].set_axisbelow(True)

fig.suptitle(
    "Temporal concentration of C6 reconductoring value",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

fig.text(
    0.5,
    -0.015,
    (
        "Values are annual operating-cost savings under "
        "the baseline 2020 demand and generation profiles."
    ),
    ha="center",
    fontsize=9,
)

fig.tight_layout()

temporal_figure_png_path = (
    NOTEBOOK4_FIGURE_DIR
    / "notebook4_c6_temporal_value.png"
)

temporal_figure_pdf_path = (
    NOTEBOOK4_FIGURE_DIR
    / "notebook4_c6_temporal_value.pdf"
)

fig.savefig(
    temporal_figure_png_path,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    temporal_figure_pdf_path,
    bbox_inches="tight",
)

plt.show()

print("Saved PNG:", temporal_figure_png_path)
print("Saved PDF:", temporal_figure_pdf_path)

assert temporal_figure_png_path.exists()
assert temporal_figure_pdf_path.exists()

assert np.isclose(
    monthly_temporal_data[
        "Captured_Million_Dollars"
    ].sum()
    * 1_000_000,
    selected_c6_value,
    atol=0.01,
)

assert np.isclose(
    hourly_temporal_data[
        "Savings_At_1_5x_Dollars"
    ].sum(),
    selected_c6_value,
    atol=0.01,
)

print(
    "\nC6 temporal-value figure passed "
    "annual-savings reconciliation."
)

## 15. Reproducible Outputs and Notebook 5 Handoff

The validated Notebook 4 outputs are exported in compact, reusable formats.

The exported configuration identifies the selected reconductoring branch,
its original and post-project ratings, the sensitivity multipliers, and the
modeling assumptions applied to the intervention.

These files allow later notebooks to reconstruct the network cases without
rerunning the full reconductoring screening. The detailed monthly optimization
checkpoints remain available separately if individual hourly baseline results
must be recovered.

The exported `CR` configuration changes only the continuous thermal rating of
C6. Resistance, reactance, network topology, LTE and STE ratings, and all other
branch limits remain unchanged.

In [ ]:
import json

NOTEBOOK4_FINAL_DIR = (
    Path.home()
    / "thesis-grid-ai"
    / "results"
    / "notebook4_final"
)

NOTEBOOK4_FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------------
# 1. Audited annual baseline summary
# ------------------------------------------------------------------

annual_baseline_summary = pd.Series(
    {
        "Analysis year": 2020,
        "Hours solved": 8784,
        "Annual network operating cost $": (
            408_307_975.428204
        ),
        "Annual copper-plate operating cost $": (
            400_619_666.836656
        ),
        "Annual congestion cost $": (
            7_688_308.591549
        ),
        "Annual congestion premium": (
            7_688_308.591549
            / 400_619_666.836656
        ),
        "Economically congested hours": 3349,
        "Economically congested hour share": (
            3349 / 8784
        ),
        "Hours with at least one binding branch": 3456,
        "Primary VRE curtailment MWh": (
            541_589.329090
        ),
        "Minimum-cost VRE curtailment MWh": (
            523_045.511448
        ),
        "Degeneracy-related VRE difference MWh": (
            541_589.329090 - 523_045.511448
        ),
    },
    name="Value",
)

# Reconcile the audited annual congestion result.
assert np.isclose(
    annual_baseline_summary[
        "Annual congestion cost $"
    ],
    implied_baseline_congestion_cost,
    atol=0.01,
)

# ------------------------------------------------------------------
# 2. Portable reconductoring configuration
# ------------------------------------------------------------------

notebook4_configuration = {
    "schema_version": "notebook4_v1",
    "analysis_year": 2020,
    "number_of_hours": 8784,
    "baseline_case": {
        "case_id": CASE_C0,
        "description": "Original RTS-GMLC network",
    },
    "reconductoring_case": {
        "case_id": CASE_CR,
        "branch_uid": SELECTED_RECONDUCTORING_UID,
        "from_bus": from_bus,
        "to_bus": to_bus,
        "endpoint_voltage_kv": float(
            endpoint_data["BaseKV"].iloc[0]
        ),
        "branch_length_field_value": float(
            selected_branch["Length"]
        ),
        "baseline_continuous_rating_mw": (
            baseline_rating_mw
        ),
        "selected_capacity_multiplier": (
            SELECTED_RECONDUCTORING_MULTIPLIER
        ),
        "selected_continuous_rating_mw": (
            selected_rating_mw
        ),
        "capacity_uplift_mw": capacity_uplift_mw,
        "sensitivity_multipliers": [
            float(value)
            for value in (
                RECONDUCTORING_SENSITIVITY_MULTIPLIERS
            )
        ],
        "economically_valuable_direction": {
            "from_bus": from_bus,
            "to_bus": to_bus,
        },
    },
    "baseline_calibration_results": {
        "annual_operating_cost_savings_dollars": (
            selected_c6_value
        ),
        "share_of_baseline_congestion_removed": (
            selected_congestion_share
        ),
        "share_of_maximum_c6_value_captured": (
            selected_c6_value / maximum_c6_value
        ),
        "baseline_c6_binding_hours": (
            baseline_binding_hours
        ),
        "selected_case_c6_binding_hours": (
            selected_binding_hours
        ),
        "sufficient_unconstrained_flow_rating_mw": (
            float(
                maximum_value_row[
                    "Maximum absolute flow MW"
                ]
            )
        ),
    },
    "modeling_assumptions": {
        "continuous_thermal_limit_changed": True,
        "resistance_changed": False,
        "reactance_changed": False,
        "topology_changed": False,
        "lte_rating_used": False,
        "ste_rating_used": False,
        "n_minus_one_security_modeled": False,
        "capital_cost_included": False,
        "interpretation": (
            "Reduced-form thermal uprating proxy "
            "for reconductoring"
        ),
    },
}

# ------------------------------------------------------------------
# 3. Export paths
# ------------------------------------------------------------------

annual_summary_path = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_annual_baseline_summary.csv"
)

case_comparison_path = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_case_comparison.csv"
)

selected_case_path = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_selected_reconductoring_case.csv"
)

capacity_sweep_path = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_capacity_sweep_summary.csv"
)

monthly_profile_path = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_c6_monthly_profile.csv"
)

hourly_profile_path = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_c6_hourly_profile.csv"
)

accounting_path = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_reconductoring_accounting.csv"
)

configuration_path = (
    NOTEBOOK4_FINAL_DIR
    / "notebook4_reconductoring_configuration.json"
)

# ------------------------------------------------------------------
# 4. Write validated outputs
# ------------------------------------------------------------------

annual_baseline_summary.to_frame().to_csv(
    annual_summary_path
)

reconductoring_case_comparison.to_csv(
    case_comparison_path
)

selected_reconductoring_case.to_frame().to_csv(
    selected_case_path
)

capacity_sweep_summary.to_csv(
    capacity_sweep_path,
    index=False,
)

monthly_temporal_data.to_csv(
    monthly_profile_path
)

hourly_temporal_data.to_csv(
    hourly_profile_path
)

reconductoring_accounting.to_frame().to_csv(
    accounting_path
)

with configuration_path.open(
    "w",
    encoding="utf-8",
) as configuration_file:
    json.dump(
        notebook4_configuration,
        configuration_file,
        indent=2,
    )

# ------------------------------------------------------------------
# 5. Export validation and manifest
# ------------------------------------------------------------------

exported_paths = [
    annual_summary_path,
    case_comparison_path,
    selected_case_path,
    capacity_sweep_path,
    monthly_profile_path,
    hourly_profile_path,
    accounting_path,
    configuration_path,
    figure_png_path,
    figure_pdf_path,
    temporal_figure_png_path,
    temporal_figure_pdf_path,
]

for exported_path in exported_paths:
    assert exported_path.exists(), (
        f"Missing exported artifact: {exported_path}"
    )

export_manifest = pd.DataFrame(
    {
        "Artifact": [
            path.name
            for path in exported_paths
        ],
        "Directory": [
            str(path.parent)
            for path in exported_paths
        ],
        "File size bytes": [
            path.stat().st_size
            for path in exported_paths
        ],
    }
)

display(export_manifest)

print(
    "\nNotebook 4 final outputs exported successfully."
)
print("Final-results directory:", NOTEBOOK4_FINAL_DIR)
print("Figure directory:", NOTEBOOK4_FIGURE_DIR)

print(
    "\nNotebook 5 should load:"
    f"\n{configuration_path}"
)

## 16. Conclusions and Handoff

### 16.1 Purpose of Notebook 4

Notebook 4 extended the validated single-hour DC optimal power-flow model
from Notebook 3 into a full-year chronological baseline.

A total of 8,784 hourly operating conditions were solved using:

- reconstructed hourly bus-level demand;
- hourly generator availability;
- the 73-bus, 120-branch RTS-GMLC network;
- 158 individually represented generators;
- linear generator operating costs; and
- separate thermal constraints for every physical branch.

Each hour was solved independently. The resulting annual baseline was then
used to identify, validate, and parameterize a representative transmission
reconductoring intervention.

---

### 16.2 Full-Year Baseline Results

The original network produced the following annual operating results:

- annual network-constrained operating cost: **\$408.308 million**;
- annual copper-plate operating cost: **\$400.620 million**;
- annual congestion cost: **\$7.688 million**;
- annual congestion premium: **1.919%**;
- economically congested hours: **3,349**, or **38.13%** of the year;
- hours with at least one physically binding line: **3,456**;
- primary-solution VRE curtailment: **541,589 MWh**; and
- minimum-cost VRE curtailment: **523,046 MWh**.

The distinction between physically binding and economically valuable
constraints proved important. Some lines reached their limits while having
zero shadow value because alternative zero-cost dispatches were available.

A secondary lexicographic optimization was therefore used to maximize VRE
dispatch while holding operating cost within a numerical tolerance of the
minimum-cost solution. Economic costs and line shadow values were taken from
the primary optimization, while minimum-cost VRE curtailment was measured
from the secondary optimization.

---

### 16.3 Transmission Screening

Only ten of the 120 physical branches had a positive capacity shadow value
during at least one hour of the year.

Branch C6 was the dominant candidate:

- UID: **C6**;
- endpoints: **bus 303 to bus 309**;
- endpoint names: **Caesar to Carter**;
- voltage: **138 kV**;
- area: **Area 3**;
- baseline continuous rating: **175 MW**;
- positive-shadow hours: **1,777**;
- baseline binding hours: **1,837**;
- annual marginal capacity value: **\$92,808 per MW-year**; and
- maximum hourly shadow value: **\$80.22 per MW-hour**.

The annual dual prediction for a 1 MW increase was independently tested with
a full-year finite-difference simulation. The predicted value was
approximately \$92,808, while the realized operating-cost saving was
approximately \$92,458. The finite-difference result therefore reproduced
approximately 99.62% of the dual prediction.

This agreement validates the economic interpretation of the line-capacity
shadow value for a small local capacity change.

---

### 16.4 Selected Reconductoring Scenario

The original-network configuration is denoted by `C0`, while the selected
reconductoring configuration is denoted by `CR`.

For C6, the original rating is

$$
\overline{F}_{\mathrm{C6}}^{C0}
=
175\ \mathrm{MW}.
$$

The selected reconductoring multiplier is

$$
m^{R}=1.50.
$$

The resulting post-project rating is

$$
\overline{F}_{\mathrm{C6}}^{CR}
=
1.50(175)
=
262.5\ \mathrm{MW}.
$$

The modeled capacity uplift is therefore

$$
\Delta\overline{F}_{\mathrm{C6}}
=
262.5-175
=
87.5\ \mathrm{MW}.
$$

Under the original 2020 demand and generation profiles, the selected
configuration produces:

- annual operating-cost savings of **\$5.301 million**;
- removal of **68.95%** of total baseline congestion cost;
- capture of **92.18%** of C6's maximum possible operating value;
- a reduction in C6 binding hours from **1,837 to 661**; and
- residual annual system congestion cost of approximately
  **\$2.387 million**.

C6 becomes economically valuable only in the direction from bus 303 toward
bus 309. No positive lower-direction shadow values were observed.

The line's value is concentrated in winter, particularly January and
November, and during evening and overnight hours. Annual savings by hour of
day reach their maximum at approximately 22:00 and are lowest around midday.

This result demonstrates that the value of the transmission intervention
cannot be explained by annual peak demand alone. It depends on the joint
spatial and temporal pattern of demand, generation availability, and network
flows.

---

### 16.5 Interpretation of the Selected Rating

A C6 rating of approximately 325 MW was sufficient to reproduce the dispatch
obtained when the branch was assigned an effectively unconstrained limit.

The corresponding multiplier is approximately

$$
\frac{324.984}{175}
=
1.857.
$$

This value is sufficient to reproduce the observed unconstrained-line flow,
but it is not interpreted as an exact engineering minimum. Alternative
cost-equivalent dispatches may exist.

The selected 1.5-times case retains some congestion and therefore preserves
room for data-center flexibility to affect system operation. A completely
unconstrained C6 case could remove much of the network interaction that the
research is intended to study.

Sensitivity cases using multipliers of 1.25, 1.50, and 2.00 are retained for
later robustness analysis.

---

### 16.6 Modeling Limitations

The following limitations must be retained when interpreting the results:

1. Each hour is optimized independently. Unit commitment, minimum up and down
   times, ramp coupling, startup costs, reserves, and intertemporal storage
   operation are not modeled.

2. Regional demand is allocated to buses using fixed within-area shares.
   Hourly regional totals are observed in the dataset, but hourly bus-level
   spatial distributions are reconstructed.

3. Generator costs use one linear full-load-average coefficient per unit.
   The model does not represent complete piecewise heat-rate curves.

4. The DC power-flow approximation ignores reactive power, voltage magnitudes,
   and electrical losses.

5. The model applies continuous branch ratings only. LTE and STE ratings and
   N-minus-one contingency security are not represented.

6. Reconductoring is represented as a thermal-rating increase. Resistance,
   reactance, topology, and other branch parameters remain unchanged.

7. Capital cost, construction time, permitting, outage requirements, and
   detailed conductor feasibility are outside the current model.

8. The calculated savings are operating-cost savings, not the net present
   value of an actual transmission investment.

9. No data-center demand or data-center flexibility has yet been introduced.

Consequently, C6 is selected as the most economically relevant experimental
network intervention—not as a complete engineering investment recommendation.

---

### 16.7 Handoff to Notebook 5

Notebook 5 will load the validated reconductoring configuration from:

`results/notebook4_final/notebook4_reconductoring_configuration.json`

It will then introduce the data-center demand model and validate:

- candidate data-center locations;
- data-center capacity and energy requirements;
- an inflexible reference demand profile;
- flexible workload-shifting variables;
- hourly power limits;
- workload completion deadlines;
- deferred-work recovery limits; and
- conservation of required computing energy.

The original and reconductored network configurations will remain fixed while
data-center flexibility is varied.

Later experiments will compare four operating-cost outcomes:

- `C0`: original network without data-center flexibility;
- `CF`: original network with data-center flexibility;
- `CR`: reconductored network without data-center flexibility; and
- `CFR`: reconducted network with data-center flexibility.

The interaction measure will be

$$
\Gamma
=
C^{CF}
+
C^{CR}
-
C^{CFR}
-
C^{C0}.
$$

A positive value,

$$
\Gamma>0,
$$

means that flexibility and reconductoring are complements: their combined
operating-cost benefit exceeds the sum of their separate benefits.

A negative value,

$$
\Gamma<0,
$$

means that they are substitutes: one intervention reduces the marginal value
of the other.

Notebook 4 therefore supplies the validated annual baseline, congestion
diagnostics, selected transmission intervention, sensitivity range, and
reproducible configuration required for the data-center experiments.